In [ ]:
# Install SoupX if it is not already installed
install.packages("SoupX")
# Load SoupX for ambient RNA correction
library(SoupX)

In [ ]:
# Load Seurat for single-cell object handling
library(Seurat)
# Load dplyr for metadata manipulation
library(dplyr)

In [ ]:
root_dir <- "./data"
# Check whether the root directory exists
file.exists(root_dir)           
dir.exists(root_dir)           

# List sample folders under the root directory
list.dirs(root_dir, recursive = FALSE)
list.files(root_dir)

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install("DropletUtils")

In [ ]:
library(SoupX)
library(Seurat)
library(dplyr)
library(Matrix)

root_dir <- "./data"

if (!dir.exists(root_dir)) {
  stop("none", root_dir)
}

read_bgi_custom <- function(data.dir) {
  # Read the sparse matrix in Matrix Market format
  mat <- Matrix::readMM(gzfile(file.path(data.dir, "matrix.mtx.gz")))
  # Read cell barcodes; each line is one barcode
  barcodes <- read.table(
    gzfile(file.path(data.dir, "barcodes.tsv.gz")),
    header = FALSE, stringsAsFactors = FALSE, sep = "\t"
  )[, 1]
  # Read feature table
  features <- read.table(
    gzfile(file.path(data.dir, "features.tsv.gz")),
    header = FALSE, stringsAsFactors = FALSE, sep = "\t"
  )
  # Use gene name if available; otherwise use gene ID
  if (ncol(features) >= 2) {
    rownames(mat) <- make.names(features[, 2], unique = TRUE)
  } else {
    rownames(mat) <- make.names(features[, 1], unique = TRUE)
  }
  # Assign barcodes as column names
  colnames(mat) <- barcodes
  # Convert to compressed sparse column matrix
  mat <- as(mat, "CsparseMatrix")
  return(mat)
}

write_bgi_custom <- function(path, mat) {
  # Create output directory if missing
  if (!dir.exists(path)) dir.create(path, recursive = TRUE, showWarnings = FALSE)
  # Build feature table with gene ID, gene name, and feature type
  features_df <- data.frame(
    gene_id = rownames(mat),
    gene_name = rownames(mat),
    feature_type = "Gene Expression",
    stringsAsFactors = FALSE
  )
  # Write matrix, barcodes, and features 
  Matrix::writeMM(mat, file = file.path(path, "matrix.mtx"))
  writeLines(colnames(mat), con = file.path(path, "barcodes.tsv"))
  write.table(features_df, file = file.path(path, "features.tsv"),
              row.names = FALSE, col.names = FALSE, sep = "\t", quote = FALSE)
  # Compress output files with gzip
  system(paste("gzip -f", file.path(path, "matrix.mtx")))
  system(paste("gzip -f", file.path(path, "barcodes.tsv")))
  system(paste("gzip -f", file.path(path, "features.tsv")))
  
  cat("BGI-style matrix saved to:", path, "\n")
}


# Run SoupX ambient RNA correction
# Define output directory for SoupX-corrected matrices
output_dir <- file.path(root_dir, "soupx_corrected")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)
# Detect sample folders starting with "cDNA-"
sample_dirs <- list.dirs(root_dir, recursive = FALSE)
sample_dirs <- sample_dirs[grep("^cDNA-", basename(sample_dirs))]
# Print detected samples
cat("Detected samples:", length(sample_dirs), "\n")
print(basename(sample_dirs))
# Stop if no sample is found 
if (length(sample_dirs) == 0) stop("No sample found")
# Store SoupX contamination results
soupx_results <- list()

for (sample_path in sample_dirs) {
  # Extract sample name from folder path
  sample_name <- basename(sample_path)
  cat("\n==========", sample_name, "==========\n")
  # Define raw and filtered matrix directories
  raw_dir <- file.path(sample_path, "output", "raw_matrix")
  filtered_dir <- file.path(sample_path, "output", "filter_matrix")
  # Skip sample if raw or filtered directory is missing 
  if (!dir.exists(raw_dir) || !dir.exists(filtered_dir)) {
    cat("[Skip] raw/filter directory not found\n")
    next
  }
  
  tryCatch({
    # Read raw matrix containing empty droplets and cells
    cat("Reading raw matrix...\n")
    raw_counts <- read_bgi_custom(raw_dir)          
    cat("  raw:", nrow(raw_counts), "genes x", ncol(raw_counts), "cells\n")

    # Read filtered matrix containing cell-containing droplets
    cat("Reading filtered matrix...\n")
    filtered_counts <- read_bgi_custom(filtered_dir)  
    cat("  filtered:", nrow(filtered_counts), "genes x", ncol(filtered_counts), "cells\n")

    # Keep only shared genes
    cat("Aligning gene sets...\n")
    common_genes <- intersect(rownames(raw_counts), rownames(filtered_counts))
    cat("Number of common genes:", length(common_genes), "\n")
    
    raw_counts <- raw_counts[common_genes, ]
    filtered_counts <- filtered_counts[common_genes, ]
    raw_counts <- as(raw_counts, "CsparseMatrix")
    filtered_counts <- as(filtered_counts, "CsparseMatrix")
    # Build SoupChannel: tod = raw droplets, toc = filtered cells
    cat("Building SoupChannel...\n")
    sc <- SoupChannel(tod = raw_counts, toc = filtered_counts)
    
    # Cluster filtered cells with Seurat for cluster-level estimation
    cat("Clustering filtered cells with Seurat...\n")
    seu <- CreateSeuratObject(counts = filtered_counts)
    seu <- NormalizeData(seu)
    seu <- FindVariableFeatures(seu, selection.method = "vst", nfeatures = 2000)
    seu <- ScaleData(seu)
    seu <- RunPCA(seu, features = VariableFeatures(object = seu))
    seu <- FindNeighbors(seu, dims = 1:20)
    seu <- FindClusters(seu, resolution = 0.8)
    
    cat("Passing cluster information...\n")
    sc <- setClusters(sc, setNames(seu$seurat_clusters, colnames(seu)))
    
    # Estimate contamination rate automatically
    cat("Estimating contamination rate...\n")
    sc <- autoEstCont(sc, forceAccept = TRUE)
    # Extract estimated rho
    rho <- sc$fit$rhoEst
    cat("Contamination rate:", round(rho * 100, 2), "%\n")
    # Subtract the estimated ambient RNA contamination from the filtered counts
    corrected_counts <- adjustCounts(sc)
    
   # ---- Save the SoupX-corrected results ----
   # Define the output directory for this sample
    sample_out <- file.path(output_dir, sample_name)
    # Create the output directory if it does not exist
    dir.create(sample_out, showWarnings = FALSE, recursive = TRUE)
    
    write_bgi_custom(file.path(sample_out, "soupx_matrix"), corrected_counts)
    saveRDS(corrected_counts, file = file.path(sample_out, "soupx_counts.rds"))
    
    soupx_results[[sample_name]] <- data.frame(
      sample = sample_name,
      rho = rho,
      rho_percent = round(rho * 100, 2),
      stringsAsFactors = FALSE
    )
    # Print completion message for this BGI sample
    cat("  [Done]\n")
    
  }, error = function(e) {
    cat("  [Error]", conditionMessage(e), "\n")
    soupx_results[[sample_name]] <<- data.frame(
      sample = sample_name, rho = NA, rho_percent = NA,
      error = conditionMessage(e), stringsAsFactors = FALSE
    )
  })
}

cat("\n========== All BGI samples done ==========\n")
summary_df <- bind_rows(soupx_results)
print(summary_df)
write.csv(summary_df, file.path(output_dir, "soupx_summary.csv"), row.names = FALSE)

In [ ]:
# after SoupX correction

In [ ]:
# ============================================
# Step 2: Post-SoupX QC and merging
# ============================================

suppressMessages(library(tidyverse))
suppressMessages(library(Seurat))
suppressMessages(library(patchwork))
suppressMessages(library(Matrix))
suppressMessages(library(cowplot))
suppressMessages(library(ggplot2))
suppressMessages(library(harmony))

# ---- 1. Path settings ----
soupx_dir <- "./data/soupx_corrected"
save_dir_step2 <- "./data"
fig_dir <- file.path(save_dir_step2, "figures_QC")
dir.create(save_dir_step2, showWarnings = FALSE, recursive = TRUE)
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

# ---- 2. Sample info (donor + region mapping)
sample_info <- data.frame(
  sample_label = c("cDNA-250114087", "cDNA-250114088", "cDNA-250110016","cDNA-250110017",
 "cDNA-250110018","cDNA-250110019","cDNA-250114089", "cDNA-250114090",
                   "cDNA-250110006", "cDNA-250110007", "cDNA-250114082", "cDNA-250114083",
                   "cDNA-250114084", "cDNA-250114085", "cDNA-250114086"),
  donor = c(rep("Mq246", 8), rep("Mq277", 7)),
  samples = c(rep("DR-IV-TD", 8), rep("MR-DR-VR-IV", 7)),
  stringsAsFactors = FALSE
)

# ---- 3. Load SoupX-corrected matrices 
cat("Loading SoupX-corrected data...\n")
seurat_list <- list()

for (i in 1:nrow(sample_info)) {
  s <- sample_info$sample_label[i]
  cat("  ", s, "\n")
  
  counts <- readRDS(file.path(soupx_dir, s, "soupx_counts.rds"))
  seu <- CreateSeuratObject(counts = counts, project = s)
  
  # Add meta data
  seu$sample_label <- s
  seu$donor <- sample_info$donor[i]
  seu$samples <- sample_info$samples[i]
  
  seurat_list[[s]] <- seu
}

# ---- 4. Merge all samples ----
cat("\nMerging all samples...\n")
seurat <- merge(seurat_list[[1]], y = seurat_list[-1])
cat("After merge:", ncol(seurat), "cells x", nrow(seurat), "genes\n")

# ---- 5. Compute QC metrics  ----
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA

MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
found_mt <- MTGenes[MTGenes %in% rownames(seurat)]
cat("Found MT genes:", length(found_mt), "\n")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = found_mt)

# ---- 6. Sample-level filter (median nFeature > 500) ----
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))
select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)
cat("Samples retained after filter:", length(select_group), "\n")

# ---- 7. Pre-QC cell counts by library ----
oriCellNum <- ncol(seurat)
cat("Cells before QC:", oriCellNum, "\n")

qc_before <- as.data.frame(table(seurat$sample_label))
colnames(qc_before) <- c("sample_label", "cells_before_QC")

# ---- 8. QC filter----
seurat <- subset(seurat,
                 nFeature_RNA > 700 &
                 percent.mt < 5 &
                 countFeatureRatio > 1.2 &
                 nCount_RNA < 100000)
cellNum1 <- ncol(seurat)
cat("Cells after QC:", cellNum1, "(retained", round(cellNum1/oriCellNum*100, 2), "%)\n")

# ---- 8b. Post-QC counts per library + before/after table ----
qc_after <- as.data.frame(table(seurat$sample_label))
colnames(qc_after) <- c("sample_label", "cells_after_QC")

qc_per_library <- merge(qc_before, qc_after, by = "sample_label", all.x = TRUE)
# Fill 0 for libraries completely removed 
qc_per_library$cells_after_QC[is.na(qc_per_library$cells_after_QC)] <- 0  
qc_per_library$retained_pct <- round(
  qc_per_library$cells_after_QC / qc_per_library$cells_before_QC * 100, 1)

# Attach donor/region info 
qc_per_library <- qc_per_library %>%
  left_join(distinct(sample_info, sample_label, donor, samples), by = "sample_label")

cat("\nCell counts before/after QC per library:\n")
print(qc_per_library, row.names = FALSE)

write.csv(qc_per_library,
          file.path(save_dir_step2, "QC_per_library_cell_counts.csv"),
          row.names = FALSE)
# ---- 9. QC plots ----
Idents(seurat) <- "sample_label"

# Violin plots
p_vln <- VlnPlot(seurat, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), 
                 pt.size = 0, ncol = 1, group.by = "sample_label")
png(file.path(fig_dir, "QC_violin.png"), width = 1500, height = 1800, res = 150)
print(p_vln)
dev.off()

# Scatter plots
p1 <- FeatureScatter(seurat, "nCount_RNA", "nFeature_RNA", group.by = "sample_label", pt.size = 0.3)
p2 <- FeatureScatter(seurat, "nCount_RNA", "percent.mt", group.by = "sample_label", pt.size = 0.3)
p3 <- FeatureScatter(seurat, "nFeature_RNA", "percent.mt", group.by = "sample_label", pt.size = 0.3)
p_scatter <- (p1 + p2 + p3) + plot_layout(ncol = 3)

png(file.path(fig_dir, "QC_scatter.png"), width = 2000, height = 650, res = 150)
print(p_scatter)
dev.off()

cat("QC figures saved to:", fig_dir, "\n")

# ---- 10. Save Step-2 object ----
filename <- file.path(save_dir_step2, "macaque_DR_soupx_batch1.rds")
saveRDS(seurat, filename)
cat("Step-2 object saved:", filename, "\n")

In [ ]:
# snRNA-seq analysis 

In [ ]:
library(Seurat)
library(dplyr)

seurat <- readRDS("./data/macaque_DR_soupx_batch1.rds")

# Sample-level filter (median > 500)
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))
select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label

seurat <- subset(seurat, sample_label %in% select_group)
oriCellNum <- ncol(seurat)

# C filter
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA
MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
seurat_qc <- subset(seurat, nFeature_RNA > 700 & nCount_RNA < 100000 &
                    percent.mt < 5 & countFeatureRatio > 1.2)
cellNum1 <- ncol(seurat_qc)

cat("Cells before QC:", oriCellNum, "\n")
cat("Cells after QC:", cellNum1, "\n")
cat("Removed:", oriCellNum - cellNum1, 
    "(", round((oriCellNum - cellNum1)/oriCellNum*100, 2), "%)\n")

In [ ]:
# Step-2 QC plots (no downsampling)
library(Seurat)
library(ggplot2)
library(patchwork)
library(dplyr)

# ==================== Output dir ====================
save_dir <- "./data/figures_QC-latest-812"
if (!dir.exists(save_dir)) {
  dir.create(save_dir, recursive = TRUE)
  cat("Created directory:", save_dir, "\n")
}

# ====================  Sample-level filter (median nFeature > 500)====================
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))

select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)

# ==================== Compute QC metrics ====================
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA
MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
seurat <- NormalizeData(seurat)

oriCellNum <- ncol(seurat)

# ====================Use all data for plotting without downsampling====================
seurat_plot <- seurat
cat("Plotting with ALL", ncol(seurat_plot), "cells\n")

# ====================  Violin plot function====================
seurat_plot_vln <- subset(seurat_plot, nCount_RNA <= 25000)
cat("Violin plot using", ncol(seurat_plot_vln), "cells (nCount_RNA <= 25000)\n")
# Publication-style violin plot function, colored by donor with nFeature ticks every 1000
myVlnPlot <- function(seurat_obj, feature, group.by = "sample_label", 
                      fill.by = "donor", title = NULL, ylabel = NULL) {
  meta <- seurat_obj@meta.data
  meta$group <- factor(meta[[group.by]])
  meta$fill_group <- factor(meta[[fill.by]])
 # Donor colors: Mq246 in blue and Mq277 in orange; adjust to the actual donor names if needed
  donor_colors <- c("Mq246" = "#1f77b4", "Mq277" = "#ff7f0e")
  
  p <- ggplot(meta, aes(x = group, y = .data[[feature]], fill = fill_group)) +
    geom_violin(scale = "width", trim = TRUE, linewidth = 0.3) +
    geom_violin(scale = "width", trim = TRUE, linewidth = 0.3, alpha = 0.85) +
    scale_fill_manual(values = donor_colors, name = "Donor") +
    theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10, color = "black"),
      axis.text.y = element_text(size = 10, color = "black"),
      axis.title.x = element_blank(),
      axis.title.y = element_text(size = 12, face = "bold"),
      plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
      legend.position = "right",
      legend.key.size = unit(0.4, "cm"),
      legend.text = element_text(size = 9),
      legend.title = element_text(size = 10, face = "bold"),
      panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5)
    ) +
    labs(title = title, y = ifelse(is.null(ylabel), feature, ylabel))
    #  Set nFeature_RNA axis ticks at intervals of 1000  
  if (feature == "nFeature_RNA") {
    ymax <- max(meta[[feature]], na.rm = TRUE)
    p <- p + scale_y_continuous(breaks = seq(0, ceiling(ymax / 1000) * 1000, 1000))
  }
  
  return(p)
}

#  (a) Pre-QC violin plots; show only nCount_RNA <= 25000 and color by donor
cat("\n--- Saving (a) Pre-QC Violin ---\n")
p_a1 <- myVlnPlot(seurat_plot_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_a2 <- myVlnPlot(seurat_plot_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_a3 <- myVlnPlot(seurat_plot_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_pre <- (p_a1 + p_a2 + p_a3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Pre-QC Metrics by Sample (Colored by Donor)", 
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "a_violin_preQC_byDonor.png"), width = 2000, height = 650, res = 150)
print(vln_pre)
dev.off()
cat("Saved: a_violin_preQC_byDonor.png\n")
#  (b) Pre-QC scatter plots using all cells without downsampling
cat("\n--- Saving (b) Pre-QC Scatter ---\n")

scatter1 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "nFeature_RNA",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs Gene Count") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter2 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs MT%") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter3 <- FeatureScatter(
  seurat_plot, feature1 = "nFeature_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("Gene Count vs MT%") + 
  theme_classic() +
  theme(legend.position = "right", plot.title = element_text(size = 12, face = "bold"),
        legend.key.size = unit(0.3, "cm"), legend.text = element_text(size = 8))

scatter_pre <- (scatter1 + scatter2 + scatter3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1.2)) +
  plot_annotation(title = "Pre-QC Scatter Plots",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "b_scatter_preQC.png"), width = 2000, height = 650, res = 150)
print(scatter_pre)
dev.off()
ggsave(file.path(save_dir, "b_scatter_preQC.pdf"),
       scatter_pre, width = 13.33, height = 4.33, device = cairo_pdf)  
cat("Saved: b_scatter_preQC.png / .pdf\n")

# ==================== (c) QC ====================
seurat_qc <- subset(seurat, 
                    nFeature_RNA > 700 & 
                    nCount_RNA < 100000 &
                    percent.mt < 5 & 
                    countFeatureRatio > 1.2)

cellNum1 <- ncol(seurat_qc)
cat("\nPost-QC:", cellNum1, "cells (removed", oriCellNum - cellNum1, 
    round((oriCellNum - cellNum1)/oriCellNum * 100, 1), "%)\n")

# ==================== Post-doublet QC violin ====================
seurat_qc_vln <- subset(seurat_qc, nCount_RNA <= 25000)
cat("Post-QC Violin plot using", ncol(seurat_qc_vln), "cells (nCount_RNA <= 25000)\n")

cat("\n--- Saving (c) Post-QC Violin ---\n")

p_d1 <- myVlnPlot(seurat_qc_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_d2 <- myVlnPlot(seurat_qc_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_d3 <- myVlnPlot(seurat_qc_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_post <- (p_d1 + p_d2 + p_d3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Post-QC Metrics by Sample (Colored by Donor)",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "c_violin_postQC_byDonor.png"), width = 2000, height = 650, res = 150)
print(vln_post)
dev.off()
ggsave(file.path(save_dir, "c_violin_postQC_byDonor.pdf"),
       vln_post, width = 13.33, height = 4.33, device = cairo_pdf) 
cat("Saved: c_violin_postQC_byDonor.png / .pdf\n")
# Validate the output file
cat("\n========== Generated Files ==========\n")
files <- c("a_violin_preQC_byDonor.png", "b_scatter_preQC.png", 
           "c_violin_postQC_byDonor.png")
for (f in files) {
  fp <- file.path(save_dir, f)
  if (file.exists(fp)) {
    cat("OK:", f, "-", round(file.size(fp)/1024, 1), "KB\n")
  } else {
    cat("MISSING:", f, "\n")
  }
}
# Summary
cat("\n========== QC Summary ==========\n")
cat("Original cells:", oriCellNum, "\n")
cat("Post-QC cells:", cellNum1, "\n")
cat("Removed:", oriCellNum - cellNum1, 
    "(", round((oriCellNum - cellNum1)/oriCellNum * 100, 2), "%)\n")
cat("Save directory:", save_dir, "\n")
cat("==================================\n")

In [ ]:
library(ggplot2)

# Extract metadata
meta <- seurat@meta.data

# Scatter plot of nCount_RNA versus nFeature_RNA, colored by percent.mt
p <- ggplot(meta, aes(x = nCount_RNA, y = nFeature_RNA, color = percent.mt)) +
  geom_point(size = 0.3, alpha = 0.5) +
  scale_color_distiller(palette = "Reds", direction = 1, name = "percent.mt (%)") +
  labs(x = "nCount_RNA", y = "nFeature_RNA") +
  theme_classic() +
  theme(
    axis.text = element_text(size = 11, color = "black"),
    axis.title = element_text(size = 12, face = "bold"),
    legend.title = element_text(size = 10),
    legend.text = element_text(size = 9)
  )

# Save and display the plot
fig_dir <- "./data/figures_QC-latest-812"
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

png(file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), width = 800, height = 700, res = 150)
print(p)
dev.off()

cat("saved:", file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), "\n")

In [ ]:
library(Seurat)
library(Matrix)
library(dplyr)
library(ggplot2)
library(patchwork)

# ========== Paths ==========
soupx_dir <- "./data/soupx_corrected"
save_dir_step2 <- "./data"
fig_dir <- file.path(save_dir_step2, "figures_QC-latest-812")
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

# ========== Sample information ==========
sample_info <- data.frame(
  sample_label = c("cDNA-250114087", "cDNA-250114088", "cDNA-250110016","cDNA-250110017",
 "cDNA-250110018","cDNA-250110019","cDNA-250114089", "cDNA-250114090",
                   "cDNA-250110006", "cDNA-250110007", "cDNA-250114082", "cDNA-250114083",
                   "cDNA-250114084", "cDNA-250114085", "cDNA-250114086"),
  donor = c(rep("Mq246", 8), rep("Mq277", 7)),
  samples = c(rep("DR-IV-TD", 8), rep("MR-DR-VR-IV", 7)),
  stringsAsFactors = FALSE
)

# ========== 1. Read all SoupX-corrected count matrices ==========
cat("Read all SoupX-corrected count matrices...\n")
counts_list <- list()

for (i in 1:nrow(sample_info)) {
  s <- sample_info$sample_label[i]
  cat("  ", s, "\n")
  
  counts <- readRDS(file.path(soupx_dir, s, "soupx_counts.rds"))
  
  # Add a sample prefix to each barcode to avoid duplicate names across samples
  colnames(counts) <- paste0(colnames(counts), "-", s)
  
  counts_list[[s]] <- counts
}

# ========== 2. Take the union of all genes, align every matrix, and fill missing values with zero==========
cat("\nalign...\n")
all_genes <- unique(unlist(lapply(counts_list, rownames)))
cat("the union of all genes:", length(all_genes), "\n")

counts_list_aligned <- lapply(names(counts_list), function(s) {
  mat <- counts_list[[s]]
  missing_genes <- setdiff(all_genes, rownames(mat))
  
  if (length(missing_genes) > 0) {
    zero_mat <- Matrix(0, nrow = length(missing_genes), ncol = ncol(mat),
                       dimnames = list(missing_genes, colnames(mat)))
    mat <- rbind(mat, zero_mat)
  }
  # Sort features consistently
  mat <- mat[all_genes, , drop = FALSE]
  return(mat)
})
names(counts_list_aligned) <- names(counts_list)

# ========== 3. Combine all count matrices with cbind into one layer==========
cat(" Combine counts...\n")
combined_counts <- do.call(cbind, counts_list_aligned)
cat(" Combined:", nrow(combined_counts), "genes x", ncol(combined_counts), "cells\n")

# ========== 4. Create a Seurat object with only one counts layer==========
seurat <- CreateSeuratObject(counts = combined_counts)
cat("Seurat:", nrow(seurat), "features,", ncol(seurat), "cells\n")
cat("Layers:", names(seurat@assays$RNA@layers), "\n")

# ========== 5.  Add metadata ==========
cell_counts <- sapply(counts_list_aligned, ncol)
seurat$sample_label <- rep(sample_info$sample_label, cell_counts)
seurat$donor <- rep(sample_info$donor, cell_counts)
seurat$samples <- rep(sample_info$samples, cell_counts)

# ========== 6. Calculate QC metrics ==========
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA

MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
found_mt <- MTGenes[MTGenes %in% rownames(seurat)]
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = found_mt)

# ========== 7.Sample-level filtering: median nFeature_RNA > 500==========
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))
select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)

oriCellNum <- ncol(seurat)
cat("pre QC :", oriCellNum, "cells\n")

# ==========8, Apply QC filtering ==========
seurat_qc <- subset(seurat,
                    nFeature_RNA > 700 &
                    percent.mt < 5 &
                    countFeatureRatio > 1.2 &
                    nCount_RNA < 100000)

cat("post QC:", ncol(seurat_qc), "cells (retained", round(ncol(seurat_qc)/oriCellNum*100,2), "%)\n")

# ========== 9,Save the RDS file; this is essential==========
seurat <- seurat_qc
rds_file <- file.path(save_dir_step2, "macaque_DR_soupx_batch1.rds")
saveRDS(seurat, rds_file)
cat("RDS saved:", rds_file, "\n")
cat("final:", ncol(seurat), "cells x", nrow(seurat), "genes, 1 layer\n")

In [ ]:
readDir <- "./data/"
seurat1 <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1.rds"))

In [ ]:
#  Estimate the doublet rate

In [ ]:
library(Seurat)
library(scDblFinder)
library(SingleCellExperiment)
library(dplyr)
library(ggplot2)

seurat1$sample_label <- as.character(seurat1$sample_label)
samples <- unique(seurat1$sample_label)

# Initialize explicitly as character to avoid factor-related problems
seurat1$scDblFinder.class <- NA_character_
seurat1$scDblFinder.score <- NA_real_

for (s in samples) {
  message("Processing sample: ", s)
  
  subset_obj <- subset(seurat1, subset = sample_label == s)
  if (ncol(subset_obj) < 100) next
  
  # Normalization and PCA
  subset_obj <- NormalizeData(subset_obj)
  subset_obj <- FindVariableFeatures(subset_obj, selection.method = "vst", nfeatures = 2000)
  subset_obj <- ScaleData(subset_obj)
  subset_obj <- RunPCA(subset_obj, features = VariableFeatures(object = subset_obj))
  
  # Convert to an SCE object and run scDblFinder
  sce <- as.SingleCellExperiment(subset_obj)
  sce <- scDblFinder(sce)
  
  # ========== Key fix: use AddMetaData to write results back by matching cell names ==========
  meta_to_add <- data.frame(
    scDblFinder.class = as.character(sce$scDblFinder.class),
    scDblFinder.score = sce$scDblFinder.score,
    stringsAsFactors = FALSE
  )
  rownames(meta_to_add) <- colnames(sce)  # Ensure that cell names are aligned
  
  seurat1 <- AddMetaData(seurat1, metadata = meta_to_add)
  
  dbl_n <- sum(sce$scDblFinder.class == "doublet", na.rm = TRUE)
  message("  Done. Doublets: ", dbl_n, "/", ncol(subset_obj), 
          " (", round(dbl_n / ncol(subset_obj) * 100, 2), "%)")
}

# ========== Calculate the doublet rate for each sample ==========
doublet_summary <- seurat1@meta.data %>%
  group_by(sample_label) %>%
  summarise(
    total_cells = n(),
    doublet_cells = sum(scDblFinder.class == "doublet", na.rm = TRUE),
    doublet_rate = round(doublet_cells / total_cells * 100, 2)
  ) %>%
  arrange(desc(doublet_rate))

print(doublet_summary)

# ========== Visualization ==========
# Bar plot
p1 <- ggplot(doublet_summary, aes(x = reorder(sample_label, -doublet_rate), y = doublet_rate)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  geom_text(aes(label = paste0(doublet_rate, "%")), vjust = -0.5, size = 3) +
  labs(x = "Sample", y = "Doublet Rate (%)", title = "scDblFinder Doublet Rate per Sample") +
  theme_bw(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
print(p1)

# If a UMAP is available, inspect the distribution of doublets
if ("umap" %in% names(seurat1@reductions)) {
  p2 <- DimPlot(seurat1, group.by = "scDblFinder.class", label = FALSE,
                cols = c("singlet" = "grey", "doublet" = "red")) +
    ggtitle("scDblFinder Predictions")
  print(p2)
}

# ========== Filter doublets and save the object ==========
seurat1_clean <- subset(seurat1, subset = scDblFinder.class == "singlet")

message("Before filtering: ", ncol(seurat1), " cells")
message("After filtering:  ", ncol(seurat1_clean), " cells")
message("Removed:          ", ncol(seurat1) - ncol(seurat1_clean), " doublets")

# Save
saveRDS(seurat1_clean, paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned.rds"))

In [ ]:
# 1. Inspect all metadata column names
colnames(seurat1@meta.data)

# 2. Inspect the distribution of sample_label
table(seurat1@meta.data$sample_label)

# 3. Inspect the distribution of samples
table(seurat1@meta.data$samples)

# 4. Inspect the cell barcode format
head(Cells(seurat1))

# 5. Extract the suffix from each barcode as the library ID
library(dplyr)
seurat1$library_id <- sapply(strsplit(Cells(seurat1), "-"), function(x) tail(x, 1))
table(seurat1$library_id)





In [ ]:
# === Read the RDS file saved in the previous step first ===
readDir <- "./data/"

# If this is the filename saved in the previous step:
seurat <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned.rds"))

# If the file above does not exist, try the version filtered for doublets but not yet QC-filtered
# seurat <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned.rds"))
cat("Loaded object with", ncol(seurat), "cells\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(patchwork)

# ============================================
# 0. Set paths and load the data
# ============================================
readDir  <- "./data/"
save_dir <- "./data"

dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)  

after_doublet   <- ncol(seurat)   
# ============================================
# 1. 兼容 donor 列
# ============================================
if (!"donor" %in% colnames(seurat@meta.data)) {
  seurat$donor <- seurat$sample_label
  donor_colors <- NULL
  message("Note: No 'donor' column found. Using sample_label for coloring.")
} else {
  donor_colors <- c("Mq246" = "#1f77b4", "Mq277" = "#ff7f0e")
}

# ============================================
# 2. Sample-level filtering: median nFeature_RNA > 500
# ============================================
df_meta <- seurat@meta.data
group_med_feature <- df_meta %>% 
  group_by(sample_label) %>% 
  summarise(med = median(nFeature_RNA))

select_group <- group_med_feature[group_med_feature$med > 500, ]$sample_label
seurat <- subset(seurat, sample_label %in% select_group)

# ============================================
# 3. Calculate QC metrics
# ============================================
seurat$countFeatureRatio <- seurat$nCount_RNA / seurat$nFeature_RNA

MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
seurat <- NormalizeData(seurat)

oriCellNum <- ncol(seurat)

# ============================================
# 4.Prepare data for plotting
# ============================================
seurat_plot <- seurat
cat("Plotting with ALL", ncol(seurat_plot), "cells\n")

seurat_plot_vln <- subset(seurat_plot, nCount_RNA <= 25000)
cat("Violin plot using", ncol(seurat_plot_vln), "cells (nCount_RNA <= 25000)\n")

# ============================================
# 5. Define the violin plot function
# ============================================
myVlnPlot <- function(seurat_obj, feature, group.by = "sample_label", 
                      fill.by = "donor", title = NULL, ylabel = NULL) {
  meta <- seurat_obj@meta.data
  meta$group <- factor(meta[[group.by]])
  meta$fill_group <- factor(meta[[fill.by]])
  
  p <- ggplot(meta, aes(x = group, y = .data[[feature]], fill = fill_group)) +
    geom_violin(scale = "width", trim = TRUE, linewidth = 0.3, alpha = 0.85)
  
  if (!is.null(donor_colors)) {
    p <- p + scale_fill_manual(values = donor_colors, name = "Donor")
  } else {
    p <- p + scale_fill_discrete(name = fill.by)
  }
  
  p <- p + theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10, color = "black"),
      axis.text.y = element_text(size = 10, color = "black"),
      axis.title.x = element_blank(),
      axis.title.y = element_text(size = 12, face = "bold"),
      plot.title = element_text(hjust = 0.5, face = "bold", size = 13),
      legend.position = "right",
      legend.key.size = unit(0.4, "cm"),
      legend.text = element_text(size = 9),
      legend.title = element_text(size = 10, face = "bold"),
      panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5)
    ) +
    labs(title = title, y = ifelse(is.null(ylabel), feature, ylabel))
  
  if (feature == "nFeature_RNA") {
    ymax <- max(meta[[feature]], na.rm = TRUE)
    p <- p + scale_y_continuous(breaks = seq(0, ceiling(ymax / 1000) * 1000, 1000))
  }
  
  return(p)
}

# ============================================
# 6. (a) QC violin plots after doublet filtering
# ============================================
cat("\n--- Saving (a) Post-Doublet Violin ---\n")
p_a1 <- myVlnPlot(seurat_plot_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_a2 <- myVlnPlot(seurat_plot_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_a3 <- myVlnPlot(seurat_plot_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_pre <- (p_a1 + p_a2 + p_a3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Post-Doublet Removal QC Metrics", 
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "a_violin_postDoublet_byDonor.png"), width = 2000, height = 650, res = 150)
print(vln_pre)
dev.off()
cat("Saved: a_violin_postDoublet_byDonor.png\n")

# ============================================
# 7. (b) Scatter plots
# ============================================
cat("\n--- Saving (b) Post-Doublet Scatter ---\n")

scatter1 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "nFeature_RNA",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs Gene Count") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter2 <- FeatureScatter(
  seurat_plot, feature1 = "nCount_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("UMI vs MT%") + 
  theme_classic() +
  theme(legend.position = "none", plot.title = element_text(size = 12, face = "bold"))

scatter3 <- FeatureScatter(
  seurat_plot, feature1 = "nFeature_RNA", feature2 = "percent.mt",
  group.by = "sample_label", pt.size = 0.3
) + ggtitle("Gene Count vs MT%") + 
  theme_classic() +
  theme(legend.position = "right", plot.title = element_text(size = 12, face = "bold"),
        legend.key.size = unit(0.3, "cm"), legend.text = element_text(size = 8))

scatter_pre <- (scatter1 + scatter2 + scatter3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1.2)) +
  plot_annotation(title = "Post-Doublet Removal Scatter Plots",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "b_scatter_postDoublet.png"), width = 2000, height = 650, res = 150)
print(scatter_pre)
dev.off()
cat("Saved: b_scatter_postDoublet.png\n")

# ============================================
# 8. (c)Apply QC filtering
# ============================================
seurat_qc <- subset(seurat, 
                    nFeature_RNA > 700 & 
                    nCount_RNA < 100000 &
                    percent.mt < 5 & 
                    countFeatureRatio > 1.2)

cellNum1 <- ncol(seurat_qc)
cat("\nPost-QC:", cellNum1, "cells (removed", oriCellNum - cellNum1, 
    round((oriCellNum - cellNum1)/oriCellNum * 100, 1), "%)\n")

seurat_qc_vln <- subset(seurat_qc, nCount_RNA <= 25000)
cat("Post-QC Violin plot using", ncol(seurat_qc_vln), "cells (nCount_RNA <= 25000)\n")

# ============================================
# 9. (d) Violin plots after QC filtering
# ============================================
cat("\n--- Saving (c) Post-QC Violin ---\n")

p_d1 <- myVlnPlot(seurat_qc_vln, "nFeature_RNA", title = "Gene Count", ylabel = "nFeature_RNA")
p_d2 <- myVlnPlot(seurat_qc_vln, "nCount_RNA", title = "UMI Count", ylabel = "nCount_RNA")
p_d3 <- myVlnPlot(seurat_qc_vln, "percent.mt", title = "Mitochondrial %", ylabel = "percent.mt")

vln_post <- (p_d1 + p_d2 + p_d3) + 
  plot_layout(ncol = 3, widths = c(1, 1, 1)) +
  plot_annotation(title = "Post-Doublet Removal + Post-QC Metrics",
                    theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5)))

png(file.path(save_dir, "c_violin_postQC_postDoublet.png"), width = 2000, height = 650, res = 150)
print(vln_post)
dev.off()
cat("Saved: c_violin_postQC_postDoublet.png\n")

# ============================================
# 10.Save the final object
# ============================================
saveRDS(seurat_qc, paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned_QCed.rds"))

# ============================================
# 11. Summarize the results
# ============================================
cat("\n========== Summary ==========\n")
cat("Original cells:        ", original_total, "\n")
cat("After doublet removal: ", after_doublet, "\n")
cat("After sample filter:   ", oriCellNum, "\n")
cat("After QC filter:       ", cellNum1, "\n")
cat("Total removed:         ", original_total - cellNum1, 
    "(", round((original_total - cellNum1)/original_total * 100, 2), "%)\n")
cat("Save directory:        ", save_dir, "\n")
cat("Final RDS:             ", paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned_QCed.rds"), "\n")
cat("==================================\n")

In [ ]:
# Ensure that percent.mt has been calculated
if (!"percent.mt" %in% colnames(seurat@meta.data)) {
  MTGenes <- c("ND6","COX3","COX1","ND5","ND4","ND2","ND4L","ATP8","CYTB","COX2","ND3","ATP6","ND1")
  seurat[["percent.mt"]] <- PercentageFeatureSet(seurat, features = MTGenes)
}

meta <- seurat@meta.data

p <- ggplot(meta, aes(x = nCount_RNA, y = nFeature_RNA, color = percent.mt)) +
  geom_point(size = 0.3, alpha = 0.5) +
  scale_color_distiller(palette = "Reds", direction = 1, name = "percent.mt (%)") +
  labs(x = "nCount_RNA", y = "nFeature_RNA") +
  theme_classic() +
  theme(
    axis.text = element_text(size = 11, color = "black"),
    axis.title = element_text(size = 12, face = "bold"),
    legend.title = element_text(size = 10),
    legend.text = element_text(size = 9)
  )

fig_dir <- "./data"
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

png(file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), width = 800, height = 700, res = 150)
print(p)
dev.off()

cat("saved:", file.path(fig_dir, "scatter_nCount_nFeature_mt.png"), "\n")

In [ ]:
# broad cell-class assignment and random forest classification

In [ ]:
# Plot each clusters
myHighlightDimPlot = function(seuratObj, highlightBy, size=1, alpha=1, showAllPoints=FALSE, ...) {
    umapTb = as_tibble(seuratObj@reductions$umap@cell.embeddings)
    if (!identical(rownames(seuratObj@reductions$umap@cell.embeddings), rownames(seuratObj@meta.data)))
        stop("Error: cell names not identical!")
    umapTb = cbind(umapTb, seuratObj@meta.data)
    highlightByClass = sort(unique(umapTb[[highlightBy]]))
    gpList = lapply(highlightByClass, function(cl) {
        subUmapTb = subset(umapTb, umapTb[[highlightBy]] == cl)
        if (showAllPoints) {
            gp = ggplot() + 
                geom_point(data=umapTb, aes(x=UMAP_1, y=UMAP_2), color="grey", shape=16, size=size, alpha=alpha) +
                geom_point(data=subUmapTb, aes(x=UMAP_1, y=UMAP_2), color="red", shape=16, size=size, alpha=alpha) +
                xlim(min(umapTb$UMAP_1), max(umapTb$UMAP_1)) +
                ylim(min(umapTb$UMAP_2), max(umapTb$UMAP_2)) +
                coord_fixed() +
                theme_void() 
        } else {
            gp = ggplot() + 
                geom_point(data=umapTb, aes(x=UMAP_1, y=UMAP_2), color="grey", shape=16, size=size, alpha=alpha) +
                geom_point(data=subUmapTb, aes(x=UMAP_1, y=UMAP_2), color="red", shape=16, size=size, alpha=alpha) +
                xlim(min(umapTb$UMAP_1), max(umapTb$UMAP_1)) +
                ylim(min(umapTb$UMAP_2), max(umapTb$UMAP_2)) +
                coord_fixed() +
                theme_void()
        }
        return(gp)
    })
    gpAll = plot_grid(plotlist=gpList, labels=highlightByClass, ...)
    return(gpAll)
}

In [ ]:
# # 1. Update Rcpp first
# install.packages("Rcpp")

# # 2. Then restart R completely; this is required
# # RStudio: Session -> Restart R
# # or In Jupyter, select Kernel -> Restart

In [ ]:
suppressMessages(library(tidyverse))
suppressMessages(library(parallel))
suppressMessages(library(Seurat))
suppressMessages(library(harmony))
suppressMessages(library(SeuratData))
suppressMessages(library(patchwork))
suppressMessages(library(Matrix))
suppressMessages(library(viridis))
suppressMessages(library(cowplot))
suppressMessages(library(ggsci))
suppressMessages(library(pheatmap))
suppressMessages(library(RColorBrewer))
suppressMessages(library(reshape2))
suppressMessages(library(LSD))
suppressMessages(library(readxl))
suppressMessages(library(ggrepel))
suppressMessages(library(randomForest))


In [ ]:
plan("multicore", workers = 12)
options(future.globals.maxSize = 500000 * 1024^5)

In [ ]:
readDir <- "./data/"
seurat1 <- readRDS(paste0(readDir, "macaque_DR_soupx_batch1_dblCleaned_QCed.rds"))

In [ ]:
saveDir <- "./data"
dir.create(saveDir, showWarnings = FALSE, recursive = TRUE)

In [ ]:
classFiltSeurat=seurat1
classFiltSeurat

In [ ]:
classFiltSeurat = SCTransform(
classFiltSeurat, assay="RNA",
ncells=ncol(classFiltSeurat[["RNA"]]),
variable.features.n=5000,
return.only.var.genes = T,
vars.to.regress="percent.mt",
method="glmGamPoi"
)

In [ ]:
classFiltSeurat = classFiltSeurat %>%
RunPCA(npcs=100, verbose = FALSE) %>%
FindNeighbors(dims = 1:50) %>%
FindClusters(verbose = FALSE) %>%
RunUMAP(dims = 1:50)
saveRDS(classFiltSeurat,'./data/20260821batch_merge_batchharmony_pre.rds')

In [ ]:
classFiltSeurat = readRDS('./data/20260821batch_merge_batchharmony_pre.rds')
# Run harmony
DefaultAssay(classFiltSeurat) = "SCT"
options(repr.plot.width=6, repr.plot.height=2.5)

In [ ]:
classFiltSeurat@meta.data$donor = gsub('Q','q',classFiltSeurat@meta.data$donor)

In [ ]:
classFiltSeurat@meta.data

In [ ]:
classFiltSeurat = RunHarmony(
classFiltSeurat, group.by.vars="donor", plot_convergence=TRUE,
assay.use="SCT", reduction.use = "pca", dims.use=1:30
)

In [ ]:
classFiltSeurat <- classFiltSeurat %>%
  FindNeighbors(reduction = "harmony", dims = 1:30, k.param = 20) %>%   # 20 is sufficient
  FindClusters(verbose = FALSE) %>%
  RunUMAP(reduction = "harmony", dims = 1:30, n.neighbors = 20)
saveRDS(classFiltSeurat,'./data/20260821harmonydonorbatch_merge_batchharmony.rds')

In [ ]:
seurat=readRDS('./data/20260821harmonydonorbatch_merge_batchharmony.rds')

In [ ]:
# seurat@meta.data
seurat@meta.data$donor = gsub('Q','q',seurat@meta.data$donor)
seurat@meta.data

In [ ]:
# Define a custom plotting function
myHighlightDimPlot <- function(seurat, highlightBy, showAllPoints = TRUE, size = 0.1) {
  # Extract UMAP coordinates
  umap_data <- as.data.frame(seurat@reductions$umap@cell.embeddings)
  umap_data$group <- seurat@meta.data[[highlightBy]]
  
  #  Check the column names and adjust them if necessary
  colnames(umap_data)[1:2] <- c("UMAP_1", "UMAP_2")  # Ensure that column names are correct
  
  # Obtain the unique donor labels
  donors <- unique(umap_data$group)
  
  # Create a list to store the plots
  plot_list <- list()
  
  # Iterate over donors and generate one plot per donor
  for (donor in donors) {
    # Create the base plot
    p <- ggplot(umap_data, aes(x = UMAP_1, y = UMAP_2)) +
      theme_minimal() +
      theme(legend.position = "right")
    
    # Display all points if needed
    if (showAllPoints) {
      p <- p + geom_point(data = umap_data, aes(x = UMAP_1, y = UMAP_2), color = "gray", size = size)
    }
    
    # Highlight cells from the current donor
    p <- p + geom_point(data = subset(umap_data, group == donor), aes(x = UMAP_1, y = UMAP_2, color = group), size = size)
    
    # Add a title
    p <- p + ggtitle(paste("Donor:", donor))
    
    # Add the plot to the list
    plot_list[[donor]] <- p
  }
  
  # Display two plots side by side using patchwork
  combined_plot <- wrap_plots(plot_list, ncol = 2)
  
  return(combined_plot)
}

In [ ]:
options(repr.plot.width=14, repr.plot.height=7)
myHighlightDimPlot(seurat, highlightBy="donor", showAllPoints=T, size=0.1)

In [ ]:
# Define the selectMarkers vector
selectMarkers = c(
    'SLC17A6',# Excit
    "GAD1", "GAD2", # Inhib,
    "GFAP",'EDNRB', # Astrocytes
    "FOXJ1",  "RSPH1",# Ependymal
    "TAGLN", "ALPHA-SMA", # VSMC
    'COL1A1','LUM',# VLMC
    'FLT1','RAMP2',# Endothelial
    "MCAM", "RGS5", # Pericytes
    "VCAN",'SOX6', # OPC
    "PLP1",'MOBP', # Oligo
    'C1QB','OLR1',# Microglia
    'Olig2',#Polydendrocytes
    'CD52',#Lymphocytes
    'MRC1',#pvMFs
    'TAGLN',#Smooth Muscle
    'TPH2','SLC6A4',#Serotonergic Neurons
    'DRD2','TH',#Dopaminergic Neurons
    'CCK','CARTPT',#Peptidergic Neurons
    'PPP1R1B','BCL11B','PDE1B',  
    'SLC5A7','CHAT',#choli
    "SNAP25"#Neuron
)

# Remove empty elements
selectMarkers = selectMarkers[!is.null(selectMarkers) & selectMarkers != ""]

# Remove duplicates
selectMarkers = selectMarkers[!duplicated(selectMarkers)]

# Set the active identities of the Seurat object
Idents(seurat) = "seurat_clusters"

# Set plotting parameters
options(repr.plot.height=10, repr.plot.width=60)

# Draw the dot plot
DotPlot(seurat, features = selectMarkers)

In [ ]:
seurat2=seurat 

In [ ]:
seurat2@meta.data$class='Uncertain'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(0,13,34),]$class='Astrocytes'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(2,3,4,7,8,14,22,25,32),]$class='Oligo'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(20),]$class='Endothelial'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(1,27,36,31,33),]$class='OPC'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(10),]$class='Microglia'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(5,6,9,11,15,17,19,18,23,24,28,26,29,30,35,37),]$class='Neurons'
seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(16),]$class='Ependymal'
# seurat2@meta.data[seurat2@meta.data$seurat_clusters %in% c(11),]$class='5-HT'

In [ ]:
colnames(seurat2@meta.data)

In [ ]:
# Inspect broad classes
table(seurat2@meta.data$class)

# Inspect broad classes and subclasses together
head(seurat2@meta.data[, c("seurat_clusters", "class")])

In [ ]:
df_test=seurat2@reductions$umap@cell.embeddings
df_test=as.data.frame(df_test)
df_test$X=rownames(df_test)

df_meta=seurat2@meta.data
df_meta=as.data.frame(df_meta)
df_meta$X=rownames(df_meta)
df_meta=df_meta[,c('nFeature_RNA','X','seurat_clusters')]

df_test=left_join(df_test,df_meta,by='X')


Idents(seurat2) = "seurat_clusters"
options(repr.plot.width=9, repr.plot.height=7)
DimPlot(seurat2, reduction="umap", label=T,raster=FALSE)+coord_fixed()

In [ ]:
classMarkerTop = c('SLC17A6','GAD1','GAD2','SNAP25','GFAP','PECAM1','FLT1','VCAN','MOBP','C1QC','FOXJ1','SLC6A4','TH')

options(repr.plot.width=20, repr.plot.height=15)
FeaturePlot(classFiltSeurat, features = classMarkerTop,raster=FALSE)
Idents(classFiltSeurat)='class'
DotPlot(classFiltSeurat, features = classMarkerTop)

In [ ]:
unique(seurat2@meta.data[seurat2@meta.data$class %in% 'Uncertain',]$seurat_clusters)


In [ ]:
classFiltSeurat = RunHarmony(
classFiltSeurat, group.by.vars="donor", plot_convergence=TRUE,
assay.use="SCT", reduction.use = "pca", dims.use=1:30
)

In [ ]:
Idents(seurat2) = "class"
options(repr.plot.width=9, repr.plot.height=7)
DimPlot(seurat2, reduction="umap", label=T,raster=FALSE)+coord_fixed()

In [ ]:
classFiltSeurat=seurat1
classFiltSeurat

In [ ]:
# Count cells in each class
cell_counts <- table(seurat2@meta.data$class)

# Print the result
print(cell_counts)

In [ ]:
# Sample 80% of cells from each class, capped at 3,000 cells per class
sampleRatio = 0.8
maxCell = 3000
sampleDf = seurat2@meta.data[, c("class"), drop=F]
sampleList = split(sampleDf, sampleDf$class)
sampleList = lapply(sampleList, function(x) {
    sampleSize = as.integer(nrow(x)*0.8)
    if (sampleSize > maxCell) {
        sampleSize = maxCell
    }
    sampleIdx = 1:nrow(x)
    sampleIdx = sample(1:nrow(x), size=sampleSize, replace=F)
    return(x[sampleIdx,,drop=F])
})
sampleDf = Reduce(rbind, sampleList)
str(sampleDf)
table(sampleDf$class)

trainSeurat = seurat2[,rownames(sampleDf)]
table(trainSeurat$class)

In [ ]:
trainSeurat$class %>% unique()

In [ ]:
Idents(trainSeurat) = "class"
classMarker = FindAllMarkers(trainSeurat)

In [ ]:
#classMarker = read_tsv(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.downsample5000ClassMarker.20220822.tsv"))
classMarker = subset(classMarker, p_val_adj < 0.01 & avg_log2FC > 1)
table(classMarker$cluster)

classMarkerTop = classMarker %>% group_by(cluster) %>% top_n(n = 20, wt = avg_log2FC)
options(repr.plot.width=length(unique(classMarkerTop$gene))/3, repr.plot.height=5)
Idents(seurat) = "class"

In [ ]:
seurat3 = seurat

#Reorder group identities, for example into a specified order
new_levels <- c("Endothelial",'OPC','Microglia','Oligo','Astrocytes','Neurons','Uncertain')
Idents(seurat3) <- factor(Idents(seurat3), levels = new_levels)

# Verify the new group order
levels(Idents(seurat3))

In [ ]:
trainDf = as.data.frame(t(as.matrix(trainSeurat[["SCT"]]@data[unique(classMarker$gene), ])))
trainDf$class = trainSeurat$class
trainDf = subset(trainDf, class != "Uncertain")
trainDf$class = factor(trainDf$class)
# We need to modify the name of the columns
colnames(trainDf) = paste0("col_", colnames(trainDf))
colnames(trainDf) = gsub("-", "_", colnames(trainDf))

rf = randomForest(col_class~., data=trainDf, ntree=1000)

In [ ]:
# Sample 80% of cells from each class, capped at 3,000 cells per class
sampleRatio = 0.8
maxCell = 3000
sampleDf = seurat2@meta.data[, c("class"), drop=F]
sampleList = split(sampleDf, sampleDf$class)
sampleList = lapply(sampleList, function(x) {
    sampleSize = as.integer(nrow(x)*0.8)
    if (sampleSize > maxCell) {
        sampleSize = maxCell
    }
    sampleIdx = 1:nrow(x)
    sampleIdx = sample(1:nrow(x), size=sampleSize, replace=F)
    return(x[sampleIdx,,drop=F])
})
sampleDf = Reduce(rbind, sampleList)
str(sampleDf)
table(sampleDf$class)

trainSeurat = seurat2[,rownames(sampleDf)]
table(trainSeurat$class)

In [ ]:
dim(rf$confusion)
colnames(rf$confusion)
rownames(rf$confusion)

In [ ]:
# Load the required packages
library(ggplot2)
library(pheatmap)

# Assume that rf is the fitted random forest model object
# rf <- readRDS(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.class.randomForest.20220822.rds"))

# Inspect the structure of the confusion matrix
str(rf$confusion)

# Inspect the structure of the confusion matrix
str(rf$confusion)

# Set plotting parameters
options(repr.plot.width=10, repr.plot.height=5)
plot(rf)

# Calculate proportions from the confusion matrix
confusionRatio <- t(apply(rf$confusion, 1, function(x) {x / sum(x)}))

# Set heatmap parameters
options(repr.plot.width=10, repr.plot.height=10)
pheatmap(confusionRatio, 
         breaks=seq(0, 1, 0.01), 
         cluster_rows=FALSE, 
         cluster_cols=FALSE, 
         border_color=NA, 
         cellwidth=30, 
         cellheight=30, 
         fontsize=15)

In [ ]:
#rf = readRDS(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.class.randomForest.20220822.rds"))
str(rf$confusion)
options(repr.plot.width=10, repr.plot.height=5)
plot(rf)

confusionRatio = t(apply(rf$confusion, 1, function(x) {x/sum(x)}))
options(repr.plot.width=10, repr.plot.height=10)
pheatmap(confusionRatio, breaks=seq(0,1,0.01), cluster_rows=F, cluster_cols=F, border_color=NA, cellwidth=30, cellheight=30, fontsize=15)

In [ ]:
# Remove the class.error column
confusionMatrix <- rf$confusion[, -which(colnames(rf$confusion) == "class.error")]

# Inspect the matrix after removing that column
str(confusionMatrix)
# Calculate proportions from the confusion matrix
confusionRatio = t(apply(confusionMatrix, 1, function(x) {x/sum(x)}))

# Draw the heatmap
pheatmap(confusionRatio, breaks=seq(0,1,0.01), cluster_rows=F, cluster_cols=F, border_color=NA, cellwidth=30, cellheight=30, fontsize=15)

In [ ]:
# Load the required libraries
library(ggplot2)
library(pheatmap)

# Assume that rf contains the fitted random forest classification result
# Uncomment the following code if the data need to be loaded
# dataDir <- "./data"
# rf <- readRDS(paste0(dataDir, "/allSnRNAseq.soupx.mergeSeuratFiltlibFilt800mt5Ratio1d2.class.randomForest.20220822.rds"))

# Inspect the structure of the confusion matrix
str(rf$confusion)

# Plot the random forest confusion matrix
options(repr.plot.width = 10, repr.plot.height = 5)
plot(rf)

# Remove the class.error column from the confusion matrix
confusion_matrix <- rf$confusion
if ("class.error" %in% colnames(confusion_matrix)) {
  confusion_matrix <- confusion_matrix[, !colnames(confusion_matrix) %in% c("class.error")]
}

#  Calculate row-wise proportions for the confusion matrix
confusionRatio <- t(apply(confusion_matrix, 1, function(x) { x / sum(x) }))

# Draw a heatmap of the confusion matrix and display the numeric values
options(repr.plot.width = 10, repr.plot.height = 10)
pheatmap(
  confusionRatio,
  breaks = seq(0, 1, 0.01), #  Set color breaks
  cluster_rows = FALSE, # Do not cluster rows
  cluster_cols = FALSE, # Do not cluster columns
  border_color = NA, # Remove cell border colors
  cellwidth = 30, # Cell width
  cellheight = 30, # Cell height
  fontsize = 15, # Font size
  color = colorRampPalette(c("#e65200", "#fffde7"))(100), # Set the color gradient
  show_rownames = TRUE, # Show row names
  show_colnames = TRUE, # Show column names
  main = "Confusion Matrix Heatmap", # Set the title
  legend = TRUE, # Show the legend
  display_numbers = TRUE, # Display numeric values in cells
  number_format = "%.2f" # Numeric format with two decimal places
)

In [ ]:
# Apply the trainer to filter cells
seuratCountDf = as.data.frame(t(as.matrix(seurat[["SCT"]]@data[unique(classMarker$gene), ])))
colnames(seuratCountDf) = paste0("col_", colnames(seuratCountDf))
colnames(seuratCountDf) = gsub("-", "_", colnames(seuratCountDf))
seuratClassPred = predict(rf, seuratCountDf, type="prob")
str(seuratClassPred)

seuratClassPredTb = tibble(
    cellId = rownames(seuratClassPred),
    maxPredClassScore = apply(seuratClassPred, 1, max),
    maxPredClass = apply(seuratClassPred, 1, function(x) {names(x)[which.max(x)]})
)
head(seuratClassPredTb)

summary(seuratClassPredTb$maxPredClassScore)
options(repr.plot.width=6, repr.plot.height=4)
plot(density(seuratClassPredTb$maxPredClassScore, adjust=0.1))

identical(seuratClassPredTb$cellId, rownames(seurat@meta.data))
tmpDf = as.data.frame(seuratClassPredTb[c("maxPredClassScore", "maxPredClass")])
rownames(tmpDf) = seuratClassPredTb$cellId

seurat = AddMetaData(seurat, tmpDf)

options(repr.plot.width=9, repr.plot.height=7)
FeaturePlot(seurat, reduction="umap", features="maxPredClassScore",raster=FALSE)+coord_fixed()


In [ ]:
# 1. Check whether removed cells are concentrated in particular clusters
seurat$passFilt <- seurat$maxPredClassScore >= 0.9
table(seurat$seurat_clusters, seurat$passFilt)

# 2.  Compare QC metrics between retained and removed cells
VlnPlot(seurat, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "passFilt", pt.size = 0)

In [ ]:
# 1. Check whether removed cells are concentrated in particular clusters
seurat$passFilt <- seurat$maxPredClassScore >= 0.8
table(seurat$seurat_clusters, seurat$passFilt)

# 2.  Compare QC metrics between retained and removed cells
VlnPlot(seurat, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "passFilt", pt.size = 0)

In [ ]:
Idents(seurat) = "seurat_clusters"
cellNum0 = ncol(seurat)
classFiltSeurat = subset(seurat, maxPredClassScore >= 0.8)
cellNum1 = ncol(classFiltSeurat)
message("Filt cell ratio: ", cellNum1/cellNum0)

In [ ]:
seurat
classFiltSeurat

In [ ]:
table(classFiltSeurat$maxPredClass)

In [ ]:
saveRDS(classFiltSeurat,'./data/20260821subclass3macaque_snrna_batch1_soupX-forest.rds')

In [ ]:
classFiltSeurat=readRDS('./data/20260821subclass3macaque_snrna_batch1_soupX-forest.rds')

In [ ]:
# Step 4: celltype annotation

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)  

In [ ]:
library(Seurat)
library(dplyr)

# ========== 1. Extract Neurons directly from maxPredClass ==========
nd_seurat <- subset(classFiltSeurat, subset = maxPredClass == "Neurons")
cat("Neurons 细胞数:", ncol(nd_seurat), "\n")

# ========== 2. Re-identify highly variable genes and rerun PCA within Neurons ==========
nd_seurat <- FindVariableFeatures(nd_seurat, selection.method = "vst", nfeatures = 3000)
nd_seurat <- RunPCA(nd_seurat, features = VariableFeatures(nd_seurat), npcs = 50, verbose = FALSE)

# ========== 3. Cluster cells and run UMAP ==========
nd_seurat <- FindNeighbors(nd_seurat, reduction = "pca", dims = 1:30)
nd_seurat <- FindClusters(nd_seurat, resolution = 0.2)   # Adjust to your needs
nd_seurat <- RunUMAP(nd_seurat, reduction = "pca", dims = 1:30)

cat("new cluster:\n")
print(table(nd_seurat$seurat_clusters))

# ========== 4. Rename clusters as Neurons_0, Neurons_1, and so on ==========
clusters <- as.character(nd_seurat$seurat_clusters)
unique_clusters <- sort(unique(as.numeric(clusters)))

neuron_counter <- 0
cluster_map <- character(length(unique_clusters))
names(cluster_map) <- as.character(unique_clusters)

for(cl in as.character(unique_clusters)) {
  cluster_map[cl] <- paste0("Neurons_", neuron_counter)
  neuron_counter <- neuron_counter + 1
}

nd_seurat$Subclass_new <- unname(cluster_map[clusters])
cat("\nrename:\n")
print(table(nd_seurat$Subclass_new))

# ========== 5. Visualization ==========
options(repr.plot.width = 14, repr.plot.height = 10)
DimPlot(nd_seurat, reduction = "umap", group.by = "Subclass_new",
        label = TRUE, repel = TRUE, raster = FALSE) + coord_fixed()

In [ ]:
# 1. Create a cluster column in classFiltSeurat, using the broad class as the default
classFiltSeurat$cluster <- as.character(classFiltSeurat$maxPredClass)

# 2. Write the Neurons subtype labels back to the full object
nd_cells <- colnames(nd_seurat)
classFiltSeurat$cluster[nd_cells] <- as.character(nd_seurat$Subclass_new)

# 3. Check the result
table(classFiltSeurat$cluster)

In [ ]:
output_path <- "./data/"

In [ ]:
print(output_path)

In [ ]:
# ========== Place this section before all plotting code==========
dir.create(output_path, showWarnings = FALSE, recursive = TRUE)


In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

# ========== Set paths==========
output_path <- "./data/"
dir.create(output_path, showWarnings = FALSE, recursive = TRUE)

# ============================================
# Extract only Neurons subtypes using the previously generated nd_seurat object
# ============================================

# Ensure that nd_seurat exists and contains Subclass_new
# If necessary, recreate the subset from classFiltSeurat as follows
# nd_seurat <- subset(classFiltSeurat, subset = grepl("^Neurons_", cluster))

cat("Neurons cell:", ncol(nd_seurat), "\n")
cat("Neurons celltype:", length(unique(nd_seurat$Subclass_new)), "\n")

# ============================================
# 1. UMAP overview of Neurons subtypes
# ============================================
cat("\n--- 1. Neurons UMAP ---\n")

p_umap <- DimPlot(nd_seurat, reduction = "umap", group.by = "Subclass_new",
                  label = TRUE, repel = TRUE, raster = FALSE, pt.size = 0.3) +
  coord_fixed() +
  ggtitle("Neurons Subclusters") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14))

print(p_umap)
ggsave(file.path(output_path, "Neurons_only_UMAP.png"), 
       plot = p_umap, width = 14, height = 10, dpi = 300)

# ============================================
# 2. Canonical marker genes
# ============================================
cat("\n--- 2. Classic Markers (Neurons only) ---\n")

marker_genes <- c("SLC17A6","SLC17A7" ,"SLC6A4", "TPH2", "GAD1", "GAD2", "TH", "SLC17A8", "SNAP25",'GFAP','PECAM1','FLT1','VCAN','MOBP','C1QC','FOXJ1')
avail_genes <- intersect(marker_genes, rownames(nd_seurat))

if (length(avail_genes) > 0) {
  
  # FeaturePlot on the Neurons UMAP
  p_feature <- FeaturePlot(nd_seurat, features = avail_genes, ncol = 3,
                           pt.size = 0.3, cols = c("lightgrey", "darkblue"),
                           order = TRUE, raster = FALSE) &
    theme(plot.title = element_text(face = "bold.italic", size = 12))
  print(p_feature)
  ggsave(file.path(output_path, "Neurons_only_FeaturePlot.png"), 
         plot = p_feature, width = 15, height = 12, dpi = 300)
  
  # DotPlot grouped by Neurons subtype
  # With 56 subtypes, the y-axis is long and the plot height must be increased
  p_dot_classic <- DotPlot(nd_seurat, features = avail_genes, 
                           group.by = "Subclass_new",
                           dot.scale = 8, cols = c("lightgrey", "darkblue")) +
    RotatedAxis() +
    ggtitle("Neurons Classic Markers") +
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 11, face = "italic"),
          axis.text.y = element_text(size = 6),   # 56 subtypes; reduce font size
          plot.title = element_text(hjust = 0.5, face = "bold", size = 13))
  print(p_dot_classic)
  ggsave(file.path(output_path, "Neurons_only_ClassicMarkers_DotPlot.png"), 
         plot = p_dot_classic, width = 10, height = 20, dpi = 300)  # Increase the height
  
  # VlnPlot grouped by Neurons subtype
  p_vln <- VlnPlot(nd_seurat, features = avail_genes, 
                   group.by = "Subclass_new",
                   pt.size = 0, ncol = 3) &
    theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 7),
          axis.text.y = element_text(size = 6),
          plot.title = element_text(face = "bold.italic", size = 12))
  print(p_vln)
  ggsave(file.path(output_path, "Neurons_only_ClassicMarkers_VlnPlot.png"), 
         plot = p_vln, width = 16, height = 24, dpi = 300)  # Increase the height
  
  cat("✅ Neurons Canonical marker genes\n")
} else {
  cat("⚠️ none\n")
}

In [ ]:
# ============================================
# 3. Run FindAllMarkers to compare Neurons subtypes
# ============================================
cat("\n--- 3. FindAllMarkers (Neurons only) ---\n")

DefaultAssay(nd_seurat) <- "RNA"
Idents(nd_seurat) <- "Subclass_new"

markers <- FindAllMarkers(
  nd_seurat,
  assay = "RNA",
  only.pos = TRUE,
  min.pct = 0.25,
  logfc.threshold = 0.25,
  verbose = TRUE
)
cat(" markers numbers:", nrow(markers), "\n")

# Save
write.csv(markers, file.path(output_path, "Neurons_only_FindAllMarkers.csv"), row.names = FALSE)

# ============================================
# 4. Top20 Markers DotPlot
# ============================================
cat("\n--- 4. Top20 Markers DotPlot ---\n")

if (nrow(markers) > 0) {
  top20 <- markers %>%
    group_by(cluster) %>%
    slice_max(order_by = avg_log2FC, n = 20) %>%
    ungroup()
  
  top20_genes <- unique(top20$gene)
  top20_genes <- top20_genes[top20_genes %in% rownames(nd_seurat)]
  
  if (length(top20_genes) > 0) {
    gene_order <- top20 %>% 
      filter(gene %in% top20_genes) %>% 
      arrange(cluster) %>% 
      pull(gene) %>% 
      unique()
    
    p_dot_top20 <- DotPlot(
      nd_seurat,
      features = gene_order,
      group.by = "Subclass_new",
      dot.scale = 4,
      cols = c("lightgrey", "red")
    ) +
      RotatedAxis() +
      ggtitle("Neurons Top20 Markers per Subcluster") +
      theme(
        axis.text.x = element_text(angle = 45, hjust = 1, size = 7),
        axis.text.y = element_text(size = 6),
        plot.title = element_text(hjust = 0.5, face = "bold", size = 12)
      )
    
    print(p_dot_top20)
    ggsave(
      file.path(output_path, "Neurons_only_Top20Markers_DotPlot.png"),
      plot = p_dot_top20,
      width = 50,
      height = 24,
      dpi = 300,
      limitsize = FALSE
    )
    cat("✅ Top20 Markers dotplot saved\n")
  }
}

In [ ]:
# ==========================================================================
# Complete script for violin plots of neuronal subtype markers
# Addresses a VlnPlot S4SXP error and the absence of a graphics device in a headless environment
# Use FetchData to retrieve values, construct plots manually with ggplot2, and save PDF files preferentially with ggsave
# Restart the R session, then run this script directly with source or Rscript 
# ==========================================================================

suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
})

# ============================ 0.Parameter section; modify as needed ============================
group_col   <- "Subclass_new"
output_path <- "./data/"
markers_csv <- file.path(output_path, "Neurons_only_FindAllMarkers.csv")
# nd_seurat_rds <- "/path/to/nd_seurat.rds"   # If the object is not in memory, set the path and uncomment the next line
# nd_seurat <- readRDS(nd_seurat_rds)

dir.create(output_path, showWarnings = FALSE, recursive = TRUE)

# ============================ 1. Object checks ============================
if (!exists("nd_seurat")) stop("nd_seurat none, readRDS")
stopifnot(group_col %in% colnames(nd_seurat@meta.data))
Idents(nd_seurat) <- group_col

cat("== Environment diagnostics ==\n")
cat("Interactive:", interactive(), " X11:", capabilities("X11"), " cairo:", capabilities("cairo"), "\n")
cat("Cells:", ncol(nd_seurat), " Genes:", nrow(nd_seurat), "\n")
cat("Group levels:", paste(sort(unique(nd_seurat@meta.data[[group_col]])), collapse = ", "), "\n")

if (!exists("markers")) {
  if (file.exists(markers_csv)) {
    markers <- read.csv(markers_csv, stringsAsFactors = FALSE)
    cat("Markers loaded from CSV:", nrow(markers), "rows\n")
  } else {
    stop("'markers' not found in memory, and file not found: ", markers_csv)
  }
}

# ============================ 2. Define marker panels============================

# ---- 2.1 Neurotransmitter identity panel ----
nt_markers <- c(
  "SLC17A6", "SLC17A7", "SLC17A8",   # Glutamatergic (SLC17A8 is also a marker of 5-HT co-release)
  "GAD1",    "GAD2",    "SLC32A1",   # GABA
  "CHAT",    "SLC18A3", "SLC5A7",    # Cholinergic
  "TH",      "SLC6A3",  "NR4A2",     # Dopaminergic
  "TPH2",    "SLC6A4",  "DDC",       # 5-HT
  "DBH",     "SLC6A2"                # Noradrenergic
)

# ---- 2.2 Marker panel for neighboring regional identities ----
region_mk_list <- list(
  `5HT_raphe`   = c("TPH2","SLC6A4","FEV","SLC17A8","LMX1B","GATA2"),
  `PAG`         = c("PAX7","PAX8","ESR1","NTN1","DPP10","GUCY1A3"),
  `PBN`         = c("EGR2","PRLR","CALCA","NMB","DRD3"),
  `LC_NE`       = c("DBH","SLC6A2","PHOX2B","GATA3"),
  `VTA_SNc_DA`  = c("TH","NR4A2","PITX3","SLC6A3","SOX6","KCNJ6"),
  `ACh_IPN_MHb` = c("CHAT","SLC18A3","SLC5A7","GPR151","TACR3"),
  `Striatum`    = c("BCL11B","FOXP2","PTPN5","TAC1","ADORA2A","DRD1","DRD2"),
  `Ctx_Hip_GABA`= c("PVALB","SST","VIP","LAMP5","NPY")
)

# ============================ 3. Core functions ============================

# ---- 3.1 Retrieve gene expression or metadata columns, including score columns, and convert them to long format ----
get_expr_df <- function(features, obj = nd_seurat, gc = group_col) {
  dat <- tryCatch(
    FetchData(obj, vars = c(features, gc)),
    error = function(e) NULL
  )
  if (is.null(dat)) {
    # Fallback when FetchData fails: retrieve values directly from the expression matrix; this supports genes only, not metadata columns
    mat <- tryCatch(GetAssayData(obj, assay = "RNA", layer = "data"),
                    error = function(e) GetAssayData(obj, assay = "RNA", slot = "data"))
    genes <- intersect(features, rownames(mat))
    dat <- as.data.frame(t(as.matrix(mat[genes, , drop = FALSE])))
    dat[[gc]] <- obj@meta.data[[gc]]
  }
  feats_ok <- intersect(features, colnames(dat))
  dat %>%
    select(all_of(c(feats_ok, gc))) %>%
    pivot_longer(cols = all_of(feats_ok), names_to = "gene", values_to = "expr")
}

# ---- 3.2 Draw and save violin plots; always save PDF and attempt to save PNG ----
plot_vln_manual <- function(features, file_base, ncol = 4, same.y = TRUE,
                            title = NULL, width = NULL, height = NULL) {
  df <- get_expr_df(features)
  genes <- intersect(features, unique(df$gene))
  if (length(genes) == 0) {
    cat("!! no gene，skip:", basename(file_base), "\n")
    return(invisible(NULL))
  }
  df <- df %>% mutate(gene = factor(gene, levels = genes))
  n_row <- ceiling(length(genes) / ncol)
  if (is.null(width))  width  <- 3.0 * ncol
  if (is.null(height)) height <- 2.8 * n_row

  p <- ggplot(df, aes(x = .data[[group_col]], y = expr, fill = .data[[group_col]])) +
    geom_violin(scale = "width", trim = TRUE) +
    facet_wrap(~gene, ncol = ncol, scales = ifelse(same.y, "fixed", "free_y")) +
    theme_bw(base_size = 8) +
    theme(axis.text.x = element_text(angle = 45, hjust = 1),
          legend.position = "none",
          strip.background = element_rect(fill = "grey90")) +
    labs(x = NULL, y = "Expression", title = title)

  ggsave(paste0(file_base, ".pdf"), p, width = width, height = height, limitsize = FALSE)
  ok_png <- tryCatch({
    ggsave(paste0(file_base, ".png"), p, width = width, height = height,
           dpi = 300, limitsize = FALSE)
    TRUE
  }, error = function(e) FALSE)
  cat(">> Done:", basename(file_base), "(pdf saved", ifelse(ok_png, ", png saved)", ")"), "\n")
  invisible(p)
}

# ---- 3.3 Regional scoring, with a manual fallback if AddModuleScore fails ----
add_scores_safe <- function(obj, sig_list) {
  obj <- tryCatch({
    obj <- AddModuleScore(obj, features = sig_list, name = "RS", seed = 1)
    for (i in seq_along(sig_list)) {
      obj@meta.data[[names(sig_list)[i]]] <- obj@meta.data[[paste0("RS", i)]]
    }
    cat("AddModuleScore completed successfully\n")
    obj
  }, error = function(e) {
    cat("!! AddModuleScore failed, falling back to manual scoring:", conditionMessage(e), "\n")
    for (nm in names(sig_list)) {
      genes <- intersect(sig_list[[nm]], rownames(obj))
      if (length(genes) == 0) next
      expr <- FetchData(obj, vars = genes)
      obj@meta.data[[nm]] <- rowMeans(expr)
    }
    obj
  })
  obj
}

# ============================ 4. Generate the final plots ============================

# ---- 4.1 Neurotransmitter identity overview ----
cat("\n--- 4.1 Neurotransmitter identity violin plots ---\n")
plot_vln_manual(nt_markers,
                file.path(output_path, "Neurons_only_NeuroTransmitter_VlnPlot"),
                ncol = 4, same.y = TRUE,
                title = "Neurotransmitter identity")

# ---- 4.2 Adjacent-region markers ----
cat("\n--- 4.2 Adjacent-region marker violin plots ---\n")
plot_vln_manual(unique(unlist(region_mk_list)),
                file.path(output_path, "Neurons_only_RegionMarkers_VlnPlot"),
                ncol = 5, same.y = FALSE,
                title = "Adjacent-region markers")

# ---- 4.3 Region module scoring ----
cat("\n--- 4.3 Region module scoring ---\n")
nd_seurat <- add_scores_safe(nd_seurat, region_mk_list)
plot_vln_manual(names(region_mk_list),
                file.path(output_path, "Neurons_only_RegionScore_VlnPlot"),
                ncol = 4, same.y = FALSE,
                title = "Region module scores")

# ---- 4.4 Top20 markers per subclass ----
cat("\n--- 4.4 Top20 markers per subclass ---\n")
top20 <- markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20, with_ties = FALSE) %>%
  ungroup()

for (cl in sort(unique(top20$cluster))) {
  genes_cl <- top20 %>%
    filter(cluster == cl) %>%
    arrange(desc(avg_log2FC)) %>%
    pull(gene)
  ncol_use <- min(5, max(length(intersect(genes_cl, rownames(nd_seurat))), 1))
  nrow_use <- ceiling(length(intersect(genes_cl, rownames(nd_seurat))) / ncol_use)
  plot_vln_manual(genes_cl,
                  file.path(output_path, paste0("Neurons_only_Top20_VlnPlot_", cl)),
                  ncol = ncol_use, same.y = FALSE,
                  width = 3.2 * ncol_use, height = 3.2 * nrow_use,
                  title = paste0("Top20 markers of ", cl))
}

# ============================ 5. Completion check ============================
cat("\n== All done. Output files: ==\n")
print(list.files(output_path, pattern = "VlnPlot", full.names = TRUE))

In [ ]:
library(Seurat)
library(dplyr)
library(pheatmap)

group_col <- "Subclass_new"
clusters_check <- c("Neurons_5","Neurons_9","Neurons_10","Neurons_11","Neurons_12",
                    "Neurons_13","Neurons_14","Neurons_15","Neurons_16","Neurons_17",
                    "Neurons_18","Neurons_20","Neurons_21","Neurons_22","Neurons_8")

sigs <- list(
  Glut   = c("SLC17A6","SLC17A7"),
  GABA   = c("GAD1","GAD2","SLC32A1"),
  DA     = c("TH","SLC6A3","NR4A2","PITX3","LMX1A"),
  X5HT   = c("TPH2","SLC6A4","FEV","LMX1B","GATA2","SLC17A8","DDC"),
  NE     = c("DBH","SLC6A2","PHOX2B","GATA3"),
  ACh    = c("CHAT","SLC18A3","SLC5A7"),
  PAG        = c("PAX7","PAX8","ESR1","DPP10","NTN1"),
  PBN        = c("EGR2","PRLR","CALCA","NMB","PDYN"),
  RTN        = c("SHOX2","PHOX2B","MEIS1"),
  PonsOlive  = c("EBF2","EBF1","OTOF","NFIB","TAC1","CALB2"),
  Striatum   = c("PTPN5","PDE1B","TAC1","PENK","FOXP2","BCL11B","ADORA2A"),
  Oligo   = c("PLP1","MBP","OPALIN","MOBP","MAG"),
  Astro   = c("AQP4","SLC1A3","GJA1","ALDH1A1","SPARCL1"),
  Endo    = c("CLDN5","VWF","FLT1","PECAM1"),
  Micro   = c("AIF1","CSF1R","CX3CR1","C1QA","FCGR3A"),
  ChoriEpend = c("TTR","KRT8","KRT18","FOXJ1"),
  Pericyte= c("RGS5","ABCC9","KCNJ8","PDGFRB")
)

# ---- Average expression (v4/v5 compatible) ----
all_genes <- unique(unlist(sigs))
missing_genes <- setdiff(all_genes, rownames(nd_seurat))
if (length(missing_genes) > 0) {
  cat("The following genes are not in the RNA assay and were automatically removed from the signatures:",
      paste(missing_genes, collapse = ", "), "\n")
}

avg <- tryCatch(
  AverageExpression(nd_seurat, assays = "RNA", layer = "data",
                    features = all_genes, group.by = group_col)$RNA,
  error = function(e)
    AverageExpression(nd_seurat, assays = "RNA", slot = "data",
                      features = all_genes, group.by = group_col)$RNA
)

# ---- Key fix: restore the "_" that Seurat replaced ----
colnames(avg) <- gsub("-", "_", colnames(avg))

matched <- intersect(clusters_check, colnames(avg))
cat("Matched clusters:", length(matched), "/", length(clusters_check), "\n")
not_matched <- setdiff(clusters_check, colnames(avg))
if (length(not_matched) > 0) {
  cat("Not matched (ID may not exist or spelling differs):", paste(not_matched, collapse = ", "), "\n")
}
stopifnot("At least one cluster matched" = length(matched) > 0)

avg <- avg[, matched, drop = FALSE]

# ---- z-score ----
m <- as.matrix(avg)
z <- t(scale(t(m)))
z[!is.finite(z)] <- 0   # handle NaN caused by genes with identical expression across all clusters

# ---- signature scores ----
sig_score <- sapply(sigs, function(g) {
  g <- intersect(g, rownames(z))
  if (length(g) == 0) return(setNames(rep(0, ncol(z)), colnames(z)))
  v <- colMeans(z[g, , drop = FALSE], na.rm = TRUE)
  v[is.na(v)] <- 0
  v
})
sig_score <- t(sig_score)
sig_score[is.na(sig_score)] <- 0

# ---- Automatic annotation table ----
transmitter <- c("Glut","GABA","DA","X5HT","NE","ACh")
region      <- c("PAG","PBN","RTN","PonsOlive","Striatum")
glial       <- c("Oligo","Astro","Endo","Micro","ChoriEpend","Pericyte")

anno <- lapply(colnames(sig_score), function(cl) {
  tibble(
    cluster      = cl,
    transmitter  = transmitter[which.max(sig_score[transmitter, cl])],
    region_hit   = region[which.max(sig_score[region, cl])],
    region_score = round(max(sig_score[region, cl]), 2),
    glial_hit    = glial[which.max(sig_score[glial, cl])],
    glial_score  = round(max(sig_score[glial, cl]), 2),
    verdict = case_when(
      max(sig_score[glial, cl]) > 1.0   ~ paste0("non-neuronal/doublet (", glial[which.max(sig_score[glial, cl])], ")"),
      max(sig_score[region, cl]) > 1.0   ~ paste0("possible ", region[which.max(sig_score[region, cl])], " contamination"),
      max(sig_score[transmitter, cl]) < 0 ~ "all transmitter markers negative, manual review needed",
      TRUE                                ~ "likely raphe-local"
    )
  )
}) %>% bind_rows()

print(as.data.frame(anno))
write.csv(anno, file.path(output_path, "cluster_auto_annotation.csv"), row.names = FALSE)

# ---- Heatmap ----
pdf(file.path(output_path, "cluster_signature_heatmap.pdf"), width = 11, height = 8)
pheatmap(sig_score, fontsize_row = 8, fontsize_col = 8,
         main = "Signature scores by cluster (row z-score)",
         cluster_cols = FALSE)
dev.off()
cat("Done, results saved\n")

In [ ]:
#Neighboring region of raphe (NE) — Neurons_22 | NE | DBH, PHOX2A, SLC6A2, TH → Locus coeruleus (LC) or A1/A2 noradrenergic nuclei
#Neighboring region of raphe (ACh) — Neurons_13, 18 | ACh | CHAT, SLC5A7 → PPN/LDT cholinergic nuclei (pedunculopontine / laterodorsal tegmental nucleus)
#Neighboring region of raphe (PAG) — Neurons_6, 8 | PAG | PAX3, PAX7, OTP → Periaqueductal gray (PAG)

In [ ]:
library(Seurat)
library(harmony)

In [ ]:
library(Seurat)
library(dplyr)

# ========== 1. Define the Neuron subclusters to be removed ==========
remove_neurons <- c("Neurons_11", "Neurons_12", "Neurons_13", "Neurons_16", "Neurons_22","Neurons_20", "Neurons_9", "Neurons_5", "Neurons_14", "Neurons_10")

# ========== 2. Remove these cells from classFiltSeurat ==========
cells_to_remove <- colnames(classFiltSeurat)[classFiltSeurat$cluster %in% remove_neurons]
cat("Number of cells to remove:", length(cells_to_remove), "\n")

# Create a new Seurat object (with the specified cells removed)
classFiltSeurat_filt <- subset(classFiltSeurat, cells = setdiff(colnames(classFiltSeurat), cells_to_remove))

cat("Total cells before removal:", ncol(classFiltSeurat), "\n")
cat("Total cells after removal:", ncol(classFiltSeurat_filt), "\n")

# ========== 3. Check cluster distribution after removal ==========
cat("\nCluster distribution after removal:\n")
print(table(classFiltSeurat_filt$cluster))

# ========== 4. Re-run dimensionality reduction on the Neurons subset (optional, if re-integration is needed) ==========
# Extract Neurons cells
neuron_clusters <- grep("^Neurons_", unique(classFiltSeurat_filt$cluster), value = TRUE)
nd_cells_filt <- colnames(classFiltSeurat_filt)[classFiltSeurat_filt$cluster %in% neuron_clusters]

nd_seurat_filt <- subset(classFiltSeurat_filt, cells = nd_cells_filt)

cat("\nRemaining Neurons cells:", ncol(nd_seurat_filt), "\n")

# Re-identify highly variable genes + PCA
nd_seurat_filt <- FindVariableFeatures(nd_seurat_filt, selection.method = "vst", nfeatures = 3000)
nd_seurat_filt <- RunPCA(nd_seurat_filt, features = VariableFeatures(nd_seurat_filt), npcs = 50, verbose = FALSE)

# New: donor-level batch correction
nd_seurat_filt <- RunHarmony(nd_seurat_filt,
                             group.by.vars = "donor",
                             reduction.use = "pca",
                             dims.use = 1:30,
                             assay.use = "RNA",
                             verbose = FALSE)

nd_seurat_filt <- FindNeighbors(nd_seurat_filt, reduction = "harmony", dims = 1:30)
nd_seurat_filt <- FindClusters(nd_seurat_filt, resolution = 0.2)
nd_seurat_filt <- RunUMAP(nd_seurat_filt, reduction = "harmony", dims = 1:30)

# Rename as Neurons_0, Neurons_1... (keep numbering consecutive)
clusters <- as.character(nd_seurat_filt$seurat_clusters)
unique_clusters <- sort(unique(as.numeric(clusters)))

neuron_counter <- 0
cluster_map <- character(length(unique_clusters))
names(cluster_map) <- as.character(unique_clusters)

for(cl in as.character(unique_clusters)) {
  cluster_map[cl] <- paste0("Neurons_", neuron_counter)
  neuron_counter <- neuron_counter + 1}

nd_seurat_filt$Subclass_new <- unname(cluster_map[clusters])

cat("\n重新聚类后的分布:\n")
print(table(nd_seurat_filt$Subclass_new))

# ========== 5. Update Neurons labels in classFiltSeurat_filt==========
# Keep the original labels for non-Neurons cells
classFiltSeurat_filt$cluster_final <- as.character(classFiltSeurat_filt$cluster)

# Assign the new labels to Neurons cells
nd_cells_new <- colnames(nd_seurat_filt)
classFiltSeurat_filt$cluster_final[nd_cells_new] <- as.character(nd_seurat_filt$Subclass_new)

cat("\n final cluster:\n")
print(table(classFiltSeurat_filt$cluster_final))

# ========== 6.Visualization ==========
#UMAP of the Neurons subset
options(repr.plot.width = 12, repr.plot.height = 10)
DimPlot(nd_seurat_filt, reduction = "umap", group.by = "Subclass_new",
        label = TRUE, repel = TRUE, raster = FALSE) + coord_fixed()

# UMAP of all cells, colored by final cluster
# Rerun UMAP on the full object first, or explicitly reuse the existing UMAP
# The following code reruns UMAP from the existing PCA space
classFiltSeurat_filt <- RunUMAP(classFiltSeurat_filt, reduction = "pca", dims = 1:30)

options(repr.plot.width = 14, repr.plot.height = 10)
DimPlot(classFiltSeurat_filt, reduction = "umap", group.by = "cluster_final",
        label = TRUE, repel = TRUE, raster = FALSE) + coord_fixed()



In [ ]:
# ========== 7.Optionally save the object ==========
saveRDS(classFiltSeurat_filt, file = "classFiltSeurat_filt.rds")
saveRDS(nd_seurat_filt, file = "nd_seurat_filt.rds")

In [ ]:
library(Seurat)
library(ggplot2)
library(patchwork)

# ========== Gene list ==========
genes <- c("SLC17A6", "SLC17A7", "SLC6A4", "TPH2", "GAD1", "GAD2", 
           "TH", "SLC17A8", "SNAP25", "GFAP", "PECAM1", "FLT1", 
           "VCAN", "MOBP", "C1QC", "FOXJ1")

# Check which genes are present in the data
genes_avail <- genes[genes %in% rownames(nd_seurat_filt)]
genes_missing <- setdiff(genes, rownames(nd_seurat_filt))
if (length(genes_missing) > 0) {
  cat("The following genes are not present in the data (please check capitalization or species):", paste(genes_missing, collapse = ", "), "\n")
}
cat("Available genes:", paste(genes_avail, collapse = ", "), "\n")

# ============================================================
# I. Plots for the Neurons subset (nd_seurat_filt)
# ============================================================

# -------- 1.1 DotPlot (by Neuron subclusters) --------
p_dot_neurons <- DotPlot(nd_seurat_filt, features = genes_avail, group.by = "Subclass_new") +
  coord_flip() +
  scale_color_gradient2(low = "blue", mid = "white", high = "red", midpoint = 0) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  ggtitle("Neurons Subclusters - Marker DotPlot")

print(p_dot_neurons)

In [ ]:
# -------- 1.2 FeaturePlot (UMAP) --------
# Split features into two groups to avoid overcrowding
p_feat_neurons_1 <- FeaturePlot(nd_seurat_filt, features = genes_avail[1:8], 
                                  cols = c("lightgrey", "darkred"), 
                                  raster = FALSE, ncol = 4)
p_feat_neurons_2 <- FeaturePlot(nd_seurat_filt, features = genes_avail[9:length(genes_avail)], 
                                  cols = c("lightgrey", "darkred"), 
                                  raster = FALSE, ncol = 4)

print(p_feat_neurons_1)
print(p_feat_neurons_2)

# -------- 1.3 Display core 5-HT markers separately--------
ht_core <- c("TPH2", "SLC6A4")
if (all(ht_core %in% genes_avail)) {
  p_ht <- FeaturePlot(nd_seurat_filt, features = ht_core, 
                      cols = c("lightgrey", "darkred"), 
                      raster = FALSE, ncol = 2) &
    theme(plot.title = element_text(face = "bold"))
  print(p_ht)
}


In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

# ============================================================
# Step 1: label 5-HT and non-5-HT neurons in nd_seurat_filt
# ============================================================
ht_clusters <- c("Neurons_4", "Neurons_10", "Neurons_16")

nd_seurat_filt$neuron_type <- ifelse(nd_seurat_filt$Subclass_new %in% ht_clusters,
                                     "5-HT", "non-5-HT-neurons")

cat("Neurons classification:\n")
print(table(nd_seurat_filt$neuron_type))

# Confirm the labels visually
DimPlot(nd_seurat_filt, group.by = "neuron_type", label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT vs non-5-HT-neurons")

In [ ]:
# ============================================================
# Step 2: Write the labels back to classFiltSeurat_filt
# ============================================================
classFiltSeurat_filt$cluster_final <- as.character(classFiltSeurat_filt$cluster)

# Update labels for Neurons cells
nd_cells_new <- colnames(nd_seurat_filt)
classFiltSeurat_filt$cluster_final[nd_cells_new] <- nd_seurat_filt$neuron_type

cat("\nFinal classification of all cells:\n")
print(table(classFiltSeurat_filt$cluster_final))



In [ ]:
# ============================================================
# Step 3: Extract 5-HT and non-5-HT neurons separately and re-cluster each
# ============================================================

# -------- 3.1 Extract 5-HT neurons and cluster them separately --------
ht_seurat <- subset(nd_seurat_filt, subset = neuron_type == "5-HT")
cat("\nNumber of 5-HT neurons:", ncol(ht_seurat), "\n")

# Re-identify highly variable genes + PCA + clustering + UMAP
ht_seurat <- FindVariableFeatures(ht_seurat, selection.method = "vst", nfeatures = 3000)
ht_seurat <- RunPCA(ht_seurat, features = VariableFeatures(ht_seurat), npcs = 50, verbose = FALSE)
ht_seurat <- FindNeighbors(ht_seurat, reduction = "pca", dims = 1:30)
ht_seurat <- FindClusters(ht_seurat, resolution = 0.5)  # adjust as appropriate
ht_seurat <- RunUMAP(ht_seurat, reduction = "pca", dims = 1:30)

# Rename as 5-HT_0, 5-HT_1...
ht_clusters_new <- as.character(ht_seurat$seurat_clusters)
ht_unique <- sort(unique(as.numeric(ht_clusters_new)))

ht_counter <- 0
ht_map <- character(length(ht_unique))
names(ht_map) <- as.character(ht_unique)

for(cl in as.character(ht_unique)) {
  ht_map[cl] <- paste0("5-HT_", ht_counter)
  ht_counter <- ht_counter + 1
}

ht_seurat$ht_subtype <- unname(ht_map[ht_clusters_new])

cat("\nDistribution of 5-HT subtypes:\n")
print(table(ht_seurat$ht_subtype))

# 5-HT visualization
p_ht_umap <- DimPlot(ht_seurat, group.by = "ht_subtype", label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT Neurons Subtypes")

p_ht_vln <- VlnPlot(ht_seurat, features = c("TPH2", "SLC6A4", "SLC17A6", "GAD1", "TH"),
                    pt.size = 0, ncol = 3)

print(p_ht_umap)
print(p_ht_vln)


In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

# ============================================================
# 1.UMAP colored by 5-HT subtype
# ============================================================
options(repr.plot.width = 10, repr.plot.height = 8)

p_ht_umap <- DimPlot(ht_seurat, group.by = "ht_subtype", 
                     label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("5-HT Neurons Subtypes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))

print(p_ht_umap)



In [ ]:
library(Seurat)
library(ggplot2)

# ============================================================
# 5-HT UMAP without labels
# ============================================================
options(repr.plot.width = 10, repr.plot.height = 8)

p_ht_umap_nolabel <- DimPlot(ht_seurat, 
                             group.by = "ht_subtype", 
                             label = FALSE,           # Hide labels
                             raster = FALSE) +
  coord_fixed() + 
  ggtitle("5-HT Neurons Subtypes (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")  # Keep the right-side legend to identify colors

print(p_ht_umap_nolabel)

In [ ]:
# ============================================================
# 2. Identify the top 20 marker genes
# ============================================================
ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

ht_markers <- FindAllMarkers(ht_seurat, 
                             only.pos = TRUE, 
                             min.pct = 0.25, 
                             logfc.threshold = 0.25)

top20 <- ht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

cat("Top 20 markers per 5-HT subtype:\n")
print(top20 %>% select(gene, cluster, avg_log2FC, p_val_adj) %>% head(40))

write.csv(top20, "ht_top20_markers.csv", row.names = FALSE)


In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

# ============================================
# 0. Parameter settings
# ============================================
output_path <- "./data/"

# ============================================
# 1. Ensure the 5-HT subset & factor levels are fixed
# ============================================
# Your data already has ht_subtype: 5-HT_0 ~ 5-HT_8
ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, 
                                levels = paste0("5-HT_", 0:13))

# Set ident to ht_subtype for FindAllMarkers
ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

cat("Distribution of 5-HT subtypes:\n")
print(table(ht_seurat$ht_subtype))

In [ ]:
# ============================================
# 2. Identify marker genes using the Wilcoxon test
# ============================================
ht_markers <- FindAllMarkers(ht_seurat,
                             only.pos = TRUE,
                             min.pct = 0.25,
                             logfc.threshold = 0.25,
                             test.use = "wilcox")

# Save the top 20 markers
top20 <- ht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(top20, paste0(output_path, "ht_top20_markers.csv"), row.names = FALSE)
cat("Top 20 markers saved\n")

# ============================================
# 3. Extract Top5 marker genes for each subtype
# ============================================
top5_genes <- ht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

cat("\nTop5 marker genes for each subtype:\n")
print(top5_genes %>% select(cluster, gene, avg_log2FC, p_val_adj))

# Gene order (arranged by 5-HT_0 → 5-HT_8, deduplicated)
gene_order <- top5_genes %>%
  arrange(cluster) %>%
  pull(gene) %>%
  unique()

cat("\nDotPlot gene order (5-HT_0→8),", length(gene_order), "genes total:\n")
cat(paste(gene_order, collapse = ", "), "\n")

# ============================================
# 4. Top5 DotPlot
# ============================================
p_dot <- DotPlot(
  ht_seurat,
  features = gene_order,
  group.by = "ht_subtype",         # use ht_subtype
  dot.scale = 5,                   # many genes, make dots slightly smaller
  cols = c("lightgrey", "red")
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("5-HT Subtype Top5 Marker Gene Expression DotPlot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),      # many genes, shrink font to avoid overlap
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )

print(p_dot)

ggsave(
  filename = paste0(output_path, "5HT_Top5_Markers_DotPlot.png"),
  plot = p_dot,
  width = 14,      # 9 subtypes + 45 genes, needs to be wider
  height = 16,     # needs to be taller
  dpi = 300
)
cat("\nTop5 DotPlot saved!\n")

# Save the table
write.csv(
  top5_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
  file = paste0(output_path, "5HT_Top5_Markers_table.csv"),
  row.names = FALSE
)

In [ ]:
# ============================================
# 5. # ============================================

# ============================================
plot_data <- ht_seurat@meta.data
plot_data$ht_subtype <- factor(plot_data$ht_subtype, levels = paste0("5-HT_", 0:13))

p_nfeature <- ggplot(plot_data, aes(x = ht_subtype, y = nFeature_RNA, fill = ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nFeature RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_blank(),
    axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"),
    axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 13),
    legend.position = "none"
  )

print(p_nfeature)

png(paste0(output_path, "violin_5HT_nFeature_RNA.png"), width = 1200, height = 800, res = 150)
print(p_nfeature)
dev.off()

pdf(paste0(output_path, "violin_5HT_nFeature_RNA.pdf"), width = 12, height = 8)
print(p_nfeature)
dev.off()
cat("nFeature_RNA Violin plot\n")

In [ ]:
# ============================================
# 6.Violin plot of nCount_RNA
# ============================================
p_ncount <- ggplot(plot_data, aes(x = ht_subtype, y = nCount_RNA, fill = ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nCount RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_blank(),
    axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"),
    axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 13),
    legend.position = "none"
  )

print(p_ncount)

png(paste0(output_path, "violin_5HT_nCount_RNA.png"), width = 1200, height = 800, res = 150)
print(p_ncount)
dev.off()

pdf(paste0(output_path, "violin_5HT_nCount_RNA.pdf"), width = 12, height = 8)
print(p_ncount)
dev.off()
cat("Violin plot of nCount_RNA\n")

In [ ]:
# ============================================
# 7. Ranger + heatmap (show values only on the diagonal)
# ============================================
expr_matrix <- GetAssayData(ht_seurat, layer = "data")
genes <- VariableFeatures(ht_seurat)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(ht_seurat$ht_subtype)

rf_ht <- ranger(
  x = X,
  y = y,
  num.trees = 500,
  num.threads = 8,
  classification = TRUE
)

cat("\nOOB Error:", rf_ht$prediction.error, "\n")

conf_mat <- rf_ht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("Diagonal accuracy:\n")
print(round(diag(confusionRatio), 2))

# Keep only subtypes with >= 20 cells
df_num <- as.data.frame(table(ht_seurat$ht_subtype))
select_cell <- as.character(df_num[df_num$Freq >= 20, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))

mat_to_plot <- confusionRatio[select_cell, select_cell, drop = FALSE]

# Show values only on the diagonal
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

# Display on screen
options(repr.plot.width = 10, repr.plot.height = 8)
pheatmap(
  mat_to_plot,
  breaks = seq(0, 1, 0.01),
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  border_color = NA,
  cellwidth = 45,
  cellheight = 45,
  fontsize = 14,
  color = viridis(100),
  main = "5-HT Subtypes Ranger Confusion Matrix",
  display_numbers = num_labels,
  fontsize_number = 12,
  number_color = "white"
)

# Save the PNG file
png(paste0(output_path, "5HT_Ranger_Heatmap.png"), width = 1200, height = 1000, res = 150)
pheatmap(
  mat_to_plot,
  breaks = seq(0, 1, 0.01),
  cluster_rows = FALSE,
  cluster_cols = FALSE,
  border_color = NA,
  cellwidth = 45,
  cellheight = 45,
  fontsize = 14,
  color = viridis(100),
  main = "5-HT Subtypes Ranger Confusion Matrix",
  display_numbers = num_labels,
  fontsize_number = 12,
  number_color = "white"
)
dev.off()
cat("Ranger heatmap saved\n")


In [ ]:
# Inspect all column names first
colnames(classFiltSeurat@meta.data)

In [ ]:
library(Seurat)

# ============================================
# Step 5: Write the refined 5-HT subtypes back to the original object
# ============================================

# Use the existing cluster column as the default value
classFiltSeurat$cell_type <- as.character(classFiltSeurat$cluster)

# 5-HT cells: overwrite with refined subtypes (5-HT_0 ~ 5-HT_8)
ht_cells <- colnames(ht_seurat)
classFiltSeurat$cell_type[ht_cells] <- as.character(ht_seurat$ht_subtype)

# Fix the factor level order
all_types <- unique(classFiltSeurat$cell_type)
ht_types <- paste0("5-HT_", 0:8)
other_types <- setdiff(all_types, ht_types)
type_levels <- c(ht_types, sort(other_types))

classFiltSeurat$cell_type <- factor(classFiltSeurat$cell_type, levels = type_levels)

cat("\nDistribution of cell_type across all cells:\n")
print(table(classFiltSeurat$cell_type))

In [ ]:
save_path <- "./data/"
dir.create(save_path, showWarnings = FALSE, recursive = TRUE)

# 1. Full object RDS (contains the cell_type column)
saveRDS(classFiltSeurat,
        file = paste0(save_path, "classFiltSeurat_with_ht_subtype.rds"))
cat("Full Seurat object saved!\n")

# 2. Metadata CSV
write.csv(classFiltSeurat@meta.data,
          file = paste0(save_path, "classFiltSeurat_metadata.csv"),
          row.names = TRUE)
cat("Metadata saved!\n")

# 3. 5-HT subset RDS
saveRDS(ht_seurat,
        file = paste0(save_path, "ht_seurat_harmony_base.rds"))
cat("5-HT subset saved!\n")

In [ ]:
# Perform preliminary 5-HT classification above to reduce the risk of 5-HT cells remaining among the non-5-HT neurons

In [ ]:
# Extract and recluster non-5-HT neurons separately

In [ ]:
library(Seurat)
save_path <- "./data/"

# ---- 1. Read the two objects back into memory ----
classFiltSeurat <- readRDS(paste0(save_path, "classFiltSeurat_with_ht_subtype.rds"))
ht_seurat       <- readRDS(paste0(save_path, "ht_seurat_harmony_base.rds"))

ht_cells <- intersect(colnames(ht_seurat), colnames(classFiltSeurat))
stopifnot(length(ht_cells) == 3844)

# ---- 2. Rebuild the labels; make corrections in this section----
cl        <- as.character(classFiltSeurat$cluster)
is_neuron <- startsWith(cl, "Neurons")     # ← The original == "Neurons" expression matched no cells; use this instead
all_bc    <- colnames(classFiltSeurat)

classFiltSeurat$cluster_final <- cl
classFiltSeurat$cluster_final[all_bc %in% ht_cells] <- "5-HT"
classFiltSeurat$cluster_final[is_neuron & !(all_bc %in% ht_cells)] <- "non-5-HT-neurons"

# ---- 3. Inspect the result ----
final_table <- table(classFiltSeurat$cluster_final)
print(final_table)


In [ ]:
cf <- classFiltSeurat$cluster_final
nf <- classFiltSeurat$nFeature_RNA

print(addmargins(table(cf, nf > 1500)))   # Cell counts per class before and after filtering

cat("non-5-HT and nFeature>1500:", sum(cf == "non-5-HT-neurons" & nf > 1500), "\n")
cat("5-HT and nFeature>1500:",     sum(cf == "5-HT" & nf > 1500), "\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(harmony)

# ============================================================
# Step 3.1: Extract non-5-HT neurons (with nFeature > 1500 filter)
# ============================================================
non_ht_seurat <- subset(classFiltSeurat,
                        subset = cluster_final == "non-5-HT-neurons" &
                                 nFeature_RNA > 1500)

cat("Number of non-5-HT neurons (after extraction + nFeature > 1500 filter):", ncol(non_ht_seurat), "\n")
stopifnot(ncol(non_ht_seurat) == 41209)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(harmony)
library(clustree)   

# ============================================================
# Step 3.2: Normalization + Harmony (unchanged)
# ============================================================
non_ht_seurat <- SCTransform(non_ht_seurat,
                             vst.flavor = "v2",
                             variable.features.n = 3000,
                             vars.to.regress = "percent.mt",
                             method = "glmGamPoi",
                             verbose = FALSE)

non_ht_seurat <- RunPCA(non_ht_seurat, npcs = 50, verbose = FALSE)

non_ht_seurat <- RunHarmony(non_ht_seurat,
                            group.by.vars = "donor",
                            reduction.use = "pca",
                            dims.use = 1:30,
                            assay.use = "SCT",
                            verbose = FALSE)

non_ht_seurat <- FindNeighbors(non_ht_seurat, reduction = "harmony", dims = 1:30)

# ============================================================
# New: multi-resolution clustering + clustree
# ============================================================
# Note: no resolution is set here (default 0.8 is used)
# Then run a series of resolutions manually, all stored in meta.data
resolutions <- seq(0.1, 1.0, by = 0.1)

for (res in resolutions) {
  non_ht_seurat <- FindClusters(non_ht_seurat,
                                resolution = res,
                                future.seed = TRUE,
                                verbose = FALSE)
  cat("Done: resolution =", res, "\n")
}

# clustree visualization (column name prefix is "SCT_snn_res.")
p_clustree <- clustree(non_ht_seurat@meta.data,
                       prefix = "SCT_snn_res.",
                       node_colour = "sc3_stability") +
  ggtitle("Clustree: non-5-HT neurons (res 0.1–1.0)") +
  theme(legend.position = "right")
print(p_clustree)

ggsave("./data/nonHT_clustree.png",
       p_clustree, width = 12, height = 8, dpi = 300)

In [ ]:


# ============================================================
# Step 3.2: Normalization + Harmony + clustering
# ============================================================
non_ht_seurat <- SCTransform(non_ht_seurat,
                             vst.flavor = "v2",
                             variable.features.n = 3000,
                             vars.to.regress = "percent.mt",
                             method = "glmGamPoi",
                             verbose = FALSE)

non_ht_seurat <- RunPCA(non_ht_seurat, npcs = 50, verbose = FALSE)

non_ht_seurat <- RunHarmony(non_ht_seurat,
                            group.by.vars = "donor",
                            reduction.use = "pca",
                            dims.use = 1:30,
                            assay.use = "SCT",
                            verbose = FALSE)

non_ht_seurat <- FindNeighbors(non_ht_seurat, reduction = "harmony", dims = 1:30)
non_ht_seurat <- FindClusters(non_ht_seurat, resolution = 0.3, future.seed = TRUE)
non_ht_seurat <- RunUMAP(non_ht_seurat, reduction = "harmony", dims = 1:30)

# ---------- Rename as nonHT_0, nonHT_1... ----------
non_ht_clusters_new <- as.character(non_ht_seurat$seurat_clusters)
non_ht_unique <- sort(unique(as.numeric(non_ht_clusters_new)))

non_ht_counter <- 0
non_ht_map <- character(length(non_ht_unique))
names(non_ht_map) <- as.character(non_ht_unique)

for (cl in as.character(non_ht_unique)) {
  non_ht_map[cl] <- paste0("nonHT_", non_ht_counter)
  non_ht_counter <- non_ht_counter + 1
}

non_ht_seurat$non_ht_subtype <- unname(non_ht_map[non_ht_clusters_new])

cat("\nDistribution of non-5-HT subtypes (res 0.3, n=41209):\n")
print(table(non_ht_seurat$non_ht_subtype))

# ---------- QC visualization ----------
p1 <- DimPlot(non_ht_seurat, group.by = "donor", raster = FALSE) +
  ggtitle("non-5-HT neurons by donor (n=41209, nFeature>1500, harmony)") +
  coord_fixed()
print(p1)

p2 <- DimPlot(non_ht_seurat, group.by = "non_ht_subtype",
              label = TRUE, repel = TRUE, raster = FALSE) +
  ggtitle("non-5-HT neurons reclustered (res 0.3, n=41209)") +
  coord_fixed()
print(p2)

p3 <- DotPlot(non_ht_seurat,
              features = c("SNAP25", "SLC17A6", "SLC17A8", "GAD1", "GAD2", "TH"),
              group.by = "non_ht_subtype") +
  RotatedAxis()
print(p3)




In [ ]:
# ---------- Save to disk immediately after it runs (don't keep it only in memory) ----------
saveRDS(non_ht_seurat,
        file = paste0(save_path, "non_ht_seurat_final.rds"))
saveRDS(classFiltSeurat,
        file = paste0(save_path, "classFiltSeurat_final.rds"))
cat("non_ht_seurat and classFiltSeurat have both been saved to disk\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

# ============================================
# 0. Parameter settings
# ============================================
output_path <- "./data/"

# ============================================
# 1. Data preparation: fix factor levels & ident
# ============================================
# Dynamically obtain and sort non-HT subtype levels
non_ht_levels <- sort(unique(as.character(non_ht_seurat$non_ht_subtype)))
non_ht_seurat$non_ht_subtype <- factor(non_ht_seurat$non_ht_subtype, levels = non_ht_levels)
non_ht_seurat <- SetIdent(non_ht_seurat, value = "non_ht_subtype")

cat("Distribution of non-5-HT subtypes:\n")
print(table(non_ht_seurat$non_ht_subtype))

In [ ]:
# ============================================
# 1. Data preparation: fix factor levels & ident (in numeric order 0,1,2...)
# ============================================
# First get all unique subtype names
non_ht_levels <- unique(as.character(non_ht_seurat$non_ht_subtype))

# Extract the numeric part and sort numerically (ensure nonHT_0,1,2... not lexicographic order)
non_ht_nums <- as.numeric(gsub("nonHT_", "", non_ht_levels))
non_ht_levels <- non_ht_levels[order(non_ht_nums)]

# Or in a single line:
# non_ht_levels <- paste0("nonHT_", sort(as.numeric(gsub("nonHT_", "", unique(as.character(non_ht_seurat$non_ht_subtype))))))

non_ht_seurat$non_ht_subtype <- factor(non_ht_seurat$non_ht_subtype, levels = non_ht_levels)
non_ht_seurat <- SetIdent(non_ht_seurat, value = "non_ht_subtype")

cat("Distribution of non-5-HT subtypes:\n")
print(table(non_ht_seurat$non_ht_subtype))

In [ ]:
# ============================================
# 2. UMAP plots
# ============================================
# 2.1 Labeled version
p_nonht_umap <- DimPlot(non_ht_seurat, group.by = "non_ht_subtype", 
                        label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))

print(p_nonht_umap)
ggsave(paste0(output_path, "nonHT_UMAP_labeled.png"), p_nonht_umap, width = 10, height = 8, dpi = 300)

# 2.2 Unlabeled version
p_nonht_umap_nolabel <- DimPlot(non_ht_seurat, group.by = "non_ht_subtype", 
                                label = FALSE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")

print(p_nonht_umap_nolabel)
ggsave(paste0(output_path, "nonHT_UMAP_nolabel.png"), p_nonht_umap_nolabel, width = 10, height = 8, dpi = 300)

cat("UMAP plots saved\n")

In [ ]:
# ============================================
# 3. Find marker genes (Wilcoxon) — run only once
# ============================================
nonht_markers <- FindAllMarkers(non_ht_seurat,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

# Save Top20
top20 <- nonht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(top20, paste0(output_path, "nonHT_top20_markers.csv"), row.names = FALSE)
cat("Top20 markers saved\n")

# ============================================
# 4. Top5 DotPlot
# ============================================
top5_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- top5_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot5 <- DotPlot(non_ht_seurat, features = gene_order5, group.by = "non_ht_subtype",
                  dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT Subtype Top5 Marker Gene Expression DotPlot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )

print(p_dot5)
ggsave(paste0(output_path, "nonHT_Top5_Markers_DotPlot.png"), p_dot5, 
       width = 14, height = 16, dpi = 300)

write.csv(top5_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_Top5_Markers_table.csv"), row.names = FALSE)
cat("Top5 DotPlot saved\n")

# ============================================
# 5. Top10 DotPlot
# ============================================
top10_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- top10_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot10 <- DotPlot(non_ht_seurat, features = gene_order10, group.by = "non_ht_subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT Subtype Top10 Marker Gene Expression DotPlot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )

print(p_dot10)
ggsave(paste0(output_path, "nonHT_Top10_Markers_DotPlot.png"), p_dot10, 
       width = 16, height = 22, dpi = 300)

write.csv(top10_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_Top10_Markers_table.csv"), row.names = FALSE)
cat("Top10 DotPlot saved\n")

# ============================================
# 6. Top20 DotPlot
# ============================================
gene_order20 <- top20 %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot20 <- DotPlot(non_ht_seurat, features = gene_order20, group.by = "non_ht_subtype",
                   dot.scale = 3, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT Subtype Top20 Marker Gene Expression DotPlot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 5.5),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )

print(p_dot20)
ggsave(paste0(output_path, "nonHT_Top20_Markers_DotPlot.png"), p_dot20, 
       width = 18, height = 32, dpi = 300)

write.csv(top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_Top20_Markers_table.csv"), row.names = FALSE)
cat("Top20 DotPlot saved\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

output_path <- "./data/"

# ============================================
# 0. Data preparation (using the cleaned object)
# ============================================
# Re-obtain and fix factor levels (sorted numerically 0,1,2...)
nonht_levels <- paste0("nonHT_", sort(as.numeric(gsub("nonHT_", "", 
                unique(as.character(non_ht_seurat$non_ht_subtype))))))
non_ht_seurat$non_ht_subtype <- factor(non_ht_seurat$non_ht_subtype, levels = nonht_levels)
non_ht_seurat_clean <- SetIdent(non_ht_seurat, value = "non_ht_subtype")

cat("Distribution of non-5-HT subtypes:\n")
print(table(non_ht_seurat_clean$non_ht_subtype))


In [ ]:
# ============================================
# 1. UMAP plots
# ============================================
# 1.1 Labeled version
p_nonht_umap <- DimPlot(non_ht_seurat_clean, group.by = "non_ht_subtype", 
                        label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_nonht_umap)
ggsave(paste0(output_path, "nonHT_UMAP_labeled.png"), p_nonht_umap, width = 10, height = 8, dpi = 300)

# 1.2 Unlabeled version
p_nonht_umap_nolabel <- DimPlot(non_ht_seurat_clean, group.by = "non_ht_subtype", 
                                label = FALSE, raster = FALSE) +
  coord_fixed() + 
  ggtitle("non-5-HT Neurons Subtypes (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")
print(p_nonht_umap_nolabel)
ggsave(paste0(output_path, "nonHT_UMAP_nolabel.png"), p_nonht_umap_nolabel, width = 10, height = 8, dpi = 300)
cat("UMAP plots saved\n")


In [ ]:
# ============================================
# 6. DotPlot + UMAP FeaturePlot for user-specified genes
# ============================================
# User-specified genes, auto-fix capitalization; ChE matched to ACHE
gene_list_input <- c('SLC17A6','SLC17A7','SLC17A8','GAD1','GAD2','TH')
gene_list <- toupper(gene_list_input)  # unify to uppercase
gene_list[gene_list == "CHE"] <- "ACHE"  # ChE usually refers to ACHE (acetylcholinesterase)

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

if(length(missing_genes) > 0) {
  cat("Warning: the following genes were not found in the expression matrix and will be skipped:\n")
  cat(paste(missing_genes, collapse = ", "), "\n\n")
}
cat("Available genes (", length(available_genes), "total):\n")
cat(paste(available_genes, collapse = ", "), "\n\n")

# 6.1 DotPlot
p_custom <- DotPlot(non_ht_seurat_clean, features = available_genes, group.by = "non_ht_subtype",
                    dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT Subtype User-Specified Marker Gene Expression DotPlot") +
  xlab("non-5-HT Subtype") + ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_custom)
ggsave(paste0(output_path, "nonHT_Custom_Markers_DotPlot.png"), p_custom, 
       width = 12, height = 10, dpi = 300)
cat("User-specified gene DotPlot saved\n")

# 6.2 UMAP FeaturePlot (faceted)
p_features <- FeaturePlot(non_ht_seurat_clean, features = available_genes, 
                          ncol = 4, raster = FALSE, cols = c("lightgrey", "red"))
print(p_features)
ggsave(paste0(output_path, "nonHT_Custom_Markers_UMAP.png"), p_features, 
       width = 16, height = 12, dpi = 300)
cat("User-specified gene UMAP FeaturePlot saved\n")

In [ ]:
# ============================================
# 6. DotPlot + UMAP FeaturePlot for user-specified genes
# ============================================
# User-specified genes, auto-fix capitalization; ChE matched to ACHE
gene_list_input <- c('SLC17A6','SLC17A8','GAD1','GAD2','SOX5','GFAP','PECAM1','FLT1','VCAN','MOBP','C1QC','SLC6A4','TH')
gene_list <- toupper(gene_list_input)  # unify to uppercase
gene_list[gene_list == "CHE"] <- "ACHE"  # ChE usually refers to ACHE (acetylcholinesterase)

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

if(length(missing_genes) > 0) {
  cat("Warning: the following genes were not found in the expression matrix and will be skipped:\n")
  cat(paste(missing_genes, collapse = ", "), "\n\n")
}
cat("Available genes (", length(available_genes), "total):\n")
cat(paste(available_genes, collapse = ", "), "\n\n")

# 6.1 DotPlot
p_custom <- DotPlot(non_ht_seurat_clean, features = available_genes, group.by = "non_ht_subtype",
                    dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT Subtype User-Specified Marker Gene Expression DotPlot") +
  xlab("non-5-HT Subtype") + ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_custom)
ggsave(paste0(output_path, "nonHT_Custom_Markers_DotPlot.png"), p_custom, 
       width = 12, height = 10, dpi = 300)
cat("User-specified gene DotPlot saved\n")

# 6.2 UMAP FeaturePlot (faceted)
p_features <- FeaturePlot(non_ht_seurat_clean, features = available_genes, 
                          ncol = 4, raster = FALSE, cols = c("lightgrey", "red"))
print(p_features)
ggsave(paste0(output_path, "nonHT_Custom_Markers_UMAP.png"), p_features, 
       width = 16, height = 12, dpi = 300)
cat("User-specified gene UMAP FeaturePlot saved\n")

In [ ]:
# ============================================
# 7. Violin plots
# ============================================
plot_data <- non_ht_seurat_clean@meta.data
plot_data$non_ht_subtype <- factor(plot_data$non_ht_subtype, levels = nonht_levels)

# 7.1 nFeature_RNA (one tick every 1000)
nfeature_max <- max(plot_data$nFeature_RNA, na.rm = TRUE)
nfeature_breaks <- seq(0, ceiling(nfeature_max / 1000) * 1000, by = 1000)

p_nfeature <- ggplot(plot_data, aes(x = non_ht_subtype, y = nFeature_RNA, fill = non_ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  scale_y_continuous(breaks = nfeature_breaks) +
  labs(x = "", y = "nFeature RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_nfeature)
ggsave(paste0(output_path, "violin_nonHT_nFeature_RNA_1k.png"), p_nfeature, width = 8, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_nFeature_RNA_1k.pdf"), p_nfeature, width = 12, height = 8)

# 7.2 nCount_RNA
p_ncount <- ggplot(plot_data, aes(x = non_ht_subtype, y = nCount_RNA, fill = non_ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nCount RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_ncount)
ggsave(paste0(output_path, "violin_nonHT_nCount_RNA.png"), p_ncount, width = 8, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_nCount_RNA.pdf"), p_ncount, width = 12, height = 8)
cat("Violin plots saved\n")


In [ ]:
# ============================================
# 8. Ranger + heatmap
# ============================================
expr_matrix <- GetAssayData(non_ht_seurat_clean, layer = "data")
genes <- VariableFeatures(non_ht_seurat_clean)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(non_ht_seurat_clean$non_ht_subtype)

rf_nonht <- ranger(x = X, y = y, num.trees = 500, num.threads = 8, classification = TRUE)

cat("\nOOB Error:", rf_nonht$prediction.error, "\n")

conf_mat <- rf_nonht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("Diagonal accuracy:\n")
print(round(diag(confusionRatio), 2))

# Keep only subtypes with >= 20 cells
df_num <- as.data.frame(table(non_ht_seurat_clean$non_ht_subtype))
select_cell <- as.character(df_num[df_num$Freq >= 20, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))
mat_to_plot <- confusionRatio[select_cell, select_cell, drop = FALSE]

# 8.1 Version with values
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

png(paste0(output_path, "nonHT_Ranger_Heatmap_withNum.png"), width = 1200, height = 1000, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 45, cellheight = 45, fontsize = 14,
         color = viridis(100), main = "non-5-HT Subtypes Ranger Confusion Matrix",
         display_numbers = num_labels, fontsize_number = 12, number_color = "white")
dev.off()

# 8.2 Version without values
png(paste0(output_path, "nonHT_Ranger_Heatmap_noNum.png"), width = 1200, height = 1000, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 45, cellheight = 45, fontsize = 14,
         color = viridis(100), main = "non-5-HT Subtypes Ranger Confusion Matrix",
         display_numbers = FALSE)
dev.off()

# Display the version without values on screen
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 45, cellheight = 45, fontsize = 14,
         color = viridis(100), main = "non-5-HT Subtypes Ranger Confusion Matrix (No Numbers)",
         display_numbers = FALSE)

cat("Ranger heatmap saved\n")

cat("\n===== non-5-HT analysis complete =====\n")

In [ ]:
# ============================================
# 6.3 Violin plots of SLC17A6 / SLC17A8 / GAD1 / GAD2 across subtypes
# ============================================
library(tidyr)   # pivot_longer is here
vln_genes <- c("SLC17A6", "SLC17A8", "GAD1", "GAD2", "TH")
vln_genes <- vln_genes[vln_genes %in% rownames(non_ht_seurat_clean)]

expr_vln <- FetchData(non_ht_seurat_clean, vars = c("non_ht_subtype", vln_genes))

expr_vln_long <- expr_vln %>%
  pivot_longer(cols = -non_ht_subtype,
               names_to = "gene", values_to = "Expression") %>%
  mutate(gene = factor(gene, levels = vln_genes),
         non_ht_subtype = factor(non_ht_subtype))   # add levels = ... here to fix the order

p_vln <- ggplot(expr_vln_long,
                aes(x = non_ht_subtype, y = Expression, fill = non_ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE) +
  geom_boxplot(width = 0.15, outlier.size = 0.15, alpha = 0.7, color = "grey30") +
  facet_wrap(~ gene, nrow = 1, scales = "free_y") +
  labs(title = "SLC17A6 / SLC17A8 / GAD1 / GAD2: Expression across non-5-HT subtypes",
       x = NULL, y = "Normalized Expression") +
  theme_bw(base_size = 13) +
  theme(legend.position = "none",
        panel.grid = element_blank(),
        strip.text = element_text(face = "bold", size = 12),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9))

print(p_vln)
ggsave(paste0(output_path, "nonHT_VlnPlot_Glut_GABA_markers.png"), p_vln,
       width = 14, height = 5, dpi = 300)
cat("User-specified gene violin plot saved\n")

In [ ]:


library(Seurat)
library(dplyr)

save_path <- "./data/"

# ============================================
# 1. Define neurotransmitter type mapping (nonHT_15 changed to Glut)
# ============================================
glut_map <- c(
  "nonHT_0"  = "Glut_1",
  "nonHT_9" = "Glut_2",
  "nonHT_10" = "Glut_3", 
  "nonHT_17" = "Glut_4",
  "nonHT_22" = "Glut_5",   
  "nonHT_24" = "Glut_6",   
  "nonHT_25" = "Glut_7"
)

glut_gaba_map <- c(
  "nonHT_0"  = "Glut-GABA_1",
  "nonHT_4"  = "Glut-GABA_2",
  "nonHT_8"  = "Glut-GABA_3",
  "nonHT_11"  = "Glut-GABA_4",
  "nonHT_13"  = "Glut-GABA_5",
  "nonHT_28"  = "Glut-GABA_6"
)

gaba_map <- c(
  "nonHT_1"  = "GABA_1",
  "nonHT_2"  = "GABA_2",
  "nonHT_3"  = "GABA_3",
  "nonHT_5"  = "GABA_4",
  "nonHT_6"  = "GABA_5",
  "nonHT_7"  = "GABA_6",
  "nonHT_12" = "GABA_7",
  "nonHT_14" = "GABA_8",
  "nonHT_15" = "GABA_9",
  "nonHT_16" = "GABA_10",
  "nonHT_18" = "GABA_11",
  "nonHT_19" = "GABA_12",
  "nonHT_20" = "GABA_13",
  "nonHT_21" = "GABA_14",
  "nonHT_23" = "GABA_15",
  "nonHT_27" = "GABA_16"
)

da_map <- c("nonHT_26" = "DA_1")

# Merge all mappings
all_map <- c(glut_map, glut_gaba_map, gaba_map, da_map)

cat("Mapping table:\n")
print(all_map)

# ============================================
# Fix: rename neuro_type
# ============================================

# Look up the mapping by cell order
neuro_vec <- all_map[as.character(non_ht_seurat_clean$non_ht_subtype)]

# Key step: remove names, otherwise Seurat will match cells by name
names(neuro_vec) <- NULL

# Assign the values
non_ht_seurat_clean$neuro_type <- neuro_vec

# Fix the factor level order
type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)

cat("\nDistribution of non-5-HT neurotransmitter types:\n")
print(table(non_ht_seurat_clean$neuro_type))

# 4. Save
# ============================================
# saveRDS(non_ht_seurat_clean, paste0(save_path, "non_ht_seurat_neurotype.rds"))
# saveRDS(classFiltSeurat, paste0(save_path, "classFiltSeurat_neurotype.rds"))

# cat("\nFix complete and saved!\n")

In [ ]:
# ============================================
# 3. Sync the update to classFiltSeurat
# ============================================

# First convert factor to character (key step! otherwise new types become NA)
classFiltSeurat$cell_type <- as.character(classFiltSeurat$cell_type)

# Extract non-5-HT cell names
nonht_cells <- colnames(non_ht_seurat_clean)

# Assign values (no NA warnings now)
classFiltSeurat$cell_type[nonht_cells] <- as.character(non_ht_seurat_clean$neuro_type)

# Reorganize factor levels for all cells
all_types <- unique(classFiltSeurat$cell_type)

ht_types       <- sort(grep("^5-HT", all_types, value = TRUE))
glut_types     <- sort(grep("^Glut_", all_types, value = TRUE))
glutgaba_types <- sort(grep("^Glut-GABA_", all_types, value = TRUE))
gaba_types     <- sort(grep("^GABA_", all_types, value = TRUE))
da_types       <- sort(grep("^DA_", all_types, value = TRUE))
other_types    <- sort(setdiff(all_types, c(ht_types, glut_types, glutgaba_types, gaba_types, da_types)))

type_levels <- c(ht_types, glut_types, glutgaba_types, gaba_types, da_types, other_types)
classFiltSeurat$cell_type <- factor(classFiltSeurat$cell_type, levels = type_levels)

cat("\nDistribution of cell_type across all cells in classFiltSeurat:\n")
print(table(classFiltSeurat$cell_type))



In [ ]:
# ============================================
## Remove cells whose cell_type starts with "Neurons_"
# ============================================

# First check how many will be removed
neuron_idx <- grepl("^Neurons_", classFiltSeurat$cell_type)
cat("Number of Neurons_* cells to remove:", sum(neuron_idx), "\n")
cat("Total cells before removal:", ncol(classFiltSeurat), "\n")

# Subset out these cells
classFiltSeurat <- subset(classFiltSeurat, subset = cell_type %in% 
                            levels(classFiltSeurat$cell_type)[!grepl("^Neurons_", levels(classFiltSeurat$cell_type))])

# Or a more direct equivalent:
# classFiltSeurat <- subset(classFiltSeurat, cells = colnames(classFiltSeurat)[!neuron_idx])

cat("Total cells after removal:", ncol(classFiltSeurat), "\n")

# Reorganize factor levels (drop empty levels)
classFiltSeurat$cell_type <- droplevels(classFiltSeurat$cell_type)

cat("\nDistribution of cell_type after removal:\n")
print(table(classFiltSeurat$cell_type))

In [ ]:
# ============================================
# 4. Save
# ============================================
saveRDS(non_ht_seurat_clean, paste0(save_path, "non_ht_seurat_neurotype.rds"))
saveRDS(classFiltSeurat, paste0(save_path, "classFiltSeurat_neurotype.rds"))

cat("\nFix complete and saved!\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)

save_path <- "./data/"

# ============================================
# Load the two objects
# ============================================
non_ht_seurat_clean <- readRDS(paste0(save_path, "non_ht_seurat_neurotype.rds"))
# classFiltSeurat     <- readRDS(paste0(save_path, "classFiltSeurat_neurotype.rds"))

# ============================================
# Post-load check (make sure nothing was loaded wrong)
# ============================================
cat("===== non_ht_seurat_clean =====\n")
print(non_ht_seurat_clean)                      # overview of assays / reductions
cat("Number of cells:", ncol(non_ht_seurat_clean), "\n")
cat("Distribution of neuro_type:\n")
print(table(non_ht_seurat_clean$neuro_type))

# cat("\n===== classFiltSeurat =====\n")
# print(classFiltSeurat)
# cat("Number of cells:", ncol(classFiltSeurat), "\n")
# cat("Distribution of cell_type_fine:\n")
# print(table(classFiltSeurat$cell_type_fine))

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

output_path <- "./data/"

# ============================================
# 0. Confirm data & factor order
# ============================================
# Make sure neuro_type exists and is a factor
if(!"neuro_type" %in% colnames(non_ht_seurat_clean@meta.data)) {
  stop("Please run the neurotransmitter type renaming code first!")
}

type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)
non_ht_seurat_clean <- SetIdent(non_ht_seurat_clean, value = "neuro_type")

cat("Distribution of non-5-HT neurotransmitter types:\n")
print(table(non_ht_seurat_clean$neuro_type))

# ============================================
# 1. UMAP plots (by neurotransmitter type)
# ============================================
# 1.1 Labeled version
p_neuro_umap <- DimPlot(non_ht_seurat_clean, group.by = "neuro_type",
                        label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() +
  ggtitle("non-5-HT Neurons by Neurotransmitter Type") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_neuro_umap)

ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_labeled.png"),
       p_neuro_umap, width = 12, height = 10, dpi = 300)
ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_labeled.pdf"),   # ← newly added
       p_neuro_umap, width = 12, height = 10, limitsize = FALSE)

# 1.2 Unlabeled version
p_neuro_umap_nolabel <- DimPlot(non_ht_seurat_clean, group.by = "neuro_type",
                                label = FALSE, raster = FALSE) +
  coord_fixed() +
  ggtitle("non-5-HT Neurons by Neurotransmitter Type (No Label)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        legend.position = "right")
print(p_neuro_umap_nolabel)

ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_nolabel.png"),
       p_neuro_umap_nolabel, width = 12, height = 10, dpi = 300)
ggsave(paste0(output_path, "nonHT_NeuroType_UMAP_nolabel.pdf"),   # ← newly added
       p_neuro_umap_nolabel, width = 12, height = 10, limitsize = FALSE)

cat("UMAP plots saved (png + pdf)\n")


In [ ]:
# ============================================
# 2. Find marker genes (Wilcoxon) — grouped by neuro_type, run only once
# ============================================
nonht_markers <- FindAllMarkers(non_ht_seurat_clean,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

# Save Top20
top20 <- nonht_markers %>%
  group_by(cluster) %>%
  top_n(n = 20, wt = avg_log2FC) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(top20, paste0(output_path, "nonHT_NeuroType_top20_markers.csv"), row.names = FALSE)
cat("Top20 markers saved\n")

# ============================================
# 3. Top5 DotPlot (by neurotransmitter type)
# ============================================
top5_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- top5_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot5 <- DotPlot(non_ht_seurat_clean, features = gene_order5, group.by = "neuro_type",
                  dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT Neurotransmitter Type Top5 Marker Gene Expression DotPlot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
ggsave(paste0(output_path, "nonHT_NeuroType_Top5_DotPlot.png"), p_dot5, 
       width = 16, height = 18, dpi = 300)

write.csv(top5_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_NeuroType_Top5_table.csv"), row.names = FALSE)
cat("Top5 DotPlot saved\n")

# ============================================
# 4. Top10 DotPlot
# ============================================
top10_genes <- nonht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- top10_genes %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot10 <- DotPlot(non_ht_seurat_clean, features = gene_order10, group.by = "neuro_type",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT Neurotransmitter Type Top10 Marker Gene Expression DotPlot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
ggsave(paste0(output_path, "nonHT_NeuroType_Top10_DotPlot.png"), p_dot10, 
       width = 18, height = 24, dpi = 300)

write.csv(top10_genes %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_NeuroType_Top10_table.csv"), row.names = FALSE)
cat("Top10 DotPlot saved\n")

# ============================================
# 5. Top20 DotPlot
# ============================================
gene_order20 <- top20 %>% arrange(cluster) %>% pull(gene) %>% unique()

p_dot20 <- DotPlot(non_ht_seurat_clean, features = gene_order20, group.by = "neuro_type",
                   dot.scale = 3, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT Neurotransmitter Type Top20 Marker Gene Expression DotPlot") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 5.5),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
ggsave(paste0(output_path, "nonHT_NeuroType_Top20_DotPlot.png"), p_dot20, 
       width = 20, height = 34, dpi = 300)

write.csv(top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "nonHT_NeuroType_Top20_table.csv"), row.names = FALSE)
cat("Top20 DotPlot saved\n")

In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

# ============================================
# 0. Confirm the merged neuro_type order
# ============================================
type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)

# ============================================
# 1. New gene list (52 genes, expanded in paired order)
# ============================================
gene_list <- c("OTOF","NPTX1","SLC17A8","LMX1B","ADGRD2","CDH5","TRIM55","IRX4",
               "PRDM6","LHX9","FST","BCHE","SEMA3A","SCTR","PENK","TACR3",
               "SKOR2","CCDC60","CCK","RIN2","ATP12A","PGM5","SLC32A1","GSTP1",
               "SLC6A5","LAMP5","FGFR2","SATB1","OTX2","SERPINF1","GALR1","SYT6",
               "TAC3","IL13RA1","GLI3","GATA3","TLL1","ADAMTSL1","GSN","CXCL14",
               "FLT1","ONECUT1","SNTG2","CHRNA2","KIF25","BCL11B","SST","CHST9",
               "FOXA2","EMX2","CPA4","RLN3","TH","SLC6A2")

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes   <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

cat("Total genes:", length(gene_list), ", available:", length(available_genes), "\n")
if(length(missing_genes) > 0) {
  cat("Warning: not found, skipped:", paste(missing_genes, collapse = ", "), "\n")
}

# ============================================
# 2. DotPlot
# ============================================
p_custom <- DotPlot(
  non_ht_seurat_clean,
  features = available_genes,
  group.by = "neuro_type",
  dot.scale = 6,
  cols = c("lightgrey", "red")
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("non-5-HT neurotransmitter types: selected markers") +
  xlab("Neurotransmitter type") +
  ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 9),
    axis.title  = element_text(size = 12, face = "bold"),
    plot.title  = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid  = element_blank()
  )

print(p_custom)

ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.png"),
  plot = p_custom,
  width = 16,
  height = 16,
  dpi = 300
)

# Also save a PDF version (vector graphic, stays sharp when zoomed)
ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.pdf"),
  plot = p_custom,
  width = 16,
  height = 16
)

cat("Both PNG and PDF saved!\n")

In [ ]:
# ============================================
# 6. DotPlot + UMAP FeaturePlot for user-specified genes
# ============================================
gene_list_input <- c('SLC17A6','SLC17A7','SLC17A8','GAD1','GAD2','TH')
gene_list <- toupper(gene_list_input)
gene_list[gene_list == "CHE"] <- "ACHE"

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

if(length(missing_genes) > 0) {
  cat("Warning: the following genes were not found and have been skipped:", paste(missing_genes, collapse = ", "), "\n\n")
}
cat("Available genes (", length(available_genes), "total):\n")
cat(paste(available_genes, collapse = ", "), "\n\n")

# 6.1 DotPlot (by neuro_type)
p_custom <- DotPlot(non_ht_seurat_clean, features = available_genes, group.by = "neuro_type",
                    dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("non-5-HT Neurotransmitter Type User-Specified Marker Gene Expression DotPlot") +
  xlab("Neurotransmitter Type") + ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_custom)
ggsave(paste0(output_path, "nonHT_NeuroType_Custom_DotPlot.png"), p_custom, 
       width = 14, height = 10, dpi = 300)
cat("User-specified gene DotPlot saved\n")

# 6.2 UMAP FeaturePlot
# p_features <- FeaturePlot(non_ht_seurat_clean, features = available_genes, 
#                           ncol = 3, raster = FALSE, cols = c("lightgrey", "red"))
# print(p_features)
# ggsave(paste0(output_path, "nonHT_NeuroType_Custom_UMAP.png"), p_features, 
#        width = 15, height = 12, dpi = 300)
# cat("User-specified gene UMAP FeaturePlot saved\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(ranger)
library(pheatmap)
library(viridis)
library(grid)  # pdf() 

output_path <- "./data/"

# ============================================
# 0. Confirm the merged neuro_type order
# ============================================
type_order <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 1:6),
  paste0("GABA_", 1:16),
  "DA_1"
)
non_ht_seurat_clean$neuro_type <- factor(non_ht_seurat_clean$neuro_type, levels = type_order)

# ============================================
# 1. New gene list (52 genes, expanded in paired order)
# ============================================
gene_list <- c("OTOF","NPTX1","SLC17A8","LMX1B","ADGRD2","CDH5","TRIM55","IRX4",
               "PRDM6","LHX9","FST","BCHE","SEMA3A","SCTR","PENK","TACR3",
               "SKOR2","CCDC60","CCK","RIN2","ATP12A","PGM5","SLC32A1","GSTP1",
               "SLC6A5","LAMP5","FGFR2","SATB1","OTX2","SERPINF1","GALR1","SYT6",
               "TAC3","IL13RA1","GLI3","GATA3","TLL1","ADAMTSL1","GSN","CXCL14",
               "FLT1","ONECUT1","SNTG2","CHRNA2","KIF25","BCL11B","SST","CHST9",
               "FOXA2","EMX2","CPA4","RLN3","TH","SLC6A2")

available_genes <- gene_list[gene_list %in% rownames(non_ht_seurat_clean)]
missing_genes   <- gene_list[!gene_list %in% rownames(non_ht_seurat_clean)]

cat("Total genes:", length(gene_list), ", available:", length(available_genes), "\n")
if(length(missing_genes) > 0) {
  cat("Warning: not found, skipped:", paste(missing_genes, collapse = ", "), "\n")
}

# ============================================
# 2. DotPlot
# ============================================
p_custom <- DotPlot(
  non_ht_seurat_clean,
  features = available_genes,
  group.by = "neuro_type",
  dot.scale = 6,
  cols = c("lightgrey", "red")
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("non-5-HT neurotransmitter types: selected markers") +
  xlab("Neurotransmitter type") +
  ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 10, face = "bold"),
    axis.text.y = element_text(size = 9),
    axis.title  = element_text(size = 12, face = "bold"),
    plot.title  = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid  = element_blank()
  )

print(p_custom)

ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.png"),
  plot = p_custom,
  width = 16,
  height = 16,
  dpi = 300
)
ggsave(
  filename = paste0(output_path, "nonHT_NeuroType_Merged_markers_DotPlot.pdf"),
  plot = p_custom,
  width = 16,
  height = 16
)
cat("DotPlot PNG + PDF saved!\n")

# ============================================
# 7. Violin plots (by neurotransmitter type)
# ============================================
plot_data <- non_ht_seurat_clean@meta.data
plot_data$neuro_type <- factor(plot_data$neuro_type, levels = type_order)

# 7.1 nFeature_RNA (one tick every 1000)
nfeature_max <- max(plot_data$nFeature_RNA, na.rm = TRUE)
nfeature_breaks <- seq(0, ceiling(nfeature_max / 1000) * 1000, by = 1000)

p_nfeature <- ggplot(plot_data, aes(x = neuro_type, y = nFeature_RNA, fill = neuro_type)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  scale_y_continuous(breaks = nfeature_breaks) +
  labs(x = "", y = "nFeature RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_nfeature)

ggsave(paste0(output_path, "violin_nonHT_NeuroType_nFeature_RNA_1k.png"),
       p_nfeature, width = 10, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_NeuroType_nFeature_RNA_1k.pdf"),
       p_nfeature, width = 14, height = 8)

# 7.2 nCount_RNA
p_ncount <- ggplot(plot_data, aes(x = neuro_type, y = nCount_RNA, fill = neuro_type)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7, linewidth = 0.3, color = "black") +
  labs(x = "", y = "nCount RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"), axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10), axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_ncount)

ggsave(paste0(output_path, "violin_nonHT_NeuroType_nCount_RNA.png"),
       p_ncount, width = 10, height = 6, dpi = 150)
ggsave(paste0(output_path, "violin_nonHT_NeuroType_nCount_RNA.pdf"),
       p_ncount, width = 14, height = 8)
cat("Violin plots PNG + PDF saved\n")

# ============================================
# 8. Ranger + heatmap (by neuro_type)
# ============================================
expr_matrix <- GetAssayData(non_ht_seurat_clean, layer = "data")
genes <- VariableFeatures(non_ht_seurat_clean)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(non_ht_seurat_clean$neuro_type)

rf_nonht <- ranger(x = X, y = y, num.trees = 500, num.threads = 8, classification = TRUE)

cat("\nOOB Error:", rf_nonht$prediction.error, "\n")

conf_mat <- rf_nonht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("Diagonal accuracy:\n")
print(round(diag(confusionRatio), 2))

# Keep only subtypes with >= 20 cells
df_num <- as.data.frame(table(non_ht_seurat_clean$neuro_type))
select_cell <- as.character(df_num[df_num$Freq >= 20, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))
mat_to_plot <- confusionRatio[select_cell, select_cell, drop = FALSE]

# 8.1 Version with values
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

# --- PNG ---
png(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_withNum.png"), width = 1400, height = 1200, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = num_labels, fontsize_number = 10, number_color = "white")
dev.off()

# --- PDF ---
pdf(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_withNum.pdf"),
    width = 14, height = 12)  # units are inches: 1400px/150dpi ≈ 9.3in, expanded to 14×12
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = num_labels, fontsize_number = 10, number_color = "white")
dev.off()

# 8.2 Version without values
# --- PNG ---
png(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_noNum.png"), width = 1400, height = 1200, res = 150)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = FALSE)
dev.off()

# --- PDF ---
pdf(paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_noNum.pdf"),
    width = 14, height = 12)
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
         display_numbers = FALSE)
dev.off()

# Display the version without values on screen
pheatmap(mat_to_plot, breaks = seq(0, 1, 0.01), cluster_rows = FALSE, cluster_cols = FALSE,
         border_color = NA, cellwidth = 40, cellheight = 40, fontsize = 12,
         color = viridis(100), main = "non-5-HT Neurotransmitter Types Ranger (No Numbers)",
         display_numbers = FALSE)

cat("Ranger heatmap PNG + PDF saved\n")

cat("\n===== non-5-HT neurotransmitter type analysis complete =====\n")

In [ ]:
library(viridis)
library(grid)

# ============================================
# 8. Ranger + heatmap (by neuro_type) — full version
# ============================================
expr_matrix <- GetAssayData(non_ht_seurat_clean, layer = "data")
genes <- VariableFeatures(non_ht_seurat_clean)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(non_ht_seurat_clean$neuro_type)

rf_nonht <- ranger(x = X, y = y, num.trees = 500, num.threads = 8, classification = TRUE)

cat("\nOOB Error:", rf_nonht$prediction.error, "\n")

conf_mat <- rf_nonht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("Diagonal accuracy:\n")
print(round(diag(confusionRatio), 2))

cat("\nCell counts for all neuro_type:\n")
print(table(non_ht_seurat_clean$neuro_type))

# ---- Take all types (threshold 10) ----
df_num <- as.data.frame(table(non_ht_seurat_clean$neuro_type))
select_cell <- as.character(df_num[df_num$Freq >= 10, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))

type_order_present <- type_order[type_order %in% select_cell]
mat_to_plot <- confusionRatio[type_order_present, type_order_present, drop = FALSE]

cat("\nNumber of types included in heatmap:", nrow(mat_to_plot), "\n")
cat("Excluded types:", setdiff(type_order, type_order_present), "\n")

# ---- Value labels ----
num_labels <- matrix("", nrow = nrow(mat_to_plot), ncol = ncol(mat_to_plot))
diag(num_labels) <- sprintf("%.2f", diag(mat_to_plot))

# ============================================
# ★ Core: save at the gtable's actual size, never cropped
# ============================================
save_pheatmap_exact <- function(ph_obj, file_base, dpi = 150) {

  # Calculate the figure's actual required width and height (inches) from the gtable
  w_in <- convertWidth(sum(ph_obj$gtable$widths),  "inch", valueOnly = TRUE)
  h_in <- convertHeight(sum(ph_obj$gtable$heights), "inch", valueOnly = TRUE)

  cat(sprintf("Actual size: %.1f x %.1f inch\n", w_in, h_in))
  # --- PNG ---
  png(paste0(file_base, ".png"),
      width = w_in, height = h_in, units = "in", res = dpi)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  # --- PDF ---
  pdf(paste0(file_base, ".pdf"), width = w_in, height = h_in)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ---- 8.1 Version with values ----
p_num <- pheatmap(mat_to_plot,
                  breaks = seq(0, 1, 0.01),
                  cluster_rows = FALSE,
                  cluster_cols = FALSE,
                  border_color = NA,
                  cellwidth = 30,
                  cellheight = 30,
                  fontsize = 10,
                  color = viridis(100),
                  main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
                  display_numbers = num_labels,
                  fontsize_number = 8,
                  number_color = "white")

save_pheatmap_exact(p_num, paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_withNum"))

# ---- 8.2 Version without values ----
p_nonum <- pheatmap(mat_to_plot,
                    breaks = seq(0, 1, 0.01),
                    cluster_rows = FALSE,
                    cluster_cols = FALSE,
                    border_color = NA,
                    cellwidth = 30,
                    cellheight = 30,
                    fontsize = 10,
                    color = viridis(100),
                    main = "non-5-HT Neurotransmitter Types Ranger Confusion Matrix",
                    display_numbers = FALSE)

save_pheatmap_exact(p_nonum, paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_noNum"))

# ---- Display on screen ----
grid.newpage()
grid.draw(p_nonum$gtable)

cat("Ranger heatmap PNG + PDF saved\n")

In [ ]:
library(viridis)
library(grid)

# ============================================
# 8. Ranger + heatmap (keep coloring and labels for only the Top2 in each column)
# ============================================
expr_matrix <- GetAssayData(non_ht_seurat_clean, layer = "data")
genes <- VariableFeatures(non_ht_seurat_clean)

X <- t(as.matrix(expr_matrix[genes, , drop = FALSE]))
y <- factor(non_ht_seurat_clean$neuro_type)

rf_nonht <- ranger(x = X, y = y, num.trees = 500, num.threads = 8, classification = TRUE)

cat("\nOOB Error:", rf_nonht$prediction.error, "\n")

conf_mat <- rf_nonht$confusion.matrix
confusionRatio <- t(apply(conf_mat, 1, function(x) {
  if(sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("Diagonal accuracy:\n")
print(round(diag(confusionRatio), 2))

# ---- Prepare the matrix ----
df_num <- as.data.frame(table(non_ht_seurat_clean$neuro_type))
select_cell <- as.character(df_num[df_num$Freq >= 10, ]$Var1)
select_cell <- intersect(select_cell, rownames(confusionRatio))

type_order_present <- type_order[type_order %in% select_cell]
mat_to_plot <- confusionRatio[type_order_present, type_order_present, drop = FALSE]

cat("\nNumber of types included in heatmap:", nrow(mat_to_plot), "\n")

# ============================================
# ★ Keep only the Top2 in each column, set the rest to color 0
# ============================================
mat_color <- mat_to_plot
mat_labels <- matrix("", nrow = nrow(mat_to_plot),
                     ncol = ncol(mat_to_plot),
                     dimnames = dimnames(mat_to_plot))

for (j in seq_len(ncol(mat_to_plot))) {
  ord <- order(mat_to_plot[, j], decreasing = TRUE)
  top2 <- ord[1:2]
  mat_color[-top2, j] <- 0
  mat_labels[top2, j] <- sprintf("%.2f", mat_to_plot[top2, j])
}

# ---- Make sure the output directory exists ----
if (!dir.exists(output_path)) {
  dir.create(output_path, recursive = TRUE)
}

# ============================================
# ★ Save at the gtable's actual size, never cropped
# ============================================
save_pheatmap_exact <- function(ph_obj, file_base, dpi = 150) {

  w_in <- convertWidth(sum(ph_obj$gtable$widths),  "inch", valueOnly = TRUE)
  h_in <- convertHeight(sum(ph_obj$gtable$heights), "inch", valueOnly = TRUE)

  cat(sprintf("Actual size: %.1f x %.1f inch\n", w_in, h_in))

  # --- PNG ---
  png(paste0(file_base, ".png"),
      width = w_in, height = h_in, units = "in", res = dpi)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  # --- PDF ---
  pdf(paste0(file_base, ".pdf"), width = w_in, height = h_in)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ---- 8.1 Version with values (only Top2 of each column labeled) ----
p_num <- pheatmap(mat_color,
                  breaks = seq(0, 1, 0.01),
                  cluster_rows = FALSE,
                  cluster_cols = FALSE,
                  border_color = NA,
                  cellwidth = 30, cellheight = 30, fontsize = 10,
                  color = viridis(100),
                  main = "non-5-HT Neurotransmitter Types Ranger (Top2 per column)",
                  display_numbers = mat_labels,
                  fontsize_number = 8, number_color = "white")

save_pheatmap_exact(p_num, paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_Top2_withNum"))

# ---- 8.2 Version without values (colors also only Top2) ----
p_nonum <- pheatmap(mat_color,
                    breaks = seq(0, 1, 0.01),
                    cluster_rows = FALSE,
                    cluster_cols = FALSE,
                    border_color = NA,
                    cellwidth = 30, cellheight = 30, fontsize = 10,
                    color = viridis(100),
                    main = "non-5-HT Neurotransmitter Types Ranger (Top2 per column)",
                    display_numbers = FALSE)

save_pheatmap_exact(p_nonum, paste0(output_path, "nonHT_NeuroType_Ranger_Heatmap_Top2_noNum"))

# ---- Display on screen ----
grid.newpage()
grid.draw(p_nonum$gtable)

cat("Ranger Top2 heatmap PNG + PDF all saved\n")

In [ ]:
library(Seurat)

# ============================================
# Load the ht_seurat object
# ============================================
ht_seurat <- readRDS("./data/classFiltSeurat_neurotype.rds")

# ============================================
# Quick check of the loaded object
# ============================================
cat("===== Object overview =====\n")
cat("Number of cells:", ncol(ht_seurat), "\n")
cat("Number of genes:", nrow(ht_seurat), "\n")
cat("Assays:", paste(Assays(ht_seurat), collapse = ", "), "\n")
cat("Reductions:", paste(Reductions(ht_seurat), collapse = ", "), "\n")
cat("Default Assay:", DefaultAssay(ht_seurat), "\n")

# View metadata columns
cat("\n===== Metadata columns =====\n")
print(colnames(ht_seurat@meta.data))

# View 5-HT subtype distribution (if ht_subtype column exists)
if("ht_subtype" %in% colnames(ht_seurat@meta.data)) {
  cat("\n===== Distribution of 5-HT subtypes =====\n")
  print(table(ht_seurat$ht_subtype))
}

# View current Ident
cat("\n===== Current Ident =====\n")
print(table(Idents(ht_seurat)))

In [ ]:
print(table(ht_seurat$cell_type))

In [ ]:
colnames(ht_seurat@meta.data)
head(ht_seurat@meta.data, 3)

In [ ]:
# # -------- Re-clustering of 5-HT neurons --------

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(ranger)
library(pheatmap)
library(viridis)
library(grid)
library(gtable)
library(future)
library(harmony)

# ============================================
# ============================================
# 1. Extract 5-HT neurons (using the cells argument, no expression evaluation)
# ============================================
cat("\nTotal cells before extraction:", ncol(ht_seurat), "\n")

cells_keep <- rownames(ht_seurat@meta.data)[
  grepl("^5-HT_", as.character(ht_seurat$cell_type))
]

ht_5ht <- subset(ht_seurat, cells = cells_keep)

cat("\nNumber of 5-HT cells after extraction:", ncol(ht_5ht), "\n")
print(table(ht_5ht$cell_type))


# ---------- 1.0 nFeature > 1500 filter (applied to ht_5ht!) ----------
cat("Number of 5-HT neurons (before filter):", ncol(ht_5ht), "\n")

ht_5ht <- subset(ht_5ht, subset = nFeature_RNA > 1500)

cat("Number of 5-HT neurons (after nFeature > 1500):", ncol(ht_5ht), "\n")

# ---------- 1.1 SCTransform + PCA ----------
ht_5ht <- SCTransform(ht_5ht,
                      vst.flavor = "v2",
                      variable.features.n = 3000,
                      vars.to.regress = "percent.mt",
                      method = "glmGamPoi",
                      verbose = FALSE)
ht_5ht <- RunPCA(ht_5ht, npcs = 50, verbose = FALSE)

# ---------- 1.2 Harmony (donor batch correction) ----------
ht_5ht <- RunHarmony(ht_5ht,
                     group.by.vars = "donor",
                     reduction.use = "pca",
                     dims.use = 1:30,
                     assay.use = "SCT",
                     verbose = FALSE)

# ---------- 1.3 Nearest-neighbor graph ----------
ht_5ht <- FindNeighbors(ht_5ht, reduction = "harmony", dims = 1:30)

In [ ]:
save_path <- "./data/"

In [ ]:
# ============================================
# ============================================
# ★ 1.4 Multi-resolution clustering + clustree (on ht_5ht)
# ============================================
resolutions_to_try <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.8, 1.0, 1.2)

for (res in resolutions_to_try) {
  ht_5ht <- FindClusters(ht_5ht,
                         resolution = res,
                         future.seed = TRUE,
                         verbose = FALSE)
}

# clustree plot: check which resolution gives the most stable clustering structure
p_clustree <- clustree(ht_5ht, prefix = "SCT_snn_res.")

print(p_clustree)

ggsave(paste0(save_path, "ht_clustree.png"), p_clustree,
       width = 12, height = 10, dpi = 300)
ggsave(paste0(save_path, "ht_clustree.pdf"), p_clustree,
       width = 12, height = 10)

# Auxiliary: number of clusters and smallest cluster size at each resolution
cat("\nClustering summary at each resolution:\n")
for (res in resolutions_to_try) {
  col <- paste0("SCT_snn_res.", res)
  tab <- table(ht_5ht@meta.data[[col]])
  cat(sprintf("res=%.1f: %d clusters, smallest cluster has %d cells\n",
              res, length(tab), min(tab)))
}

In [ ]:
# ============================================
# ============================================
# ★ 2. Final clustering + UMAP at the resolution chosen from clustree
# ============================================
final_res <- 0.5   # ← change to your chosen resolution after reviewing clustree

# The corresponding column was already generated in the 1.4 loop, just set the ident
ht_5ht <- SetIdent(ht_5ht,
                   value = paste0("SCT_snn_res.", final_res))

n_clust <- length(unique(Idents(ht_5ht)))

ht_5ht$ht_subtype <- paste0("5-HT_", as.character(Idents(ht_5ht)))
ht_5ht$ht_subtype <- factor(ht_5ht$ht_subtype,
                            levels = paste0("5-HT_", 0:(n_clust - 1)))
ht_5ht <- SetIdent(ht_5ht, value = "ht_subtype")

cat("\nFinal subtype distribution (res =", final_res, "):\n")
print(table(ht_5ht$ht_subtype))

# ---------- UMAP ----------
ht_5ht <- RunUMAP(ht_5ht, reduction = "harmony", dims = 1:30)

# ---------- UMAP overview plot (save PNG + PDF along the way) ----------
p_umap <- DimPlot(ht_5ht, group.by = "ht_subtype",
                  label = TRUE, repel = TRUE) +
  coord_fixed() +
  ggtitle(paste0("5-HT Neuron Subtypes (res = ", final_res, ")")) +
  theme(plot.title = element_text(hjust = 0.5))
print(p_umap)

ggsave(paste0(save_path, "ht_5ht_UMAP_res", gsub("\\.", "", final_res), ".png"),
       p_umap, width = 7, height = 6, dpi = 300)
ggsave(paste0(save_path, "ht_5ht_UMAP_res", gsub("\\.", "", final_res), ".pdf"),
       p_umap, width = 7, height = 6)



In [ ]:
# ---------- Save the 5-HT subset object (filename includes the actual resolution) ----------
saveRDS(ht_5ht,
        paste0(save_path, "ht_5ht_nFeature1500_SCT_harmony_res",
               gsub("\\.", "", final_res), ".rds"))
cat("5-HT subset object saved\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(future)
library(harmony)

save_path <- "./data/"

# ============================================
# ============================================
# 0. Load the full object + back it up immediately
# ============================================
all_seurat <- readRDS(paste0(save_path, "classFiltSeurat_neurotype.rds"))

cat("===== Full object overview =====\n")
cat("Number of cells:", ncol(all_seurat), "\n")
cat("Metadata columns:", paste(colnames(all_seurat@meta.data), collapse = ", "), "\n")

In [ ]:
# ============================================
# ============================================
# 3. Write the new subtype names back to the full object → check table first → then save
# ============================================

all_seurat$cell_type_new <- as.character(all_seurat$cell_type)

# Mapping table of new 5-HT subtypes (object name ht_5ht, consistent with the clustering code)
map_new <- setNames(as.character(ht_5ht$ht_subtype), colnames(ht_5ht))
all_seurat$cell_type_new[names(map_new)] <- map_new

# ---------- Handle 5-HT cells discarded by the nFeature filter ----------
old_5ht_cells <- rownames(all_seurat@meta.data)[
  grepl("^5-HT_", as.character(all_seurat$cell_type))
]
not_updated <- setdiff(old_5ht_cells, names(map_new))

cat("Number of 5-HT cells not covered by the new clustering:", length(not_updated), "\n")

# Label these cells as 5-HT_filtered to avoid confusion with the old names of new subtypes
all_seurat$cell_type_new[not_updated] <- "5-HT_filtered"

all_seurat$cell_type_new <- factor(all_seurat$cell_type_new)

# ---------- Check ----------
cat("\n===== Distribution of all cells after writing back =====\n")
print(table(all_seurat$cell_type_new))

cat("\n===== 5-HT section verification =====\n")
cat(sprintf("Total original 5-HT cells: %d\n", length(old_5ht_cells)))
cat(sprintf("Re-clustered and renamed: %d\n", length(old_5ht_cells) - length(not_updated)))
cat(sprintf("Filtered out due to nFeature < 1500: %d (labeled as 5-HT_filtered)\n", length(not_updated)))
cat(sprintf("Number of new 5-HT subtypes: %d\n", length(unique(ht_5ht$ht_subtype))))

# Total count self-check: total cell count must be identical before and after write-back (labels only, no cell removal)
cat(sprintf("Total cell count check: before write-back %d → after write-back %d\n",
            nrow(all_seurat@meta.data), length(all_seurat$cell_type_new)))



In [ ]:
# ---------- Execute the save only after confirming the table output is correct ----------
saveRDS(all_seurat,
        paste0(save_path, "all_seurat_with_newHT_res",
               gsub("\\.", "", final_res), ".rds"))
cat("Full object saved\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)

# ============================================================
save_path <- "./data/"

ht_seurat <- readRDS(paste0(save_path, "ht_5ht_nFeature1500_SCT_harmony_res05.rds"))

# ============================================
# ============================================
# 0. Quick check
# ============================================
cat("===== Object overview =====\n")
cat("Number of cells:", ncol(ht_seurat), "\n")
cat("Number of genes:", nrow(ht_seurat), "\n")
cat("Assays:", paste(Assays(ht_seurat), collapse = ", "), "\n")
cat("Reductions:", paste(Reductions(ht_seurat), collapse = ", "), "\n")

cat("\n===== Subtype distribution =====\n")
print(table(ht_seurat$ht_subtype))

# Make sure the factor order and Ident are correct
ht_levels <- paste0("5-HT_", sort(as.numeric(gsub("5-HT_", "", 
                        unique(as.character(ht_seurat$ht_subtype))))))
ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = ht_levels)
ht_seurat <- SetIdent(ht_seurat, value = "ht_subtype")

# ★ Unified save function: PNG + PDF
save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# Fig 1: UMAP overview
# ============================================
p_umap <- DimPlot(ht_seurat, group.by = "ht_subtype", 
                  label = TRUE, repel = TRUE, raster = FALSE) +
  coord_fixed() +
  ggtitle("5-HT Neuron Subtypes") +
  theme(plot.title = element_text(hjust = 0.5))
print(p_umap)
save_fig(p_umap, paste0(save_path, "Fig1_UMAP_subtypes"), 7, 6)

# ============================================
# Fig 2: Check batches by donor / sample
# ============================================
p_donor <- DimPlot(ht_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
p_sample <- DimPlot(ht_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p_donor)
print(p_sample)
save_fig(p_donor,  paste0(save_path, "Fig2_UMAP_donor"),  7, 6)
save_fig(p_sample, paste0(save_path, "Fig2_UMAP_sample"), 7, 6)

# ============================================
# Fig 3: Core marker FeaturePlot
# ============================================
p_feat <- FeaturePlot(ht_seurat, 
                      features = c("TPH2", "SLC6A4", "FEV", "DDC"),
                      ncol = 2, raster = FALSE) & coord_fixed()
print(p_feat)
save_fig(p_feat, paste0(save_path, "Fig3_Feature_5HT_markers"), 9, 8)

# ============================================
# Fig 4: Core markers + contamination markers DotPlot (QC re-check)
# ============================================
p_dot <- DotPlot(ht_seurat,
                 features = c("TPH2","SLC6A4","FEV","DDC","TPH1",
                              "CHAT","SLC5A7","SLC17A6","GAD1","GAD2",
                              "CST3","APOE","PTPRZ1","ARHGAP15"),
                 group.by = "ht_subtype") + 
  RotatedAxis() +
  scale_color_gradientn(colors = c("lightgrey", "red"))
print(p_dot)
save_fig(p_dot, paste0(save_path, "Fig4_DotPlot_QC"), 8, 5)

# ============================================
# ============================================
# Fig 5: Violin plots of classic markers
# ============================================
p_vln <- VlnPlot(ht_seurat,
                 features = c("TPH2", "SLC6A4", "SLC17A6", "GAD1", "TH", "FEV"),
                 pt.size = 0, ncol = 3)
print(p_vln)
save_fig(p_vln, paste0(save_path, "Fig5_VlnPlot_markers"), 10, 6)

# ============================================
# Fig 6: Bar plot of cell counts per subtype
# ============================================
cell_counts <- as.data.frame(table(ht_seurat$ht_subtype))
colnames(cell_counts) <- c("subtype", "n")

p_bar <- ggplot(cell_counts, aes(x = subtype, y = n, fill = subtype)) +
  geom_col() +
  geom_text(aes(label = n), vjust = -0.5, size = 3) +
  theme_classic() +
  labs(x = "", y = "Cell number", title = "5-HT subtype cell counts") +
  theme(legend.position = "none", 
        axis.text.x = element_text(angle = 45, hjust = 1))
print(p_bar)
save_fig(p_bar, paste0(save_path, "Fig6_subtype_counts"), 6, 4.5)

# ============================================
# Fig 7: Subtype proportions per donor (stacked bar chart)
# ============================================
prop_df <- ht_seurat@meta.data %>%
  count(donor, ht_subtype) %>%
  group_by(donor) %>%
  mutate(prop = n / sum(n))

p_prop <- ggplot(prop_df, aes(x = donor, y = prop, fill = ht_subtype)) +
  geom_col(position = "fill") +
  theme_classic() +
  labs(x = "", y = "Proportion", fill = "Subtype",
       title = "Subtype composition per donor") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
print(p_prop)
save_fig(p_prop, paste0(save_path, "Fig7_subtype_proportion_donor"), 7, 5)

cat("\nAll figures saved to:", save_path, "\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(ranger)
library(pheatmap)
library(viridis)

output_path <- "./data/"

# ============================================
# ============================================
# 0. Load the res 0.5 5-HT object and set the identity
# ============================================
ht_seurat <- readRDS(paste0(output_path, "ht_5ht_nFeature1500_SCT_harmony_res05.rds"))

cat("Number of cells:", ncol(ht_seurat), "\n")
print(table(ht_seurat$ht_subtype))

# Fix the factor levels in numeric order (5-HT_0 ~ 5-HT_14)
ht_levels <- paste0("5-HT_", sort(as.numeric(gsub("5-HT_", "",
                        unique(as.character(ht_seurat$ht_subtype))))))
ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = ht_levels)
Idents(ht_seurat) <- ht_seurat$ht_subtype

# ★ Unified save function: PNG + PDF
save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 1. Find marker genes (Wilcoxon)
# ============================================
# PrepSCTFindMarkers is required before finding markers on SCT data (corrects for sequencing depth differences)
ht_seurat <- PrepSCTFindMarkers(ht_seurat)

ht_markers <- FindAllMarkers(ht_seurat,
                             only.pos = TRUE,
                             min.pct = 0.25,
                             logfc.threshold = 0.25,
                             test.use = "wilcox")

# Save the full marker table + Top20
write.csv(ht_markers, paste0(output_path, "ht5ht_AllMarkers.csv"), row.names = FALSE)

ht_top20 <- ht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(ht_top20, paste0(output_path, "ht5ht_Subtype_top20_markers.csv"), row.names = FALSE)
cat("5-HT Top20 markers saved\n")

# ============================================
# 2. Top5 DotPlot (15 clusters × up to 75 genes)
# ============================================
ht_top5 <- ht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

ht_gene_order5 <- unique(ht_top5$gene)

p_ht_dot5 <- DotPlot(ht_seurat, features = ht_gene_order5, group.by = "ht_subtype",
                     dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("5-HT subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_ht_dot5)
save_fig(p_ht_dot5, paste0(output_path, "ht5ht_Fig8_Top5_DotPlot"), 14, 14)

write.csv(ht_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "ht5ht_Subtype_Top5_table.csv"), row.names = FALSE)
cat("Top5 DotPlot saved\n")

# ============================================
# 3. Top10 DotPlot (15 clusters × up to 150 genes)
# ============================================
ht_top10 <- ht_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

ht_gene_order10 <- unique(ht_top10$gene)

p_ht_dot10 <- DotPlot(ht_seurat, features = ht_gene_order10, group.by = "ht_subtype",
                      dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("5-HT subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_ht_dot10)
save_fig(p_ht_dot10, paste0(output_path, "ht5ht_Fig9_Top10_DotPlot"), 15, 18)

write.csv(ht_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "ht5ht_Subtype_Top10_table.csv"), row.names = FALSE)
cat("Top10 DotPlot saved\n")

# ============================================
# 4. Top20 DotPlot (15 clusters × up to 300 genes)
# ============================================
ht_gene_order20 <- unique(ht_top20$gene)

p_ht_dot20 <- DotPlot(ht_seurat, features = ht_gene_order20, group.by = "ht_subtype",
                      dot.scale = 3, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("5-HT subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 5.5),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_ht_dot20)
save_fig(p_ht_dot20, paste0(output_path, "ht5ht_Fig10_Top20_DotPlot"), 16, 26)

write.csv(ht_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "ht5ht_Subtype_Top20_table.csv"), row.names = FALSE)
cat("Top20 DotPlot saved\n")

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)

output_path <- "./data"

# ============================================
# ============================================
# 0. Load the 5-HT object (skip if already in memory)
# ============================================
ht_seurat <- readRDS(paste0(output_path, "ht_5ht_nFeature1500_SCT_harmony_res05.rds"))

# Fix the subtype order: 5-HT_0 ~ 5-HT_14
ht_levels <- paste0("5-HT_", sort(as.numeric(gsub("5-HT_", "",
                        unique(as.character(ht_seurat$ht_subtype))))))
ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = ht_levels)

# ★ Unified save function: PNG + PDF
save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 1. User-specified gene list (grouped by 5-HT subtypes 0~14, 44 genes total)
# ============================================
ht_gene_list_raw <- c(
  # 5-HT_0
  "DAPL1", "CHIN2", "DOCK8",
  # 5-HT_1
  "NTS", "CNBD1", "MCOLN3",
  # 5-HT_2
  "SATB2", "LVRN", "ITPR2",
  # 5-HT_3
  "PDLIM3", "LDB2", "SYT10",
  # 5-HT_4
  "TLL1", "NXPH1", "SLCO1C1",
  # 5-HT_5
  "PLCH1", "DLK1", "BMPR1B",
  # 5-HT_6
  "RASSF6", "KCNH1", "KCNH7",
  # 5-HT_7
  "PDE1C", "LUZP2", "C1QL1",
  # 5-HT_8
  "BCL11A", "ANO1", "NFIX",
  # 5-HT_9
  "ABI3BP", "ARHGAP36", "RSPO2",
  # 5-HT_10
  "PENK", "TRPC4", "GRB14",
  # 5-HT_11
  "CALCR", "MBNL3", "SFRP1",
  # 5-HT_12
  "IGF1", "GRIN3A", "PARD3B",
  # 5-HT_13
  "UTS2B", "BCHE", "CLRN1",
  # 5-HT_14
  "EPHA4", "CHST9", "SDC2"
)

# ============================================
# ============================================
# 2. Check gene availability
# ============================================
ht_available_genes <- ht_gene_list_raw[ht_gene_list_raw %in% rownames(ht_seurat)]
ht_missing_genes <- ht_gene_list_raw[!ht_gene_list_raw %in% rownames(ht_seurat)]

cat("Total genes:", length(ht_gene_list_raw), "\n")
cat("Available genes:", length(ht_available_genes), "\n")

if (length(ht_missing_genes) > 0) {
  cat("\nWarning: the following genes were not found and have been skipped:\n")
  cat(paste(ht_missing_genes, collapse = ", "), "\n")
}

# ============================================
# 3. DotPlot (x-axis = subtypes, y-axis = genes)
# ============================================
p_ht_custom <- DotPlot(
  ht_seurat,
  features = ht_available_genes,
  group.by = "ht_subtype",
  dot.scale = 4,
  cols = c("lightgrey", "red")
) +
  coord_flip() +
  RotatedAxis() +
  ggtitle("5-HT subtypes: selected markers") +
  xlab("5-HT subtype") +
  ylab("Gene") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9, face = "bold"),
    axis.text.y = element_text(size = 8),
    axis.title = element_text(size = 12, face = "bold"),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )

print(p_ht_custom)

save_fig(p_ht_custom, paste0(output_path, "ht5ht_Fig11_CustomMarkers_DotPlot"), 12, 14)
cat("5-HT user-specified gene DotPlot saved!\n")

In [ ]:
library(viridis)
library(ranger)
library(pheatmap)
library(grid)

output_path <- "./data/"

# ============================================
# ============================================
# 0. Load the 5-HT object + fix the subtype order
# ============================================
# ht_seurat <- readRDS(paste0(output_path, "ht_5ht_nFeature1500_SCT_harmony_res05.rds"))

ht_type_order <- paste0("5-HT_", sort(as.numeric(gsub("5-HT_", "",
                            unique(as.character(ht_seurat$ht_subtype))))))
ht_seurat$ht_subtype <- factor(ht_seurat$ht_subtype, levels = ht_type_order)
Idents(ht_seurat) <- ht_seurat$ht_subtype

# ============================================
# 1. Ranger (HVGs from the RNA assay)
# ============================================
DefaultAssay(ht_seurat) <- "RNA"

if (length(VariableFeatures(ht_seurat, assay = "RNA")) == 0) {
  cat("VariableFeatures of the RNA assay are empty, recomputing...\n")
  ht_seurat <- FindVariableFeatures(ht_seurat, selection.method = "vst", nfeatures = 3000)
}

expr_matrix_ht <- GetAssayData(ht_seurat, layer = "data", assay = "RNA")
genes_ht <- intersect(VariableFeatures(ht_seurat, assay = "RNA"),
                      rownames(expr_matrix_ht))

cat("Number of genes in the RNA assay:", nrow(expr_matrix_ht), "\n")
cat("Number of HVGs:", length(genes_ht), "\n")

X_ht <- t(as.matrix(expr_matrix_ht[genes_ht, , drop = FALSE]))
y_ht <- factor(ht_seurat$ht_subtype)

cat("Dimensions of X_ht:", dim(X_ht), "\n")

rf_ht <- ranger(x = X_ht, y = y_ht,
                num.trees = 500, num.threads = 8,
                classification = TRUE)

cat("\n5-HT OOB Error:", rf_ht$prediction.error, "\n")

conf_mat_ht <- rf_ht$confusion.matrix
confusionRatio_ht <- t(apply(conf_mat_ht, 1, function(x) {
  if (sum(x) == 0) return(rep(0, length(x)))
  x / sum(x)
}))

cat("5-HT diagonal accuracy:\n")
print(round(diag(confusionRatio_ht), 2))

# ---- Prepare the matrix (fixed subtype order, threshold 20) ----
df_num_ht <- as.data.frame(table(ht_seurat$ht_subtype))
select_cell_ht <- as.character(df_num_ht[df_num_ht$Freq >= 20, ]$Var1)

ht_type_order_present <- ht_type_order[ht_type_order %in% select_cell_ht]
mat_to_plot_ht <- confusionRatio_ht[ht_type_order_present,
                                    ht_type_order_present, drop = FALSE]

cat("\nNumber of types included in the 5-HT heatmap:", nrow(mat_to_plot_ht), "\n")

# ============================================
# ============================================
# 2. Exact-size save function (PNG + PDF, never cropped)
# ============================================
save_pheatmap_exact <- function(ph_obj, file_base, dpi = 150) {

  w_in <- convertWidth(sum(ph_obj$gtable$widths),  "inch", valueOnly = TRUE)
  h_in <- convertHeight(sum(ph_obj$gtable$heights), "inch", valueOnly = TRUE)

  cat(sprintf("Actual size: %.1f x %.1f inch\n", w_in, h_in))

  png(paste0(file_base, ".png"),
      width = w_in, height = h_in, units = "in", res = dpi)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  pdf(paste0(file_base, ".pdf"), width = w_in, height = h_in)
  grid.newpage()
  grid.draw(ph_obj$gtable)
  dev.off()

  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ---- Version with values ----
ht_num_labels <- matrix("", nrow = nrow(mat_to_plot_ht),
                        ncol = ncol(mat_to_plot_ht))
diag(ht_num_labels) <- sprintf("%.2f", diag(mat_to_plot_ht))

p_ht_num <- pheatmap(mat_to_plot_ht,
                     breaks = seq(0, 1, 0.01),
                     cluster_rows = FALSE,
                     cluster_cols = FALSE,
                     border_color = NA,
                     cellwidth = 30, cellheight = 30, fontsize = 10,
                     color = viridis(100),
                     main = "5-HT Subtypes Ranger Confusion Matrix",
                     display_numbers = ht_num_labels,
                     fontsize_number = 8, number_color = "white")

save_pheatmap_exact(p_ht_num,
                    paste0(output_path, "ht5ht_Ranger_Heatmap_withNum"))

# ---- Version without values ----
p_ht_nonum <- pheatmap(mat_to_plot_ht,
                       breaks = seq(0, 1, 0.01),
                       cluster_rows = FALSE,
                       cluster_cols = FALSE,
                       border_color = NA,
                       cellwidth = 30, cellheight = 30, fontsize = 10,
                       color = viridis(100),
                       main = "5-HT Subtypes Ranger Confusion Matrix",
                       display_numbers = FALSE)

save_pheatmap_exact(p_ht_nonum,
                    paste0(output_path, "ht5ht_Ranger_Heatmap_noNum"))

# ---- Display on screen ----
grid.newpage()
grid.draw(p_ht_nonum$gtable)

cat("All 5-HT Ranger heatmaps processed\n")

In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

# ★ Unified save function: PNG + PDF
save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 6. Violin plots (by 5-HT subtype)
# ============================================
ht_plot_data <- ht_seurat@meta.data
ht_plot_data$ht_subtype <- factor(ht_plot_data$ht_subtype, levels = ht_type_order)

# 6.1 nFeature_RNA (one tick every 1000)
ht_nfeature_max <- max(ht_plot_data$nFeature_RNA, na.rm = TRUE)
ht_nfeature_breaks <- seq(0, ceiling(ht_nfeature_max / 1000) * 1000, by = 1000)

p_ht_nfeature <- ggplot(ht_plot_data,
                        aes(x = ht_subtype, y = nFeature_RNA, fill = ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7,
              linewidth = 0.3, color = "black") +
  scale_y_continuous(breaks = ht_nfeature_breaks) +
  labs(x = "", y = "nFeature RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"),
    axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_ht_nfeature)
save_fig(p_ht_nfeature,
         paste0(output_path, "ht5ht_violin_nFeature_RNA_1k"), 12, 6)

# 6.2 nCount_RNA
p_ht_ncount <- ggplot(ht_plot_data,
                      aes(x = ht_subtype, y = nCount_RNA, fill = ht_subtype)) +
  geom_violin(scale = "width", trim = TRUE, width = 0.7,
              linewidth = 0.3, color = "black") +
  labs(x = "", y = "nCount RNA") +
  theme_bw() +
  theme(
    panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
    panel.border = element_blank(), axis.line = element_line(color = "black"),
    axis.ticks = element_line(color = "black"),
    axis.ticks.length = unit(6, "pt"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10),
    axis.title = element_text(size = 13),
    legend.position = "none"
  )
print(p_ht_ncount)
save_fig(p_ht_ncount,
         paste0(output_path, "ht5ht_violin_nCount_RNA"), 12, 6)

cat("5-HT violin plots saved\n")

In [ ]:
library(Seurat)

cat("===== Loading the saved Seurat object =====\n")

save_path <- "./data/"

classFiltSeurat_filt <- readRDS(
  paste0(save_path, "all_seurat_with_newHT_res05.rds")
)

cat("Loaded successfully, total cells:", ncol(classFiltSeurat_filt), "\n")
cat("Total genes:", nrow(classFiltSeurat_filt), "\n")

cat("\nDistribution of cell_type:\n")
print(table(classFiltSeurat_filt$cell_type))

if ("major_group" %in% colnames(classFiltSeurat_filt@meta.data)) {
  cat("\nDistribution of major_group:\n")
  print(table(classFiltSeurat_filt$major_group))
}

cat("\nObject loaded into variable: classFiltSeurat_filt\n")

In [ ]:
# ============================================
# Cluster non-neuronal cells separately
# ============================================

In [ ]:
library(Seurat)
library(ggplot2)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

# ★ Unified save function: PNG + PDF
save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# # ============================================
# # 0. Load the full object
# # ============================================
# all_seurat <- readRDS(paste0(output_path, "classFiltSeurat_neurotype.rds"))

# cat("===== Re-clustering of Astrocytes =====\n")

# ---------- 1. Extract Astrocytes ----------
astro_cells <- colnames(all_seurat)[all_seurat$cell_type == "Astrocytes"]
astro_seurat <- subset(all_seurat, cells = astro_cells)
cat("Number of Astrocytes:", ncol(astro_seurat), "\n")
print(table(astro_seurat$donor))

# ---------- 2. RNA normalization + PCA ----------
DefaultAssay(astro_seurat) <- "RNA"
astro_seurat <- NormalizeData(astro_seurat, verbose = FALSE)
astro_seurat <- FindVariableFeatures(astro_seurat, selection.method = "vst",
                                     nfeatures = 2000, verbose = FALSE)
astro_seurat <- ScaleData(astro_seurat, verbose = FALSE)
astro_seurat <- RunPCA(astro_seurat, features = VariableFeatures(astro_seurat),
                       npcs = 50, verbose = FALSE)

# ---------- 3. Harmony (donor batch correction, consistent with the 5-HT workflow) ----------
astro_seurat <- RunHarmony(astro_seurat,
                           group.by.vars = "donor",
                           reduction.use = "pca",
                           dims.use = 1:30,
                           verbose = FALSE)

# ---------- 4. Nearest-neighbor graph + multi-resolution clustering + clustree ----------
astro_seurat <- FindNeighbors(astro_seurat, reduction = "harmony", dims = 1:30,
                              verbose = FALSE)

resolutions_to_try <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.8, 1.0)
for (res in resolutions_to_try) {
  astro_seurat <- FindClusters(astro_seurat, resolution = res,
                               future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(astro_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "astro_clustree"), 12, 9)

cat("\nClustering summary at each resolution:\n")
for (res in resolutions_to_try) {
  tab <- table(astro_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.1f: %d clusters, smallest cluster has %d cells\n",
              res, length(tab), min(tab)))
}



In [ ]:
# ---------- 5. Choose resolution → UMAP + rename ----------
final_res <- 0.1   # ← modify after reviewing clustree

astro_seurat <- SetIdent(astro_seurat,
                         value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(astro_seurat)))
cl_new  <- as.character(Idents(astro_seurat))
cl_map  <- setNames(paste0("Astrocytes_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

astro_seurat$subtype <- factor(unname(cl_map[cl_new]),
                               levels = paste0("Astrocytes_", 0:(n_clust - 1)))
astro_seurat <- SetIdent(astro_seurat, value = "subtype")

cat("\nDistribution of Astrocyte subtypes (res =", final_res, "):\n")
print(table(astro_seurat$subtype))

astro_seurat <- RunUMAP(astro_seurat, reduction = "harmony", dims = 1:30,
                        verbose = FALSE)

# ---------- 6. Visualization ----------
p1 <- DimPlot(astro_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Astrocyte Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "astro_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(astro_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "astro_Fig2_UMAP_donor"), 7, 6)

p2 <- VlnPlot(astro_seurat, features = c("GFAP", "AQP4", "SLC1A3"),
              pt.size = 0, ncol = 3)
print(p2)
save_fig(p2, paste0(output_path, "astro_Fig3_Vln_ClassicMarkers"), 10, 4)

# Contamination / identity re-check: confirm no neurons or oligodendrocytes are mixed in
p_qc <- DotPlot(astro_seurat,
                features = c("GFAP","AQP4","SLC1A3","ALDH1L1",
                             "SLC1A2","PLP1","MBP","SNAP25","SYT1","SLC17A6","GAD1"),
                group.by = "subtype") +
  RotatedAxis() +
  scale_color_gradientn(colors = c("lightgrey", "red"))
print(p_qc)
save_fig(p_qc, paste0(output_path, "astro_Fig4_DotPlot_QC"), 9, 5)



In [ ]:
# ============================================
# ============================================
# Identity confirmation DotPlot
# ============================================
check_genes <- c(
  # Astrocytic (should light up only in 0/1)
  "GFAP", "AQP4", "SLC1A2", "SLC1A3", "AGT", "FGFR3", "SLC39A12",
  # Neuronal (should light up only in 2)
  "SNAP25", "SYT1", "TUBB3", "NEFL", "CALB2", "TPH2", "ACHE",
  # Microglial (should light up only in 3)
  "CSF1R", "PTPRC", "AIF1", "TYROBP", "C1QA", "CX3CR1",
  # Other lineages to exclude (should be dark everywhere)
  "PLP1", "CLDN5", "DCN"
)

p_dot <- DotPlot(astro_seurat, features = check_genes,
                 group.by = "subtype",
                 dot.scale = 6,
                 cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("Astro subtypes: identity check") +
  theme(
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 10, face = "bold")
  )

print(p_dot)
save_fig(p_dot, paste0(output_path, "astro_Fig8_DotPlot_IdentityCheck"), 12, 4.5)

cat("Confirmation plot saved\n")

In [ ]:
astro_removed <- subset(astro_seurat,
                        subset = subtype %in% c("Astrocytes_2", "Astrocytes_3"))
# saveRDS(astro_removed, paste0(output_path, "astro_removed_2_neuron_3_micro.rds"))
cat("Removed and saved separately:", ncol(astro_removed), "cells\n")
print(table(astro_removed$subtype))

# Clean astrocytes
astro_clean <- subset(astro_seurat,
                      subset = subtype %in% c("Astrocytes_0", "Astrocytes_1"))
cat("Number of astrocytes after cleaning:", ncol(astro_clean), "\n")

# ============================================
# 1. Re-normalization + PCA (must be rerun after cleaning, cannot reuse the old space)
# ============================================
DefaultAssay(astro_clean) <- "RNA"

astro_clean <- NormalizeData(astro_clean, verbose = FALSE)
astro_clean <- FindVariableFeatures(astro_clean, selection.method = "vst",
                                    nfeatures = 2000, verbose = FALSE)
astro_clean <- ScaleData(astro_clean, verbose = FALSE)
astro_clean <- RunPCA(astro_clean, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony (donor correction)
# ============================================
astro_clean <- RunHarmony(astro_clean,
                          group.by.vars = "donor",
                          reduction.use = "pca",
                          dims.use = 1:30,
                          verbose = FALSE)

# ============================================
# 3. Nearest-neighbor graph + multi-resolution clustering + clustree
# ============================================
astro_clean <- FindNeighbors(astro_clean, reduction = "harmony",
                             dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.8, 1.0)
for (res in resolutions_to_try) {
  astro_clean <- FindClusters(astro_clean, resolution = res,
                              future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(astro_clean, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "astroClean_clustree"), 12, 9)

cat("\nClustering summary at each resolution:\n")
for (res in resolutions_to_try) {
  tab <- table(astro_clean@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.1f: %d clusters, smallest cluster has %d cells\n",
              res, length(tab), min(tab)))
}


In [ ]:
# ============================================
# ============================================
# 4. Choose resolution → final clustering + naming + UMAP
# ============================================
final_res <- 0.1   # ← modify after reviewing clustree

astro_clean <- SetIdent(astro_clean, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(astro_clean)))
cl_new  <- as.character(Idents(astro_clean))
cl_map  <- setNames(paste0("Astrocytes_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

astro_clean$subtype <- factor(unname(cl_map[cl_new]),
                              levels = paste0("Astrocytes_", 0:(n_clust - 1)))
astro_clean <- SetIdent(astro_clean, value = "subtype")

cat("\nDistribution of subtypes after cleaning (res =", final_res, "):\n")
print(table(astro_clean$subtype))

astro_clean <- RunUMAP(astro_clean, reduction = "harmony", dims = 1:30,
                       verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(astro_clean, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Astrocytes (cleaned) res = ", final_res))
print(p1)
save_fig(p1, paste0(output_path, "astroClean_Fig1_UMAP"), 7, 6)

p1b <- DimPlot(astro_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "astroClean_Fig2_UMAP_donor"), 7, 6)


# Differential markers (recomputed after cleaning)
astro_clean_markers <- FindAllMarkers(astro_clean, only.pos = TRUE,
                                      min.pct = 0.25, logfc.threshold = 0.25)
write.csv(astro_clean_markers,
          paste0(output_path, "astroClean_AllMarkers.csv"), row.names = FALSE)


In [ ]:

# ============================================
# ============================================
# 6. Save
# ============================================
saveRDS(astro_clean,
        paste0(output_path, "astroClean_seurat_harmony_res",
               gsub("\\.", "", final_res), ".rds"))
cat("Cleaned astro object saved\n")

# ---------- Write back to the full object (check the table first, then uncomment) ----------
all_seurat <- readRDS(paste0(output_path, "classFiltSeurat_neurotype.rds"))

all_seurat$cell_type_new <- as.character(all_seurat$cell_type)

# Write the new subtype names back
map_clean <- setNames(as.character(astro_clean$subtype), colnames(astro_clean))
all_seurat$cell_type_new[names(map_clean)] <- map_clean

# Label the removed ones separately
map_2 <- colnames(astro_removed)[astro_removed$subtype == "Astrocytes_2"]
map_3 <- colnames(astro_removed)[astro_removed$subtype == "Astrocytes_3"]
all_seurat$cell_type_new[map_2] <- "Neuron_contam"
all_seurat$cell_type_new[map_3] <- "Microglia"

all_seurat$cell_type_new <- factor(all_seurat$cell_type_new)

cat("\n===== Distribution of all cells after write-back =====\n")
print(table(all_seurat$cell_type_new))

In [ ]:
# ---------- Distinguish the two batches of Microglia ----------
is_original_micro <- all_seurat$cell_type_new == "Microglia" &
                     all_seurat$cell_type == "Microglia"

is_from_astro <- all_seurat$cell_type_new == "Microglia" &
                 all_seurat$cell_type == "Astrocytes"

cat("Original Microglia:", sum(is_original_micro), "\n")
cat("Microglia re-labeled from Astro:", sum(is_from_astro), "\n")

In [ ]:
cells_to_remove <- c(
  colnames(all_seurat)[all_seurat$cell_type_new == "Neuron_contam"],
  colnames(all_seurat)[is_from_astro]
)

cat("Cells to remove:", length(cells_to_remove),
    "(Neuron_contam + Microglia_fromAstro)\n")

# saveRDS(all_seurat, paste0(output_path, "all_seurat_backup_beforeDel.rds"))

all_seurat <- subset(all_seurat, cells = setdiff(colnames(all_seurat), cells_to_remove))
all_seurat$cell_type_new <- factor(droplevels(all_seurat$cell_type_new))

cat("\n===== Distribution of all cells after removal =====\n")
print(table(all_seurat$cell_type_new))
# Verify: Neuron_contam is gone, Microglia is exactly 6277

# saveRDS(all_seurat, paste0(output_path, "all_seurat_astroClean_v2.rds"))

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

cat("===== Astrocytes (cleaned) Marker analysis =====\n")

# ============================================
# 0. Extract the cleaned astrocytes from all_seurat in memory
# ============================================
astro_seurat <- subset(all_seurat,
                       subset = cell_type_new %in% c("Astrocytes_0", "Astrocytes_1"))

cat("Number of extracted astrocytes:", ncol(astro_seurat), "\n")
print(table(astro_seurat$cell_type_new))

DefaultAssay(astro_seurat) <- "RNA"

astro_seurat$subtype <- factor(as.character(astro_seurat$cell_type_new),
                               levels = c("Astrocytes_0", "Astrocytes_1"))

# 1. Set Idents
Idents(astro_seurat) <- astro_seurat$subtype

# 2. Find marker genes
astro_markers <- FindAllMarkers(astro_seurat,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

write.csv(astro_markers,
          paste0(output_path, "astroClean_AllMarkers.csv"), row.names = FALSE)

# 3. Save Top20 as CSV
astro_top20 <- astro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(astro_top20,
          paste0(output_path, "astroClean_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers saved\n")
# 4. Top5 DotPlot
astro_top5 <- astro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(astro_top5$gene)

p_astro_dot5 <- DotPlot(astro_seurat, features = gene_order5,
                        group.by = "subtype",
                        dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Astro subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_astro_dot5)
save_fig(p_astro_dot5, paste0(output_path, "astroClean_Fig4_Top5_DotPlot"), 9, 6)

write.csv(astro_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "astroClean_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 DotPlot saved\n")

# 5. Top10 DotPlot
astro_top10 <- astro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(astro_top10$gene)

p_astro_dot10 <- DotPlot(astro_seurat, features = gene_order10,
                         group.by = "subtype",
                         dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Astro subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_astro_dot10)
save_fig(p_astro_dot10, paste0(output_path, "astroClean_Fig5_Top10_DotPlot"), 9, 8)

write.csv(astro_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "astroClean_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 DotPlot saved\n")

# 6. Top20 DotPlot
gene_order20 <- unique(astro_top20$gene)

p_astro_dot20 <- DotPlot(astro_seurat, features = gene_order20,
                         group.by = "subtype",
                         dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Astro subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_astro_dot20)
save_fig(p_astro_dot20, paste0(output_path, "astroClean_Fig6_Top20_DotPlot"), 10, 12)

write.csv(astro_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "astroClean_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 DotPlot saved\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 0. Load the full object + extract Oligo
# ============================================
# all_seurat <- readRDS(paste0(output_path, "classFiltSeurat_neurotype.rds"))

cat("===== Re-clustering of Oligo =====\n")

oligo_cells <- colnames(all_seurat)[all_seurat$cell_type == "Oligo"]
oligo_seurat <- subset(all_seurat, cells = oligo_cells)
cat("Number of Oligo cells:", ncol(oligo_seurat), "\n")
print(table(oligo_seurat$donor))

# ============================================
# 1. RNA normalization + PCA
# ============================================
DefaultAssay(oligo_seurat) <- "RNA"
oligo_seurat <- NormalizeData(oligo_seurat, verbose = FALSE)
oligo_seurat <- FindVariableFeatures(oligo_seurat, selection.method = "vst",
                                     nfeatures = 2000, verbose = FALSE)
oligo_seurat <- ScaleData(oligo_seurat, verbose = FALSE)
oligo_seurat <- RunPCA(oligo_seurat, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony (donor correction)
# ============================================
oligo_seurat <- RunHarmony(oligo_seurat,
                           group.by.vars = "donor",
                           reduction.use = "pca",
                           dims.use = 1:30,
                           verbose = FALSE)

# ============================================
# 3. Nearest-neighbor graph + multi-resolution clustering + clustree
# ============================================
oligo_seurat <- FindNeighbors(oligo_seurat, reduction = "harmony",
                              dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.8, 1.0)
for (res in resolutions_to_try) {
  oligo_seurat <- FindClusters(oligo_seurat, resolution = res,
                               future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(oligo_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "oligo_clustree"), 12, 9)

cat("\nClustering summary at each resolution:\n")
for (res in resolutions_to_try) {
  tab <- table(oligo_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.1f: %d clusters, smallest cluster has %d cells\n",
              res, length(tab), min(tab)))
}

In [ ]:
# ============================================
## ============================================
# 4. Choose resolution → final clustering + naming + UMAP
# ============================================
final_res <- 0.1   # ← modify after reviewing clustree (0.1~0.2 is usually enough for Oligo)

oligo_seurat <- SetIdent(oligo_seurat, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(oligo_seurat)))
cl_new  <- as.character(Idents(oligo_seurat))
cl_map  <- setNames(paste0("Oligo_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

oligo_seurat$subtype <- factor(unname(cl_map[cl_new]),
                               levels = paste0("Oligo_", 0:(n_clust - 1)))
oligo_seurat <- SetIdent(oligo_seurat, value = "subtype")

cat("\nDistribution of Oligo subtypes (res =", final_res, "):\n")
print(table(oligo_seurat$subtype))

oligo_seurat <- RunUMAP(oligo_seurat, reduction = "harmony", dims = 1:30,
                        verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(oligo_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Oligo Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "oligo_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(oligo_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "oligo_Fig2_UMAP_donor"), 7, 6)

p1c <- DimPlot(oligo_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p1c)
save_fig(p1c, paste0(output_path, "oligo_Fig2_UMAP_sample"), 7, 6)

# ---------- Classic marker violin plots ----------
p2 <- VlnPlot(oligo_seurat, features = c("MBP", "MOG", "PLP1", "MAG",
                                         "MOBP", "OPALIN"),
              pt.size = 0, ncol = 3)
print(p2)
save_fig(p2, paste0(output_path, "oligo_Fig3_Vln_ClassicMarkers"), 10, 6)


# ---------- Marker table ----------
oligo_markers <- FindAllMarkers(oligo_seurat, only.pos = TRUE,
                                min.pct = 0.25, logfc.threshold = 0.25)
write.csv(oligo_markers,
          paste0(output_path, "oligo_AllMarkers.csv"), row.names = FALSE)

In [ ]:
DefaultAssay(oligo_seurat) <- "RNA"

oligo_levels <- paste0("Oligo_", sort(as.numeric(
  gsub("Oligo_", "", unique(as.character(oligo_seurat$subtype))))))
oligo_seurat$subtype <- factor(oligo_seurat$subtype, levels = oligo_levels)

Idents(oligo_seurat) <- oligo_seurat$subtype

# ============================================
# ============================================
# 1. Find markers
# ============================================
oligo_markers <- FindAllMarkers(oligo_seurat,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

write.csv(oligo_markers,
          paste0(output_path, "oligo_AllMarkers.csv"), row.names = FALSE)

# ============================================
# 2. Top20 CSV
# ============================================
oligo_top20 <- oligo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(oligo_top20,
          paste0(output_path, "oligo_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers saved\n")

# ============================================
# 3. Top5 DotPlot
# ============================================
oligo_top5 <- oligo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(oligo_top5$gene)

p_oligo_dot5 <- DotPlot(oligo_seurat, features = gene_order5,
                        group.by = "subtype",
                        dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Oligo subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_oligo_dot5)
save_fig(p_oligo_dot5, paste0(output_path, "oligo_Fig5_Top5_DotPlot"), 9, 6)

write.csv(oligo_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligo_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 DotPlot saved\n")

# ============================================
# 4. Top10 DotPlot
# ============================================
oligo_top10 <- oligo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(oligo_top10$gene)

p_oligo_dot10 <- DotPlot(oligo_seurat, features = gene_order10,
                         group.by = "subtype",
                         dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Oligo subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_oligo_dot10)
save_fig(p_oligo_dot10, paste0(output_path, "oligo_Fig6_Top10_DotPlot"), 9, 8)

write.csv(oligo_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligo_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 DotPlot saved\n")

# ============================================
# 5. Top20 DotPlot
# ============================================
gene_order20 <- unique(oligo_top20$gene)

p_oligo_dot20 <- DotPlot(oligo_seurat, features = gene_order20,
                         group.by = "subtype",
                         dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Oligo subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_oligo_dot20)
save_fig(p_oligo_dot20, paste0(output_path, "oligo_Fig7_Top20_DotPlot"), 10, 12)

write.csv(oligo_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligo_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 DotPlot saved\n")

In [ ]:
# ---------- 1. Confirmation DotPlot: focus on OPALIN/MYRF vs neuronal genes in Oligo_0 ----------
DefaultAssay(oligo_seurat) <- "RNA"

check_genes <- c("OPALIN", "MYRF", "MAG", "CLDN11", "MBP",      # oligodendrocytes (should light up in 0/1)
                 "SNAP25", "SYT1", "TUBB3", "NEFL", "CHGB",     # neuronal (should light up only in 2)
                 "PDGFRA", "CSPG4",                            # OPC
                 "GFAP", "AQP4",                               # astrocytes
                 "CSF1R", "AIF1")                              # microglia

p_check <- DotPlot(oligo_seurat, features = check_genes,
                   group.by = "subtype",
                   dot.scale = 6, cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("Oligo subtypes: identity check") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        axis.text.y = element_text(size = 10, face = "bold"))
print(p_check)
ggsave(paste0(output_path, "oligo_Fig8_DotPlot_IdentityCheck.png"),
       p_check, width = 12, height = 4.5, dpi = 300)
ggsave(paste0(output_path, "oligo_Fig8_DotPlot_IdentityCheck.pdf"),
       p_check, width = 12, height = 4.5)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 0. Remove Oligo_2 (neurons), keep only true oligodendrocytes
# ============================================
oligo_clean <- subset(oligo_seurat,
                      subset = subtype %in% c("Oligo_0", "Oligo_1"))
cat("Number of oligodendrocytes after cleaning:", ncol(oligo_clean), "\n")

# ============================================
# 1. Re-normalization + PCA
# ============================================
DefaultAssay(oligo_clean) <- "RNA"

oligo_clean <- NormalizeData(oligo_clean, verbose = FALSE)
oligo_clean <- FindVariableFeatures(oligo_clean, selection.method = "vst",
                                    nfeatures = 2000, verbose = FALSE)
oligo_clean <- ScaleData(oligo_clean, verbose = FALSE)
oligo_clean <- RunPCA(oligo_clean, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony (donor correction)
# ============================================
oligo_clean <- RunHarmony(oligo_clean,
                          group.by.vars = "donor",
                          reduction.use = "pca",
                          dims.use = 1:30,
                          verbose = FALSE)

# ============================================
# 3. Nearest-neighbor graph + multi-resolution clustering + clustree
# ============================================
oligo_clean <- FindNeighbors(oligo_clean, reduction = "harmony",
                             dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.1, 0.2, 0.3, 0.4, 0.5, 0.8)
for (res in resolutions_to_try) {
  oligo_clean <- FindClusters(oligo_clean, resolution = res,
                              future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(oligo_clean, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "oligoClean_clustree"), 12, 9)

cat("\nClustering summary at each resolution:\n")
for (res in resolutions_to_try) {
  tab <- table(oligo_clean@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.1f: %d clusters, smallest cluster has %d cells\n",
              res, length(tab), min(tab)))
}


In [ ]:
# ============================================
# ============================================
# 4. Choose resolution → final clustering + naming + UMAP
# ============================================
final_res_clean <- 0.1   # ← modify after reviewing clustree

oligo_clean <- SetIdent(oligo_clean, value = paste0("RNA_snn_res.", final_res_clean))

n_clust <- length(unique(Idents(oligo_clean)))
cl_new  <- as.character(Idents(oligo_clean))
cl_map  <- setNames(paste0("Oligo_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

oligo_clean$subtype <- factor(unname(cl_map[cl_new]),
                              levels = paste0("Oligo_", 0:(n_clust - 1)))
oligo_clean <- SetIdent(oligo_clean, value = "subtype")

cat("\nDistribution of subtypes after cleaning (res =", final_res_clean, "):\n")
print(table(oligo_clean$subtype))

oligo_clean <- RunUMAP(oligo_clean, reduction = "harmony", dims = 1:30,
                       verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(oligo_clean, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Oligo (cleaned) res = ", final_res_clean))
print(p1)
save_fig(p1, paste0(output_path, "oligoClean_Fig1_UMAP"), 7, 6)

p1b <- DimPlot(oligo_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "oligoClean_Fig2_UMAP_donor"), 7, 6)

# ---------- Classic marker violin plots ----------
p2 <- VlnPlot(oligo_clean, features = c("MBP", "MOG", "PLP1", "MAG",
                                        "MOBP", "OPALIN", "ENPP6", "BCAS1"),
              pt.size = 0, ncol = 4)
print(p2)
save_fig(p2, paste0(output_path, "oligoClean_Fig3_Vln_ClassicMarkers"), 12, 6)


# ---------- Marker table ----------
oligo_clean_markers <- FindAllMarkers(oligo_clean, only.pos = TRUE,
                                      min.pct = 0.25, logfc.threshold = 0.25)
write.csv(oligo_clean_markers,
          paste0(output_path, "oligoClean_AllMarkers.csv"), row.names = FALSE)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

cat("===== Recomputing markers for Oligo (cleaned) =====\n")

# ============================================
# 0. Make sure the object state is correct (skip if already in memory)
# ============================================
DefaultAssay(oligo_clean) <- "RNA"

oligo_levels <- paste0("Oligo_", sort(as.numeric(
  gsub("Oligo_", "", unique(as.character(oligo_clean$subtype))))))
oligo_clean$subtype <- factor(oligo_clean$subtype, levels = oligo_levels)
Idents(oligo_clean) <- oligo_clean$subtype

# ============================================
# 1. ★ Recompute markers
# ============================================
oligo_clean_markers <- FindAllMarkers(oligo_clean,
                                      only.pos = TRUE,
                                      min.pct = 0.25,
                                      logfc.threshold = 0.25,
                                      test.use = "wilcox")

write.csv(oligo_clean_markers,
          paste0(output_path, "oligoClean_AllMarkers.csv"), row.names = FALSE)
cat("Marker recomputation complete,", nrow(oligo_clean_markers), "rows\n")

# ============================================
# 2. Extract Top20 + CSV
# ============================================
oligo_clean_top20 <- oligo_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(oligo_clean_top20,
          paste0(output_path, "oligoClean_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers saved\n")

# ============================================
# 3. Top5 DotPlot
# ============================================
oligo_clean_top5 <- oligo_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(oligo_clean_top5$gene)

p_dot5 <- DotPlot(oligo_clean, features = gene_order5,
                  group.by = "subtype",
                  dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Oligo subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
save_fig(p_dot5, paste0(output_path, "oligoClean_Fig5_Top5_DotPlot"), 9, 5)

write.csv(oligo_clean_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligoClean_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 DotPlot saved\n")

# ============================================
# 4. Top10 DotPlot
# ============================================
oligo_clean_top10 <- oligo_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(oligo_clean_top10$gene)

p_dot10 <- DotPlot(oligo_clean, features = gene_order10,
                   group.by = "subtype",
                   dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Oligo subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
save_fig(p_dot10, paste0(output_path, "oligoClean_Fig6_Top10_DotPlot"), 9, 6)

write.csv(oligo_clean_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligoClean_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 DotPlot saved\n")

# ============================================
# 5. Top20 DotPlot
# ============================================
gene_order20 <- unique(oligo_clean_top20$gene)

p_dot20 <- DotPlot(oligo_clean, features = gene_order20,
                   group.by = "subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned Oligo subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
save_fig(p_dot20, paste0(output_path, "oligoClean_Fig7_Top20_DotPlot"), 10, 9)

write.csv(oligo_clean_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "oligoClean_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 DotPlot saved\n")

# ============================================
# 6. Final save of the cleaned object
# ============================================
saveRDS(oligo_clean,
        paste0(output_path, "oligoClean_seurat_harmony_res",
               gsub("\\.", "", final_res_clean), ".rds"))
cat("Cleaned oligo object saved\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================

cat("===== Re-clustering of OPC =====\n")

opc_cells <- colnames(all_seurat)[all_seurat$cell_type == "OPC"]
opc_seurat <- subset(all_seurat, cells = opc_cells)
cat("Number of OPC cells:", ncol(opc_seurat), "\n")
print(table(opc_seurat$donor))

# ============================================
# 1. RNA normalization + PCA
# ============================================
DefaultAssay(opc_seurat) <- "RNA"
opc_seurat <- NormalizeData(opc_seurat, verbose = FALSE)
opc_seurat <- FindVariableFeatures(opc_seurat, selection.method = "vst",
                                   nfeatures = 2000, verbose = FALSE)
opc_seurat <- ScaleData(opc_seurat, verbose = FALSE)
opc_seurat <- RunPCA(opc_seurat, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony (donor correction)
# ============================================
opc_seurat <- RunHarmony(opc_seurat,
                         group.by.vars = "donor",
                         reduction.use = "pca",
                         dims.use = 1:30,
                         verbose = FALSE)

# ============================================
# 3. Nearest-neighbor graph + multi-resolution clustering + clustree
# ============================================
opc_seurat <- FindNeighbors(opc_seurat, reduction = "harmony",
                            dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.05, 0.1, 0.2, 0.3, 0.5, 0.8)
for (res in resolutions_to_try) {
  opc_seurat <- FindClusters(opc_seurat, resolution = res,
                             future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(opc_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "opc_clustree"), 12, 9)

cat("\nClustering summary at each resolution:\n")
for (res in resolutions_to_try) {
  tab <- table(opc_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.1f: %d clusters, smallest cluster has %d cells\n",
              res, length(tab), min(tab)))
}

In [ ]:
# ============================================
# ============================================
# 4. Choose resolution → final clustering + naming + UMAP
# ============================================
final_res <- 0.05   # ← modify after reviewing clustree (0.05~0.2 is usually enough for OPC)

opc_seurat <- SetIdent(opc_seurat, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(opc_seurat)))
cl_new  <- as.character(Idents(opc_seurat))
cl_map  <- setNames(paste0("OPC_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

opc_seurat$subtype <- factor(unname(cl_map[cl_new]),
                             levels = paste0("OPC_", 0:(n_clust - 1)))
opc_seurat <- SetIdent(opc_seurat, value = "subtype")

cat("\nDistribution of OPC subtypes (res =", final_res, "):\n")
print(table(opc_seurat$subtype))

opc_seurat <- RunUMAP(opc_seurat, reduction = "harmony", dims = 1:30,
                      verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(opc_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("OPC Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "opc_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(opc_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "opc_Fig2_UMAP_donor"), 7, 6)

p1c <- DimPlot(opc_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p1c)
save_fig(p1c, paste0(output_path, "opc_Fig2_UMAP_sample"), 7, 6)

# ---------- Classic marker violin plots ----------
p2 <- VlnPlot(opc_seurat, features = c("PDGFRA", "CSPG4", "SOX10",
                                       "PTPRZ1", "OLIG1", "OLIG2"),
              pt.size = 0, ncol = 3)
print(p2)
save_fig(p2, paste0(output_path, "opc_Fig3_Vln_ClassicMarkers"), 10, 6)

# ============================================
# 5. ★ Identity check DotPlot (OPCs easily hide oligodendrocytes / neurons / proliferating cells)
# ============================================
check_genes <- c(
  # OPC
  "PDGFRA", "CSPG4", "PTPRZ1", "BCAN", "OLIG1", "OLIG2", "SOX10",
  # Mature oligodendrocytes (if a subtype shows high expression of these → it is actually a differentiating OL)
  "MBP", "PLP1", "MOG", "MAG", "OPALIN", "ST18", "ENPP6",
  # Neurons (based on prior experience: neurons may be mixed in)
  "SNAP25", "SYT1", "TUBB3", "NEFL", "CHGB",
  # Astrocytes
  "GFAP", "AQP4", "SLC1A2",
  # Microglia
  "CSF1R", "AIF1",
  #  # Proliferation
  "MKI67", "TOP2A", "CENPF"
)

p_check <- DotPlot(opc_seurat, features = check_genes,
                   group.by = "subtype",
                   dot.scale = 6, cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("OPC subtypes: identity check") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        axis.text.y = element_text(size = 10, face = "bold"))
print(p_check)
save_fig(p_check, paste0(output_path, "opc_Fig4_DotPlot_IdentityCheck"), 14, 5)

In [ ]:
library(dplyr)

# ============================================
# ============================================
# 6. Find markers (OPC grouped by subtype)
# ============================================
opc_markers <- FindAllMarkers(opc_seurat,
                              only.pos = TRUE,
                              min.pct = 0.25,
                              logfc.threshold = 0.25,
                              test.use = "wilcox")

write.csv(opc_markers,
          paste0(output_path, "opc_AllMarkers.csv"), row.names = FALSE)
cat("Marker computation complete,", nrow(opc_markers), "rows\n")

# ============================================
# 7. Extract Top20 + CSV
# ============================================
opc_top20 <- opc_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(opc_top20,
          paste0(output_path, "opc_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers saved\n")

# ============================================
# 8. Top5 DotPlot
# ============================================
opc_top5 <- opc_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(opc_top5$gene)

p_opc_dot5 <- DotPlot(opc_seurat, features = gene_order5,
                      group.by = "subtype",
                      dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("OPC subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_opc_dot5)
save_fig(p_opc_dot5, paste0(output_path, "opc_Fig5_Top5_DotPlot"), 9, 5)

write.csv(opc_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opc_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 DotPlot saved\n")

# ============================================
# 9. Top10 DotPlot
# ============================================
opc_top10 <- opc_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(opc_top10$gene)

p_opc_dot10 <- DotPlot(opc_seurat, features = gene_order10,
                       group.by = "subtype",
                       dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("OPC subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_opc_dot10)
save_fig(p_opc_dot10, paste0(output_path, "opc_Fig6_Top10_DotPlot"), 9, 6)

write.csv(opc_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opc_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 DotPlot saved\n")

# ============================================
# 10. Top20 DotPlot
# ============================================
gene_order20 <- unique(opc_top20$gene)

p_opc_dot20 <- DotPlot(opc_seurat, features = gene_order20,
                       group.by = "subtype",
                       dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("OPC subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_opc_dot20)
save_fig(p_opc_dot20, paste0(output_path, "opc_Fig7_Top20_DotPlot"), 10, 9)

write.csv(opc_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opc_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 DotPlot saved\n")



In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 0. Object state (skip if already in memory)
# ============================================
DefaultAssay(opc_seurat) <- "RNA"

opc_levels <- paste0("OPC_", sort(as.numeric(
  gsub("OPC_", "", unique(as.character(opc_seurat$subtype))))))
opc_seurat$subtype <- factor(opc_seurat$subtype, levels = opc_levels)
Idents(opc_seurat) <- opc_seurat$subtype

cat("Cell counts per subtype:\n")
print(table(opc_seurat$subtype))

# ============================================
# Fig 1: Full identity check (see all subtypes at a glance)
# ============================================
check_genes <- c(
  # Quiescent OPC (should light up in OPC_0)
  "PDGFRA", "CSPG4", "PTPRZ1", "SOX10", "CSPG5", "GPM6A", "EGFR",
  # Differentiating (should light up in OPC_1)
  "GPR17", "MYRF", "ENPP6", "BCAS1", "PLP1", "MBP", "UGT8",
  # Proliferating (should light up in OPC_4)
  "MKI67", "TOP2A", "CCNE2", "E2F1",
  # Microglia (should light up in OPC_3, dark everywhere else)
  "CSF1R", "AIF1", "TYROBP", "C1QA", "C1QB", "INPP5D", "ITGAM", "CX3CR1", "APBB1IP",
  # Neurons (focus on OPC_2)
  "SNAP25", "SYT1", "TUBB3", "NEFL", "CHGB", "SCG5",
  # Astrocytes / endothelium (exclude other contamination)
  "GFAP", "AQP4", "SLC1A2", "CLDN5"
)

p_check <- DotPlot(opc_seurat, features = check_genes,
                   group.by = "subtype",
                   dot.scale = 6, cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("OPC subtypes: full identity check") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        axis.text.y = element_text(size = 10, face = "bold"))

print(p_check)
save_fig(p_check, paste0(output_path, "opc_Fig5_DotPlot_IdentityConfirm"), 16, 5.5)

# ============================================
# Fig 2: Zoom in on OPC_2 (it only shows SEMA4C/SCG5 signals, check separately)
# ============================================
opc2_genes <- c("SEMA4C", "SCG5", "MXD4", "FADS2", "CALR",
                # Controls: neurons vs OPC
                "SNAP25", "SYT1", "NEFL", "PDGFRA", "CSPG4", "SOX10", "GPM6A")

p_opc2 <- DotPlot(opc_seurat, features = opc2_genes,
                  group.by = "subtype",
                  dot.scale = 7, cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("OPC_2 zoom-in: neuron or OPC?") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
        axis.text.y = element_text(size = 10, face = "bold"))

print(p_opc2)
save_fig(p_opc2, paste0(output_path, "opc_Fig6_DotPlot_OPC2_ZoomIn"), 11, 5)

# ============================================
# Fig 3: Microglia-focused check (confirm the identity of OPC_3, and check for any microglia missed in 0/1/4)
# ============================================
micro_genes <- c("CSF1R", "AIF1", "C1QA", "C1QB", "C1QC", "TYROBP",
                 "INPP5D", "ITGAM", "CX3CR1", "APBB1IP", "MS4A7", "CD74")

p_micro <- DotPlot(opc_seurat, features = micro_genes,
                   group.by = "subtype",
                   dot.scale = 7, cols = c("lightgrey", "red")) +
  RotatedAxis() +
  ggtitle("Microglia markers across OPC subtypes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
        axis.text.y = element_text(size = 10, face = "bold"))

print(p_micro)
save_fig(p_micro, paste0(output_path, "opc_Fig7_DotPlot_MicrogliaConfirm"), 12, 5)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 0. Remove OPC_3 (microglia), keep only true oligodendrocyte lineage
# ============================================
opc_removed <- subset(opc_seurat, subset = subtype == "OPC_3")
cat("Removed microglial OPC_3:", ncol(opc_removed), "cells\n")
# Keep a copy if needed: saveRDS(opc_removed, paste0(output_path, "opc_removed_3_microglia.rds"))

opc_clean <- subset(opc_seurat,
                    subset = subtype %in% c("OPC_0", "OPC_1", "OPC_2", "OPC_4"))
cat("Number of OPC cells after cleaning:", ncol(opc_clean), "\n")
print(table(opc_clean$subtype))

# ============================================
# 1. Re-normalization + PCA (must be rerun after cleaning)
# ============================================
DefaultAssay(opc_clean) <- "RNA"

opc_clean <- NormalizeData(opc_clean, verbose = FALSE)
opc_clean <- FindVariableFeatures(opc_clean, selection.method = "vst",
                                  nfeatures = 2000, verbose = FALSE)
opc_clean <- ScaleData(opc_clean, verbose = FALSE)
opc_clean <- RunPCA(opc_clean, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony (donor correction)
# ============================================
opc_clean <- RunHarmony(opc_clean,
                        group.by.vars = "donor",
                        reduction.use = "pca",
                        dims.use = 1:30,
                        verbose = FALSE)

# ============================================
# 3. Nearest-neighbor graph + multi-resolution clustering + clustree
# ============================================
opc_clean <- FindNeighbors(opc_clean, reduction = "harmony",
                           dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.05, 0.1, 0.2, 0.3, 0.5)
for (res in resolutions_to_try) {
  opc_clean <- FindClusters(opc_clean, resolution = res,
                            future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(opc_clean, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "opcClean_clustree"), 12, 9)

cat("\nClustering summary at each resolution:\n")
for (res in resolutions_to_try) {
  tab <- table(opc_clean@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.2f: %d clusters, smallest cluster has %d cells\n",
              res, length(tab), min(tab)))
}




In [ ]:
# ============================================
# ============================================
# 4. Choose resolution → final clustering + naming + UMAP
# ============================================
final_res_clean <- 0.05   # ← modify after reviewing clustree

opc_clean <- SetIdent(opc_clean, value = paste0("RNA_snn_res.", final_res_clean))

n_clust <- length(unique(Idents(opc_clean)))
cl_new  <- as.character(Idents(opc_clean))
cl_map  <- setNames(paste0("OPC_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

opc_clean$subtype <- factor(unname(cl_map[cl_new]),
                            levels = paste0("OPC_", 0:(n_clust - 1)))
opc_clean <- SetIdent(opc_clean, value = "subtype")

cat("\nDistribution of subtypes after cleaning (res =", final_res_clean, "):\n")
print(table(opc_clean$subtype))

opc_clean <- RunUMAP(opc_clean, reduction = "harmony", dims = 1:30,
                     verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(opc_clean, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("OPC (cleaned) res = ", final_res_clean))
print(p1)
save_fig(p1, paste0(output_path, "opcClean_Fig1_UMAP"), 7, 6)

p1b <- DimPlot(opc_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "opcClean_Fig2_UMAP_donor"), 7, 6)

# ---------- Classic marker violin plots ----------
p2 <- VlnPlot(opc_clean, features = c("PDGFRA", "CSPG4", "PTPRZ1", "SOX10",
                                      "GPR17", "MKI67"),
              pt.size = 0, ncol = 3)
print(p2)
save_fig(p2, paste0(output_path, "opcClean_Fig3_Vln_ClassicMarkers"), 10, 6)


In [ ]:
library(dplyr)

# ============================================
# ============================================
# 6. Recompute markers (by cleaned subtype)
# ============================================
opc_clean_markers <- FindAllMarkers(opc_clean,
                                    only.pos = TRUE,
                                    min.pct = 0.25,
                                    logfc.threshold = 0.25,
                                    test.use = "wilcox")

write.csv(opc_clean_markers,
          paste0(output_path, "opcClean_AllMarkers.csv"), row.names = FALSE)
cat("Marker recomputation complete,", nrow(opc_clean_markers), "rows\n")

# ============================================
# 7. Extract Top20 + CSV
# ============================================
opc_clean_top20 <- opc_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(opc_clean_top20,
          paste0(output_path, "opcClean_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers saved\n")

# ============================================
# 8. Top5 DotPlot
# ============================================
opc_clean_top5 <- opc_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(opc_clean_top5$gene)

p_dot5 <- DotPlot(opc_clean, features = gene_order5,
                  group.by = "subtype",
                  dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned OPC subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
save_fig(p_dot5, paste0(output_path, "opcClean_Fig5_Top5_DotPlot"), 9, 6)

write.csv(opc_clean_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opcClean_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 DotPlot saved\n")

# ============================================
# 9. Top10 DotPlot
# ============================================
opc_clean_top10 <- opc_clean_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(opc_clean_top10$gene)

p_dot10 <- DotPlot(opc_clean, features = gene_order10,
                   group.by = "subtype",
                   dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned OPC subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
save_fig(p_dot10, paste0(output_path, "opcClean_Fig6_Top10_DotPlot"), 10, 8)

write.csv(opc_clean_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opcClean_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 DotPlot saved\n")

# ============================================
# 10. Top20 DotPlot
# ============================================
gene_order20 <- unique(opc_clean_top20$gene)

p_dot20 <- DotPlot(opc_clean, features = gene_order20,
                   group.by = "subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Cleaned OPC subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
save_fig(p_dot20, paste0(output_path, "opcClean_Fig7_Top20_DotPlot"), 11, 14)

write.csv(opc_clean_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "opcClean_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 DotPlot saved\n")



In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 0. Extract Microglia (extract from cell_type consistently, same as the other major groups)
# ============================================
micro_cells <- colnames(classFiltSeurat_filt)[
  classFiltSeurat_filt$cell_type == "Microglia"
]

micro_seurat <- subset(classFiltSeurat_filt, cells = micro_cells)
cat("Number of Microglia cells:", ncol(micro_seurat), "\n")
print(table(micro_seurat$donor))

# ============================================
# 1. RNA normalization + PCA
# ============================================
DefaultAssay(micro_seurat) <- "RNA"
micro_seurat <- NormalizeData(micro_seurat, verbose = FALSE)
micro_seurat <- FindVariableFeatures(micro_seurat, selection.method = "vst",
                                     nfeatures = 2000, verbose = FALSE)
micro_seurat <- ScaleData(micro_seurat, verbose = FALSE)
micro_seurat <- RunPCA(micro_seurat, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony (donor correction)
# ============================================
micro_seurat <- RunHarmony(micro_seurat,
                           group.by.vars = "donor",
                           reduction.use = "pca",
                           dims.use = 1:30,
                           verbose = FALSE)

# ============================================
# 3. Nearest-neighbor graph + multi-resolution clustering + clustree
# ============================================
micro_seurat <- FindNeighbors(micro_seurat, reduction = "harmony",
                              dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.05, 0.1, 0.2, 0.3, 0.5)
for (res in resolutions_to_try) {
  micro_seurat <- FindClusters(micro_seurat, resolution = res,
                               future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(micro_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "micro_clustree"), 12, 9)

cat("\nClustering summary at each resolution:\n")
for (res in resolutions_to_try) {
  tab <- table(micro_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.2f: %d clusters, smallest cluster has %d cells\n",
              res, length(tab), min(tab)))
}


In [ ]:
# ============================================
# 4. Choose resolution → final clustering + naming + UMAP
# ============================================
final_res <- 0.05   # ← modify after reviewing clustree

micro_seurat <- SetIdent(micro_seurat, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(micro_seurat)))
cl_new  <- as.character(Idents(micro_seurat))
cl_map  <- setNames(paste0("Microglia_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

micro_seurat$subtype <- factor(unname(cl_map[cl_new]),
                               levels = paste0("Microglia_", 0:(n_clust - 1)))
micro_seurat <- SetIdent(micro_seurat, value = "subtype")

cat("\nDistribution of Microglia subtypes (res =", final_res, "):\n")
print(table(micro_seurat$subtype))

micro_seurat <- RunUMAP(micro_seurat, reduction = "harmony", dims = 1:30,
                        verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(micro_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Microglia Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "micro_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(micro_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "micro_Fig2_UMAP_donor"), 7, 6)

p1c <- DimPlot(micro_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p1c)
save_fig(p1c, paste0(output_path, "micro_Fig2_UMAP_sample"), 7, 6)


In [ ]:
library(dplyr)

# ============================================
# ============================================
# 7. Recompute markers (grouped by subtype)
# ============================================
micro_markers <- FindAllMarkers(micro_seurat,
                                only.pos = TRUE,
                                min.pct = 0.25,
                                logfc.threshold = 0.25,
                                test.use = "wilcox")

write.csv(micro_markers,
          paste0(output_path, "micro_AllMarkers.csv"), row.names = FALSE)
cat("Marker recomputation complete,", nrow(micro_markers), "rows\n")

# ============================================
# 8. Extract Top20 + CSV
# ============================================
micro_top20 <- micro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(micro_top20,
          paste0(output_path, "micro_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers saved\n")

# ============================================
# 9. Top5 DotPlot
# ============================================
micro_top5 <- micro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(micro_top5$gene)

p_dot5 <- DotPlot(micro_seurat, features = gene_order5,
                  group.by = "subtype",
                  dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Microglia subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
save_fig(p_dot5, paste0(output_path, "micro_Fig5_Top5_DotPlot"), 9, 6)

write.csv(micro_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "micro_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 DotPlot saved\n")

# ============================================
# 10. Top10 DotPlot
# ============================================
micro_top10 <- micro_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(micro_top10$gene)

p_dot10 <- DotPlot(micro_seurat, features = gene_order10,
                   group.by = "subtype",
                   dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Microglia subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
save_fig(p_dot10, paste0(output_path, "micro_Fig6_Top10_DotPlot"), 10, 8)

write.csv(micro_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "micro_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 DotPlot saved\n")

# ============================================
# 11. Top20 DotPlot
# ============================================
gene_order20 <- unique(micro_top20$gene)

p_dot20 <- DotPlot(micro_seurat, features = gene_order20,
                   group.by = "subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Microglia subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
save_fig(p_dot20, paste0(output_path, "micro_Fig7_Top20_DotPlot"), 11, 12)

write.csv(micro_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "micro_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 DotPlot saved\n")

In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ---------- 1. Remove Microglia_1 ----------
cat("Subtype distribution before removal:\n")
print(table(micro_seurat$subtype))

micro_removed <- subset(micro_seurat, subset = subtype == "Microglia_1")
saveRDS(micro_removed, paste0(output_path, "micro_removed_1.rds"))
cat("Removed Microglia_1:", ncol(micro_removed), "cells\n")

micro_clean <- subset(micro_seurat, subset = subtype != "Microglia_1")

# Renumber subtypes (keep consecutive 0..n, avoid gaps in numbering)
micro_clean$subtype <- factor(as.character(micro_clean$subtype))
print(table(micro_clean$subtype))

# ---------- 2. Recompute UMAP using the existing harmony space (no re-clustering) ----------
micro_clean <- RunUMAP(micro_clean, reduction = "harmony", dims = 1:30,
                       verbose = FALSE)

# ---------- 3. UMAP ----------
p_umap <- DimPlot(micro_clean, group.by = "subtype", label = TRUE, repel = TRUE,
                  raster = FALSE) +
  coord_fixed() +
  ggtitle("Microglia subtypes (Microglia_1 removed)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_umap)
save_fig(p_umap, paste0(output_path, "microClean_Fig1_UMAP"), 7, 6)

p_donor <- DimPlot(micro_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p_donor)
save_fig(p_donor, paste0(output_path, "microClean_Fig2_UMAP_donor"), 7, 6)

cat("Number of cells after removal:", ncol(micro_clean), "\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 0. Extract Endothelial cells (extract from the original cell_type)
# ============================================
endo_cells <- colnames(classFiltSeurat_filt)[
  classFiltSeurat_filt$cell_type == "Endothelial"
]

endo_seurat <- subset(classFiltSeurat_filt, cells = endo_cells)
cat("Number of Endothelial cells:", ncol(endo_seurat), "\n")
print(table(endo_seurat$donor))

# ============================================
# 1. RNA normalization + PCA
# ============================================
DefaultAssay(endo_seurat) <- "RNA"
endo_seurat <- NormalizeData(endo_seurat, verbose = FALSE)
endo_seurat <- FindVariableFeatures(endo_seurat, selection.method = "vst",
                                    nfeatures = 2000, verbose = FALSE)
endo_seurat <- ScaleData(endo_seurat, verbose = FALSE)
endo_seurat <- RunPCA(endo_seurat, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony (donor correction)
# ============================================
endo_seurat <- RunHarmony(endo_seurat,
                          group.by.vars = "donor",
                          reduction.use = "pca",
                          dims.use = 1:30,
                          verbose = FALSE)

# ============================================
# 3. Nearest-neighbor graph + multi-resolution clustering + clustree
# ============================================
endo_seurat <- FindNeighbors(endo_seurat, reduction = "harmony",
                             dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.02, 0.05, 0.1, 0.2, 0.3, 0.5)
for (res in resolutions_to_try) {
  endo_seurat <- FindClusters(endo_seurat, resolution = res,
                              future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(endo_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "endo_clustree"), 12, 9)

cat("\nClustering summary at each resolution:\n")
for (res in resolutions_to_try) {
  tab <- table(endo_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.2f: %d clusters, smallest cluster has %d cells\n",
              res, length(tab), min(tab)))
}


In [ ]:
# ============================================
# ============================================
# 4. Choose resolution → final clustering + naming + UMAP
# ============================================
final_res <- 0.02   # ← modify after reviewing clustree (few endothelial cells, start with low resolution)

endo_seurat <- SetIdent(endo_seurat, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(endo_seurat)))
cl_new  <- as.character(Idents(endo_seurat))
cl_map  <- setNames(paste0("Endothelial_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

endo_seurat$subtype <- factor(unname(cl_map[cl_new]),
                              levels = paste0("Endothelial_", 0:(n_clust - 1)))
endo_seurat <- SetIdent(endo_seurat, value = "subtype")

cat("\nDistribution of Endothelial subtypes (res =", final_res, "):\n")
print(table(endo_seurat$subtype))

endo_seurat <- RunUMAP(endo_seurat, reduction = "harmony", dims = 1:30,
                       verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(endo_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Endothelial Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "endo_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(endo_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "endo_Fig2_UMAP_donor"), 7, 6)

p1c <- DimPlot(endo_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p1c)
save_fig(p1c, paste0(output_path, "endo_Fig2_UMAP_sample"), 7, 6)

# ---------- Classic marker violin plots ----------
p2 <- VlnPlot(endo_seurat, features = c("FLT1", "CLDN5", "PECAM1",
                                        "VWF", "KDR", "ESAM"),
              pt.size = 0, ncol = 3)
print(p2)
save_fig(p2, paste0(output_path, "endo_Fig3_Vln_ClassicMarkers"), 10, 6)

In [ ]:
library(dplyr)

# ============================================
# ============================================
# 7. Recompute markers (grouped by subtype)
# ============================================
endo_markers <- FindAllMarkers(endo_seurat,
                               only.pos = TRUE,
                               min.pct = 0.25,
                               logfc.threshold = 0.25,
                               test.use = "wilcox")

write.csv(endo_markers,
          paste0(output_path, "endo_AllMarkers.csv"), row.names = FALSE)
cat("Marker recomputation complete,", nrow(endo_markers), "rows\n")

# ============================================
# 8. Extract Top20 + CSV
# ============================================
endo_top20 <- endo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(endo_top20,
          paste0(output_path, "endo_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers saved\n")

# ============================================
# 9. Top5 DotPlot
# ============================================
endo_top5 <- endo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(endo_top5$gene)

p_dot5 <- DotPlot(endo_seurat, features = gene_order5,
                  group.by = "subtype",
                  dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Endothelial subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
save_fig(p_dot5, paste0(output_path, "endo_Fig5_Top5_DotPlot"), 9, 5)

write.csv(endo_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "endo_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 DotPlot saved\n")

# ============================================
# 10. Top10 DotPlot
# ============================================
endo_top10 <- endo_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(endo_top10$gene)

p_dot10 <- DotPlot(endo_seurat, features = gene_order10,
                   group.by = "subtype",
                   dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Endothelial subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
save_fig(p_dot10, paste0(output_path, "endo_Fig6_Top10_DotPlot"), 9, 6)

write.csv(endo_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "endo_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 DotPlot saved\n")

# ============================================
# 11. Top20 DotPlot
# ============================================
gene_order20 <- unique(endo_top20$gene)

p_dot20 <- DotPlot(endo_seurat, features = gene_order20,
                   group.by = "subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Endothelial subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
save_fig(p_dot20, paste0(output_path, "endo_Fig7_Top20_DotPlot"), 10, 10)

write.csv(endo_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "endo_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 DotPlot saved\n")

# # ============================================
# # 12. Save the object (with new subtype names)
# # ============================================
# saveRDS(endo_seurat,
#         paste0(output_path, "endo_seurat_harmony_res",
#                gsub("\\.", "", final_res), ".rds"))
# cat("Endothelial object saved\n")

In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ---------- 1. Remove Endothelial_1 ----------
cat("Subtype distribution before removal:\n")
print(table(endo_seurat$subtype))

endo_removed <- subset(endo_seurat, subset = subtype == "Endothelial_1")
saveRDS(endo_removed, paste0(output_path, "endo_removed_1.rds"))
cat("Removed Endothelial_1:", ncol(endo_removed), "cells\n")

endo_clean <- subset(endo_seurat, subset = subtype != "Endothelial_1")
endo_clean$subtype <- factor(as.character(endo_clean$subtype))
cat("Subtype distribution after removal:\n")
print(table(endo_clean$subtype))

# ---------- 2. Recompute UMAP using the existing harmony space ----------
endo_clean <- RunUMAP(endo_clean, reduction = "harmony", dims = 1:30,
                      verbose = FALSE)

# ---------- 3. UMAP ----------
p_umap <- DimPlot(endo_clean, group.by = "subtype", label = TRUE, repel = TRUE,
                  raster = FALSE) +
  coord_fixed() +
  ggtitle("Endothelial subtypes (Endothelial_1 removed)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_umap)
save_fig(p_umap, paste0(output_path, "endoClean_Fig1_UMAP"), 7, 6)

p_donor <- DimPlot(endo_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p_donor)
save_fig(p_donor, paste0(output_path, "endoClean_Fig2_UMAP_donor"), 7, 6)

cat("Number of cells after removal:", ncol(endo_clean), "\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(clustree)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 0. Extract Ependymal cells (extract from the original cell_type)
# ============================================
epen_cells <- colnames(classFiltSeurat_filt)[
  classFiltSeurat_filt$cell_type == "Ependymal"
]

epen_seurat <- subset(classFiltSeurat_filt, cells = epen_cells)
cat("Number of Ependymal cells:", ncol(epen_seurat), "\n")
print(table(epen_seurat$donor))

# ============================================
# 1. RNA normalization + PCA
# ============================================
DefaultAssay(epen_seurat) <- "RNA"
epen_seurat <- NormalizeData(epen_seurat, verbose = FALSE)
epen_seurat <- FindVariableFeatures(epen_seurat, selection.method = "vst",
                                    nfeatures = 2000, verbose = FALSE)
epen_seurat <- ScaleData(epen_seurat, verbose = FALSE)
epen_seurat <- RunPCA(epen_seurat, npcs = 50, verbose = FALSE)

# ============================================
# 2. Harmony (donor correction)
# ============================================
epen_seurat <- RunHarmony(epen_seurat,
                          group.by.vars = "donor",
                          reduction.use = "pca",
                          dims.use = 1:30,
                          verbose = FALSE)

# ============================================
# 3. Nearest-neighbor graph + multi-resolution clustering + clustree
# ============================================
epen_seurat <- FindNeighbors(epen_seurat, reduction = "harmony",
                             dims = 1:30, verbose = FALSE)

resolutions_to_try <- c(0.02, 0.05, 0.1, 0.2, 0.3, 0.5)
for (res in resolutions_to_try) {
  epen_seurat <- FindClusters(epen_seurat, resolution = res,
                              future.seed = TRUE, verbose = FALSE)
}

p_clustree <- clustree(epen_seurat, prefix = "RNA_snn_res.")
print(p_clustree)
save_fig(p_clustree, paste0(output_path, "epen_clustree"), 12, 9)

cat("\nClustering summary at each resolution:\n")
for (res in resolutions_to_try) {
  tab <- table(epen_seurat@meta.data[[paste0("RNA_snn_res.", res)]])
  cat(sprintf("res=%.2f: %d clusters, smallest cluster has %d cells\n",
              res, length(tab), min(tab)))
}

In [ ]:
# ============================================
# ============================================
# 4. Choose resolution → final clustering + naming + UMAP
# ============================================
final_res <- 0.02   # ← modify after reviewing clustree

epen_seurat <- SetIdent(epen_seurat, value = paste0("RNA_snn_res.", final_res))

n_clust <- length(unique(Idents(epen_seurat)))
cl_new  <- as.character(Idents(epen_seurat))
cl_map  <- setNames(paste0("Ependymal_", seq_along(sort(unique(as.numeric(cl_new)))) - 1),
                    as.character(sort(unique(as.numeric(cl_new)))))

epen_seurat$subtype <- factor(unname(cl_map[cl_new]),
                              levels = paste0("Ependymal_", 0:(n_clust - 1)))
epen_seurat <- SetIdent(epen_seurat, value = "subtype")

cat("\nDistribution of Ependymal subtypes (res =", final_res, "):\n")
print(table(epen_seurat$subtype))

epen_seurat <- RunUMAP(epen_seurat, reduction = "harmony", dims = 1:30,
                       verbose = FALSE)

# ---------- UMAP ----------
p1 <- DimPlot(epen_seurat, group.by = "subtype", label = TRUE, repel = TRUE,
              raster = FALSE) +
  coord_fixed() +
  ggtitle(paste0("Ependymal Subtypes (res = ", final_res, ")"))
print(p1)
save_fig(p1, paste0(output_path, "epen_Fig1_UMAP_subtypes"), 7, 6)

p1b <- DimPlot(epen_seurat, group.by = "donor", raster = FALSE) + coord_fixed()
print(p1b)
save_fig(p1b, paste0(output_path, "epen_Fig2_UMAP_donor"), 7, 6)

p1c <- DimPlot(epen_seurat, group.by = "sample_label", raster = FALSE) + coord_fixed()
print(p1c)
save_fig(p1c, paste0(output_path, "epen_Fig2_UMAP_sample"), 7, 6)

In [ ]:
library(dplyr)

# ============================================
# ============================================
# 7. Recompute markers (grouped by subtype)
# ============================================
epen_markers <- FindAllMarkers(epen_seurat,
                               only.pos = TRUE,
                               min.pct = 0.25,
                               logfc.threshold = 0.25,
                               test.use = "wilcox")

write.csv(epen_markers,
          paste0(output_path, "epen_AllMarkers.csv"), row.names = FALSE)
cat("Marker recomputation complete,", nrow(epen_markers), "rows\n")

# ============================================
# 8. Extract Top20 + CSV
# ============================================
epen_top20 <- epen_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 20) %>%
  arrange(cluster, desc(avg_log2FC))

write.csv(epen_top20,
          paste0(output_path, "epen_Subtype_top20_markers.csv"),
          row.names = FALSE)
cat("Top20 markers saved\n")

# ============================================
# 9. Top5 DotPlot
# ============================================
epen_top5 <- epen_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 5) %>%
  ungroup() %>%
  arrange(cluster)

gene_order5 <- unique(epen_top5$gene)

p_dot5 <- DotPlot(epen_seurat, features = gene_order5,
                  group.by = "subtype",
                  dot.scale = 6, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Ependymal subtypes: Top5 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 9),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot5)
save_fig(p_dot5, paste0(output_path, "epen_Fig5_Top5_DotPlot"), 9, 5)

write.csv(epen_top5 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "epen_Subtype_Top5_table.csv"),
          row.names = FALSE)
cat("Top5 DotPlot saved\n")

# ============================================
# 10. Top10 DotPlot
# ============================================
epen_top10 <- epen_markers %>%
  group_by(cluster) %>%
  slice_max(order_by = avg_log2FC, n = 10) %>%
  ungroup() %>%
  arrange(cluster)

gene_order10 <- unique(epen_top10$gene)

p_dot10 <- DotPlot(epen_seurat, features = gene_order10,
                   group.by = "subtype",
                   dot.scale = 5, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Ependymal subtypes: Top10 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
    axis.text.y = element_text(size = 8),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14)
  )
print(p_dot10)
save_fig(p_dot10, paste0(output_path, "epen_Fig6_Top10_DotPlot"), 9, 6)

write.csv(epen_top10 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "epen_Subtype_Top10_table.csv"),
          row.names = FALSE)
cat("Top10 DotPlot saved\n")

# ============================================
# 11. Top20 DotPlot
# ============================================
gene_order20 <- unique(epen_top20$gene)

p_dot20 <- DotPlot(epen_seurat, features = gene_order20,
                   group.by = "subtype",
                   dot.scale = 4, cols = c("lightgrey", "red")) +
  coord_flip() + RotatedAxis() +
  ggtitle("Ependymal subtypes: Top20 markers") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
    axis.text.y = element_text(size = 7),
    plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )
print(p_dot20)
save_fig(p_dot20, paste0(output_path, "epen_Fig7_Top20_DotPlot"), 10, 10)

write.csv(epen_top20 %>% select(cluster, gene, avg_log2FC, pct.1, pct.2, p_val_adj),
          paste0(output_path, "epen_Subtype_Top20_table.csv"),
          row.names = FALSE)
cat("Top20 DotPlot saved\n")

# # ============================================
# # 12. Save the object (with new subtype names)
# # ============================================
# saveRDS(epen_seurat,
#         paste0(output_path, "epen_seurat_harmony_res",
#                gsub("\\.", "", final_res), ".rds"))
# cat("Ependymal object saved\n")

In [ ]:
library(Seurat)
library(ggplot2)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ---------- 1. Remove Ependymal_1 ----------
cat("Subtype distribution before removal:\n")
print(table(epen_seurat$subtype))

epen_removed <- subset(epen_seurat, subtype = NULL, subset = subtype == "Ependymal_1")
saveRDS(epen_removed, paste0(output_path, "epen_removed_1.rds"))
cat("Removed Ependymal_1:", ncol(epen_removed), "cells\n")

epen_clean <- subset(epen_seurat, subset = subtype != "Ependymal_1")
epen_clean$subtype <- factor(as.character(epen_clean$subtype))
cat("Subtype distribution after removal:\n")
print(table(epen_clean$subtype))

# ---------- 2. Recompute UMAP using the existing harmony space ----------
epen_clean <- RunUMAP(epen_clean, reduction = "harmony", dims = 1:30,
                      verbose = FALSE)

# ---------- 3. UMAP ----------
p_umap <- DimPlot(epen_clean, group.by = "subtype", label = TRUE, repel = TRUE,
                  raster = FALSE) +
  coord_fixed() +
  ggtitle("Ependymal subtypes (Ependymal_1 removed)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_umap)
save_fig(p_umap, paste0(output_path, "epenClean_Fig1_UMAP"), 7, 6)

p_donor <- DimPlot(epen_clean, group.by = "donor", raster = FALSE) + coord_fixed()
print(p_donor)
save_fig(p_donor, paste0(output_path, "epenClean_Fig2_UMAP_donor"), 7, 6)

cat("Number of cells after removal:", ncol(epen_clean), "\n")

In [ ]:
library(Seurat)

# ============================================
# ============================================
# 0. Make sure classFiltSeurat_filt is in memory
# ============================================
# Skip if already in memory

# ============================================
# 1. Summary table: use only the cleaned objects
# ============================================
subtype_sources <- list(
  ht5ht  = list(obj = ht_5ht,       col = "ht_subtype"),  # 5-HT subtypes (your cleaned version)
  astro  = list(obj = astro_clean,  col = "subtype"),      # Astrocytes_0/1 (2/3 removed)
  oligo  = list(obj = oligo_clean,  col = "subtype"),      # Oligo_0/1 (2 removed)
  opc    = list(obj = opc_clean,    col = "subtype"),      # OPC_* (3 removed, 2 may remain)
  micro  = list(obj = micro_clean,  col = "subtype"),      # Microglia_* (1 removed)
  endo   = list(obj = endo_clean,   col = "subtype"),      # Endothelial_* (1 removed)
  epen   = list(obj = epen_clean,   col = "subtype")       # Ependymal_* (1 removed)
)

# ============================================
# 2. Build the unified column
# ============================================
classFiltSeurat_filt$cell_type_fine <- as.character(classFiltSeurat_filt$cell_type)

# Record which major classes were re-clustered (cells in these classes not written back = removed)
reclustered_classes <- c("Astrocytes", "Oligo", "OPC", "Microglia",
                         "Endothelial", "Ependymal")

for (nm in names(subtype_sources)) {
  obj <- subtype_sources[[nm]]$obj
  col <- subtype_sources[[nm]]$col

  if (is.null(obj)) {
    cat("Warning: skipped (object does not exist, confirm the step has been run):", nm, "\n")
    next
  }

  map_back <- setNames(as.character(obj@meta.data[[col]]), colnames(obj))
  hit <- names(map_back) %in% colnames(classFiltSeurat_filt)

  classFiltSeurat_filt$cell_type_fine[names(map_back)] <- map_back

  cat(sprintf("%-8s wrote back %d cells (%d matched successfully)\n",
              nm, length(map_back), sum(hit)))
}

# ============================================
# 3. Label the removed cells separately (they belong to no remaining subtype)
# ============================================
in_reclustered <- classFiltSeurat_filt$cell_type %in% reclustered_classes
got_label <- !is.na(classFiltSeurat_filt$cell_type_fine) &
             classFiltSeurat_filt$cell_type_fine != classFiltSeurat_filt$cell_type &
             !is.na(match(classFiltSeurat_filt$cell_type_fine,
                          unlist(lapply(subtype_sources,
                                        function(x) if(!is.null(x$obj))
                                          as.character(x$obj@meta.data[[x$col]])))))

# More direct algorithm: within a re-clustered class, cells whose cell_type_fine
# still equals the original class name = not written back = removed
removed_mask <- in_reclustered &
                (classFiltSeurat_filt$cell_type_fine == classFiltSeurat_filt$cell_type)

classFiltSeurat_filt$cell_type_fine[removed_mask] <- "Removed"

n_removed <- sum(removed_mask)
cat("\nRemoved cells (labeled as Removed):", n_removed, "\n")
print(sort(table(classFiltSeurat_filt$cell_type[removed_mask]), decreasing = TRUE))

classFiltSeurat_filt$cell_type_fine <- factor(classFiltSeurat_filt$cell_type_fine)



In [ ]:
# ============================================
# 4. Print the summary table (fixed sorting)
# ============================================
tb <- table(classFiltSeurat_filt$cell_type_fine)

sort_key <- function(x) {
  num <- rep(-1L, length(x))
  has_num <- grepl("_?[0-9]+$", x)
  if (any(has_num)) {
    m <- regmatches(x[has_num], regexpr("_?[0-9]+$", x[has_num]))
    num[has_num] <- suppressWarnings(as.numeric(gsub("_", "", m)))
  }
  data.frame(name = x,
             base = gsub("_?[0-9]+$", "", x),
             num  = num,
             stringsAsFactors = FALSE)
}

sk <- sort_key(names(tb))
sk <- sk[order(sk$base, sk$num), ]
tb_sorted <- tb[sk$name]

cat("\n===== Distribution of all cells after merging (", length(tb_sorted), "types) =====\n")
print(tb_sorted)

# ============================================
# 5. Verification
# ============================================
cat("\n--- Verification ---\n")
cat("Total cells:", ncol(classFiltSeurat_filt), "\n")
cat("Removed:", sum(tb_sorted[names(tb_sorted) == "Removed"]),
    ", all others combined:", sum(tb_sorted[names(tb_sorted) != "Removed"]), "\n")

is_neuron  <- grepl("^(5-HT|Glut|GABA|DA)_", names(tb_sorted))
is_removed <- names(tb_sorted) == "Removed"
cat("Neuronal subtypes combined:", sum(tb_sorted[is_neuron]),
    ", non-neuronal subtypes combined:", sum(tb_sorted[!is_neuron & !is_removed]), "\n")




In [ ]:
# ============================================
# 6. Save (uncomment after verification)
# ============================================
saveRDS(classFiltSeurat_filt,
        paste0(output_path, "classFiltSeurat_filt_allSubtypes.rds"))
write.csv(data.frame(cell_type = names(tb_sorted), n = as.integer(tb_sorted)),
          paste0(output_path, "cell_type_fine_distribution.csv"),
          row.names = FALSE)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 0. Remove "Removed" cells, keep only clean cells
# ============================================
cat("Total cells before integration:", ncol(classFiltSeurat_filt), "\n")

seurat_all <- subset(classFiltSeurat_filt,
                     subset = cell_type_fine != "Removed")
cat("After removing Removed:", ncol(seurat_all), "\n")

# ============================================
# 1. Build the major-class column: neuron subtypes → "Neurons", others by prefix
# ============================================
ct <- as.character(seurat_all$cell_type_fine)

seurat_all$cell_class <- dplyr::case_when(
  grepl("^(5-HT|Glut|GABA|Glut-GABA|DA)_", ct) ~ "Neurons",
  startsWith(ct, "Astrocytes")  ~ "Astrocytes",
  startsWith(ct, "Oligo")       ~ "Oligo",
  startsWith(ct, "OPC")         ~ "OPC",
  startsWith(ct, "Microglia")   ~ "Microglia",
  startsWith(ct, "Endothelial") ~ "Endothelial",
  startsWith(ct, "Ependymal")   ~ "Ependymal",
  TRUE ~ "Other"
)
seurat_all$cell_class <- factor(seurat_all$cell_class,
                                levels = c("Neurons","Astrocytes","Oligo","OPC",
                                           "Microglia","Endothelial","Ependymal"))

seurat_all$cell_type_fine <- factor(seurat_all$cell_type_fine)

cat("\nDistribution of major classes:\n")
print(table(seurat_all$cell_class))

# ============================================
# 2. RNA normalization + PCA + Harmony
# ============================================
DefaultAssay(seurat_all) <- "RNA"

seurat_all <- NormalizeData(seurat_all, verbose = FALSE)
seurat_all <- FindVariableFeatures(seurat_all, selection.method = "vst",
                                   nfeatures = 3000, verbose = FALSE)
seurat_all <- ScaleData(seurat_all, verbose = FALSE)
seurat_all <- RunPCA(seurat_all, npcs = 50, verbose = FALSE)

seurat_all <- RunHarmony(seurat_all,
                         group.by.vars = "donor",
                         reduction.use = "pca",
                         dims.use = 1:30,
                         verbose = FALSE)

# ============================================
# 3. UMAP
# ============================================
seurat_all <- RunUMAP(seurat_all, reduction = "harmony", dims = 1:30,
                      verbose = FALSE)

# ============================================
# 4. Fig 1: colored by the 8 major classes
# ============================================
p_class <- DimPlot(seurat_all, group.by = "cell_class",
                   raster = FALSE, shuffle = TRUE) +
  coord_fixed() +
  ggtitle("All cells (cleaned): major classes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
        legend.text = element_text(size = 10))
print(p_class)
save_fig(p_class, paste0(output_path, "all_Fig1_UMAP_majorClasses"), 8, 7)

# Unlabeled version (larger and cleaner)
p_class_nolabel <- DimPlot(seurat_all, group.by = "cell_class",
                           raster = TRUE, shuffle = TRUE) +
  coord_fixed() +
  NoLegend() +
  ggtitle("All cells: major classes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14))
save_fig(p_class_nolabel, paste0(output_path, "all_Fig1b_UMAP_majorClasses_noLegend"),
         7, 6)

# ============================================
# 5. Fig 2: colored by all 57 minor classes
# ============================================
# Subtype order: the numeric 0→n order arranged earlier
fine_levels <- names(tb_sorted)[names(tb_sorted) != "Removed"]
seurat_all$cell_type_fine <- factor(seurat_all$cell_type_fine,
                                    levels = fine_levels)

p_fine <- DimPlot(seurat_all, group.by = "cell_type_fine",
                  label = TRUE, repel = TRUE, raster = TRUE,
                  shuffle = TRUE) +
  coord_fixed() +
  NoLegend() +
  ggtitle("All cells: all subtypes (57)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14))
print(p_fine)
save_fig(p_fine, paste0(output_path, "all_Fig2_UMAP_allSubtypes"), 12, 10)

# ---------- Bonus: subtype plot faceted by major class (commonly used in publications) ----------
p_split <- DimPlot(seurat_all, group.by = "cell_type_fine",
                   split.by = "cell_class",
                   raster = TRUE, shuffle = TRUE) +
  coord_fixed() +
  NoLegend() +
  ggtitle("All subtypes split by major class")
save_fig(p_split, paste0(output_path, "all_Fig3_UMAP_subtypes_split"), 20, 5)

# ============================================
# 6. Check batches by donor / sample
# ============================================
p_donor <- DimPlot(seurat_all, group.by = "donor", raster = TRUE, shuffle = TRUE) +
  coord_fixed()
save_fig(p_donor, paste0(output_path, "all_Fig4_UMAP_donor"), 8, 7)

# # ============================================
# # 7. Save
# # ============================================
# saveRDS(seurat_all,
#         paste0(output_path, "seurat_all_harmony_integrated.rds"))
# cat("Integrated full object saved\n")

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(harmony)

output_path <- "./data/"

save_fig <- function(plot_obj, file_base, w, h) {
  ggsave(paste0(file_base, ".png"), plot_obj,
         width = w, height = h, dpi = 300)
  ggsave(paste0(file_base, ".pdf"), plot_obj,
         width = w, height = h)
  cat("Saved:", file_base, "(PNG + PDF)\n")
}

# ============================================
# 0. Extract all non-neuronal subtypes
# ============================================
# Extract from the integrated object (Removed already excluded);
# if seurat_all does not exist, extract from classFiltSeurat_filt instead
non_neuron_levels <- c("Astrocytes_0","Astrocytes_1",
                       "Oligo_0","Oligo_1",
                       "OPC_0","OPC_1","OPC_2","OPC_3",
                       "Microglia_0",
                       "Endothelial_0",
                       "Ependymal_0")

nn_seurat <- subset(seurat_all,
                    subset = cell_type_fine %in% non_neuron_levels)
# If the object above does not exist, use:
# nn_seurat <- subset(classFiltSeurat_filt,
#                     subset = cell_type_fine %in% non_neuron_levels &
#                              cell_type_fine != "Removed")

nn_seurat$cell_type_fine <- factor(as.character(nn_seurat$cell_type_fine),
                                   levels = non_neuron_levels)

cat("Number of non-neuronal cells:", ncol(nn_seurat), "\n")
print(table(nn_seurat$cell_type_fine))

# Major-class column (for faceting)
nn_seurat$cell_class <- dplyr::case_when(
  startsWith(as.character(nn_seurat$cell_type_fine), "Astrocytes")  ~ "Astrocytes",
  startsWith(as.character(nn_seurat$cell_type_fine), "Oligo")       ~ "Oligo",
  startsWith(as.character(nn_seurat$cell_type_fine), "OPC")         ~ "OPC",
  startsWith(as.character(nn_seurat$cell_type_fine), "Microglia")   ~ "Microglia",
  startsWith(as.character(nn_seurat$cell_type_fine), "Endothelial") ~ "Endothelial",
  startsWith(as.character(nn_seurat$cell_type_fine), "Ependymal")   ~ "Ependymal"
)
nn_seurat$cell_class <- factor(nn_seurat$cell_class,
                               levels = c("Oligo","OPC","Astrocytes","Microglia",
                                          "Endothelial","Ependymal"))
print(table(nn_seurat$cell_class))

# ============================================
# 1. RNA + PCA + Harmony (re-integration within non-neuronal cells)
# ============================================
DefaultAssay(nn_seurat) <- "RNA"

nn_seurat <- NormalizeData(nn_seurat, verbose = FALSE)
nn_seurat <- FindVariableFeatures(nn_seurat, selection.method = "vst",
                                  nfeatures = 3000, verbose = FALSE)
nn_seurat <- ScaleData(nn_seurat, verbose = FALSE)
nn_seurat <- RunPCA(nn_seurat, npcs = 50, verbose = FALSE)

nn_seurat <- RunHarmony(nn_seurat,
                        group.by.vars = "donor",
                        reduction.use = "pca",
                        dims.use = 1:30,
                        verbose = FALSE)

nn_seurat <- RunUMAP(nn_seurat, reduction = "harmony", dims = 1:30,
                     verbose = FALSE)



In [ ]:
# ============================================
# 2. Fig 1: colored by the 6 major classes (check lineage separation first)
# ============================================
p_class <- DimPlot(nn_seurat, group.by = "cell_class",
                   raster = FALSE, shuffle = TRUE) +
  coord_fixed() +
  ggtitle("Non-neuronal cells: major classes") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
        legend.text = element_text(size = 11))
print(p_class)
save_fig(p_class, paste0(output_path, "nn_Fig1_UMAP_classes"), 8, 7)

# ============================================
# 3. Fig 2: colored by the 11 subtypes (with labels)
# ============================================
p_fine <- DimPlot(nn_seurat, group.by = "cell_type_fine",
                  label = TRUE, repel = TRUE, raster = TRUE,
                  shuffle = TRUE) +
  coord_fixed() +
  NoLegend() +
  ggtitle("Non-neuronal subtypes (11)") +
  theme(plot.title = element_text(hjust = 0.5, face = "bold", size = 14))
print(p_fine)
save_fig(p_fine, paste0(output_path, "nn_Fig2_UMAP_subtypes"), 9, 8)

# ---------- Fig 3: faceted by major class (check internal structure of each class) ----------
p_split <- DimPlot(nn_seurat, group.by = "cell_type_fine",
                   split.by = "cell_class",
                   label = TRUE, repel = TRUE,
                   raster = TRUE, shuffle = TRUE) +
  coord_fixed() +
  NoLegend() +
  ggtitle("Non-neuronal subtypes split by class")
save_fig(p_split, paste0(output_path, "nn_Fig3_UMAP_split"), 20, 5)

# ---------- Fig 4: donor batch check ----------
p_donor <- DimPlot(nn_seurat, group.by = "donor",
                   raster = TRUE, shuffle = TRUE) +
  coord_fixed() +
  ggtitle("Non-neuronal: donor distribution")
save_fig(p_donor, paste0(output_path, "nn_Fig4_UMAP_donor"), 8, 7)

# # ============================================
# # 4. Save
# # ============================================
# saveRDS(nn_seurat,
#         paste0(output_path, "nn_seurat_harmony_integrated.rds"))
# cat("Integrated non-neuronal object saved\n")

In [ ]:
suppressMessages(library(tidyverse))
library(Seurat)
library(patchwork)

output_path <- "./data/"

# ============================================
# ============================================
# 0. Load the saved object with subtype annotations
# ============================================
classFiltSeurat <- readRDS(paste0(output_path, "classFiltSeurat_filt_allSubtypes.rds"))
cat("Loaded, number of cells:", ncol(classFiltSeurat), "\n")
print(colnames(classFiltSeurat@meta.data))   # confirm the cell_type_fine column exists

## ===== 1. Preparation =====
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)

## ===== 2. Genes + cell type order =====
top_genes <- c("SNAP25", "SLC17A6", "SLC17A8", "GAD1", "GAD2", "TH")

target_types <- c(
  paste0("Glut_", 1:7),
  paste0("Glut-GABA_", 2:6),
  paste0("GABA_", 1:16),
  "DA_1"
)

type2class <- c(
  setNames(rep("GLUT", 7),      paste0("Glut_", 1:7)),
  setNames(rep("GLUT_GABA", 5), paste0("Glut-GABA_", 2:6)),
  setNames(rep("GABA", 16),     paste0("GABA_", 1:16)),
  c("DA_1" = "DA")
)

class_colors <- c("GLUT"="#4DAF4A", "GLUT_GABA"="#984EA3",
                  "GABA"="#E41A1C", "DA"="#333333")

sub_obj <- subset(classFiltSeurat, subset = cell_type_fine %in% target_types)
cat("Number of extracted cells:", ncol(sub_obj), "\n")
print(table(sub_obj$cell_type_fine))

## ===== 3. Extract + long table =====
expr <- FetchData(sub_obj, vars = c("cell_type_fine", top_genes))

expr_long <- expr %>%
  pivot_longer(cols = -cell_type_fine, names_to = "gene", values_to = "Expression") %>%
  mutate(gene = factor(gene, levels = top_genes),
         cell_type_fine = factor(cell_type_fine, levels = target_types),
         class = type2class[as.character(cell_type_fine)],
         class = factor(class, levels = c("GLUT", "GLUT_GABA", "GABA", "DA")))

## ===== 4. Top major-class color bar =====
p_bar <- ggplot(expr_long, aes(x = cell_type_fine, y = 1, fill = class)) +
  geom_tile() +
  scale_fill_manual(values = class_colors) +
  scale_x_discrete(expand = c(0, 0)) +
  theme_void() +
  theme(legend.position = "none",
        plot.margin = margin(0, 5.5, 0, 5.5))

## ===== 5. Main plot: one row per gene, all in a single column =====
p_main <- ggplot(expr_long, aes(x = cell_type_fine, y = Expression, fill = class)) +
  geom_violin(scale = "width", trim = TRUE, color = "grey30", linewidth = 0.3) +
  facet_wrap(~ gene, nrow = 6, ncol = 1,
             scales = "free_y", strip.position = "right") +
  scale_fill_manual(values = class_colors) +
  scale_x_discrete(expand = c(0, 0)) +
  labs(x = NULL, y = "Expression Level") +
  theme_bw(base_size = 13) +
  theme(legend.position = "none",
        panel.grid = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(face = "bold", size = 12),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9))

## ===== 6. Combine + save =====
p <- p_bar / p_main + plot_layout(heights = c(1, 18))

print(p)

ggsave(paste0(output_path, "VlnPlot_genes_stacked_neuro_types.png"),
       p, width = 12, height = 14, dpi = 300)
ggsave(paste0(output_path, "VlnPlot_genes_stacked_neuro_types.pdf"),
       p, width = 12, height = 14)
cat("Figure saved\n")